# Name That LLM: frozen Qwen3-8B-Base diagnostic

Attach the official competition data and the pinned local base-model
snapshot. Select **GPU T4 x2** and **Internet off**, then **Save & Run All**.
This initial experiment extracts frozen features and fits a small
classifier. Its score has not been measured; it is a diagnostic starting
point for comparing features, pooling, and validation choices.

`RUN_GROUPED_CV = False` below keeps the first submission run small.
Turn it on for honest prompt-grouped validation. The notebook embeds
project source and never downloads a model or calls a hosted API.


In [ ]:
# Embedded project source; regenerate this notebook after editing src/ntl.
import hashlib
import importlib
import json
import os
from pathlib import Path
import sys

# Scoring is offline. These also prevent accidental Hugging Face requests.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SOURCE_JSON = '{"ntl/__init__.py": "\\"\\"\\"Reproducible, offline-friendly tools for the Name That LLM competition.\\"\\"\\"\\n\\n__version__ = \\"0.1.0\\"\\n", "ntl/__main__.py": "from .cli import main\\n\\nraise SystemExit(main())\\n", "ntl/claude_glm.py": "\\"\\"\\"Claude-GLM specialist: frozen Qwen features (control) vs Qwen + StyleDistance (candidate).\\n\\nA seven-class head over the five Claude models and the two GLMs is trained on those rows\\nof each training partition only. It redistributes the current system\'s probability mass\\non the seven: for those classes p\' = (1 - s) p + s M q, where M is the current system\'s\\nmass on the seven and q the specialist\'s distribution; every other class keeps its\\nprobability, and s = 0 is the current system exactly. Inner prompt folds of each outer\\ntraining partition choose s from {0, 0.25, 0.5, 0.75, 1}, scored on the full 18 classes.\\nEvery passage of the original five prompt folds is scored and compared with the current\\nsystem by a paired prompt bootstrap.\\n\\nBoth heads are the frozen Qwen recipe\'s classifier (standardization, then C=0.005\\nlogistic regression). The candidate appends StyleDistance, scaled after standardization\\nso that its block carries the same total variance as the Qwen block; the Qwen block is\\ntreated exactly as in the control.\\n\\nFor the topic robustness check, `--groups` regroups the rows (whole prompts move\\ntogether), as `topic_cv` did: the current system\'s own inner weight choice, the\\nspecialist\'s inner folds and the bootstrap then all hold out and resample whole topics.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport time\\nfrom dataclasses import asdict\\nfrom pathlib import Path\\n\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.metrics import balanced_accuracy_score\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import FunctionTransformer, StandardScaler\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom . import ensemble, neural\\nfrom .constants import LABELS\\nfrom .data import fingerprint_frame, read_test, read_train, write_submission\\nfrom .encoders import ENCODERS, extract_embeddings\\nfrom .frozen_gemini import load_feature_archive, save_feature_archive\\nfrom .lora import AdapterConfig, validate_plan\\nfrom .lora_compare import current_inner, current_on_split\\nfrom .lora_final import _aligned_plan\\nfrom .modeling import aligned_probabilities\\nfrom .selection import first_best\\n\\nSPECIALIST = (\\"claude-fable-5\\", \\"claude-opus-4-6\\", \\"claude-opus-4-8\\", \\"claude-opus-5\\", \\"claude-sonnet-5\\",\\n              \\"glm-5.3\\", \\"glm-5.3-flash\\")\\nINDEX = np.array([LABELS.index(label) for label in SPECIALIST])\\n# Canonical label index -> specialist class index (-1 outside the specialist).\\n_POSITION = np.full(len(LABELS), -1)\\n_POSITION[INDEX] = np.arange(len(SPECIALIST))\\nWEIGHTS = (0.0, 0.25, 0.5, 0.75, 1.0)\\nFEATURE_SETS = {\\"qwen\\": (\\"qwen\\",), \\"qwen+styledistance\\": (\\"qwen\\", \\"styledistance\\")}\\n\\n\\ndef feature_names(feature_set: str) -> tuple[str, ...]:\\n    \\"\\"\\"The blocks of a feature set: a named one, or blocks joined by \\"+\\" (Qwen first).\\"\\"\\"\\n    return FEATURE_SETS.get(feature_set, tuple(feature_set.split(\\"+\\")))\\nBLEND = [f\\"blend::{label}\\" for label in LABELS]\\nRECIPE_KIND = \\"claude-glm-specialist-recipe\\"\\n# The validated candidate; the control (Qwen only) gave no gain.\\nFINAL_FEATURE_SET = \\"qwen+styledistance\\"\\n# Final weight choice on the full-data inner folds, as lora_final seeds them.\\nFINAL_SEED_OFFSET = 100\\n\\n\\ndef _json(path: Path, value: dict) -> None:\\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef _column_weights(features: np.ndarray, weights: np.ndarray) -> np.ndarray:\\n    return features * weights\\n\\n\\ndef specialist_head(widths: tuple[int, ...], frozen: neural.FrozenConfig) -> Pipeline:\\n    \\"\\"\\"The frozen recipe\'s head; every block after the first is scaled to the first\'s total variance.\\"\\"\\"\\n    weights = np.concatenate([np.full(width, np.sqrt(widths[0] / width)) for width in widths])\\n    return Pipeline([\\n        (\\"scale\\", StandardScaler()),\\n        (\\"balance\\", FunctionTransformer(_column_weights, kw_args={\\"weights\\": weights})),\\n        (\\"logistic\\", LogisticRegression(C=frozen.C, max_iter=2000, random_state=frozen.seed)),\\n    ])\\n\\n\\ndef specialist_probabilities(blocks: dict[str, np.ndarray], feature_set: str, truth: np.ndarray,\\n                             fit: np.ndarray, rows: np.ndarray, frozen: neural.FrozenConfig) -> np.ndarray:\\n    \\"\\"\\"Fit on the specialist-class rows among `fit`; seven-class probabilities for `rows`.\\"\\"\\"\\n    names = feature_names(feature_set)\\n    features = np.hstack([blocks[name] for name in names])\\n    fit = fit[np.isin(truth[fit], INDEX)]\\n    targets = _POSITION[truth[fit]]\\n    if set(targets) != set(range(len(SPECIALIST))):\\n        raise ValueError(\\"Every specialist class needs training rows\\")\\n    head = specialist_head(tuple(blocks[name].shape[1] for name in names), frozen)\\n    with threadpool_limits(limits=2):\\n        head.fit(features[fit], targets)\\n        probabilities = head.predict_proba(features[rows])\\n    if not np.array_equal(head.classes_, np.arange(len(SPECIALIST))):\\n        raise ValueError(\\"Specialist head must contain every specialist class\\")\\n    return probabilities\\n\\n\\ndef redistribute(current: np.ndarray, specialist: np.ndarray, weight: float) -> np.ndarray:\\n    \\"\\"\\"(1 - s) p + s M q on the specialist\'s classes; every other class keeps its probability.\\"\\"\\"\\n    result = np.array(current, dtype=float, copy=True)\\n    mass = result[:, INDEX].sum(axis=1, keepdims=True)\\n    result[:, INDEX] = (1 - weight) * result[:, INDEX] + weight * mass * specialist\\n    return result\\n\\n\\ndef groups_sha256(train: pd.DataFrame) -> str:\\n    \\"\\"\\"The row grouping (prompt or topic) every split and the bootstrap use.\\"\\"\\"\\n    pairs = sorted(zip(train[\\"id\\"].astype(str), train[\\"prompt_id\\"].astype(str)))\\n    return hashlib.sha256(json.dumps(pairs, separators=(\\",\\", \\":\\")).encode()).hexdigest()\\n\\n\\ndef regroup(train: pd.DataFrame, groups: pd.DataFrame) -> pd.DataFrame:\\n    \\"\\"\\"Rows regrouped, e.g. by topic, as topic_cv does; whole prompts move together.\\"\\"\\"\\n    mapping = groups.set_index(\\"id\\")[\\"prompt_id\\"]\\n    if mapping.index.duplicated().any() or set(mapping.index) != set(train[\\"id\\"]):\\n        raise ValueError(\\"Groups must cover every training row exactly once\\")\\n    group = train[\\"id\\"].map(mapping).astype(str).to_numpy()\\n    if pd.Series(group).groupby(train[\\"prompt_id\\"].to_numpy()).nunique().max() != 1:\\n        raise ValueError(\\"Groups must keep each prompt\'s passages together\\")\\n    return train.assign(prompt_id=group)\\n\\n\\ndef _aligned_current(current_oof: pd.DataFrame, train: pd.DataFrame, folds: np.ndarray) -> np.ndarray:\\n    frame = current_oof.set_index(\\"id\\").loc[train[\\"id\\"]]\\n    if (frame[\\"true_model\\"].to_numpy() != train[\\"model\\"].to_numpy()).any():\\n        raise ValueError(\\"Current-system OOF labels differ from the official rows\\")\\n    if (frame[\\"fold\\"].to_numpy() != folds).any():\\n        raise ValueError(\\"Current-system OOF must come from the plan\'s outer folds\\")\\n    if (frame[\\"prompt_id\\"].astype(str).to_numpy() != train[\\"prompt_id\\"].astype(str).to_numpy()).any():\\n        raise ValueError(\\"Current-system OOF must come from the same row grouping (prompts or topics)\\")\\n    return frame[BLEND].to_numpy()\\n\\n\\ndef build_current_inner(train: pd.DataFrame, features: np.ndarray, plan: pd.DataFrame, outer_fold: int,\\n                        config: ensemble.BlendConfig | None = None) -> pd.DataFrame:\\n    \\"\\"\\"The current system refit, with its own nested weight, on each inner split of one outer fold.\\"\\"\\"\\n    config = config or ensemble.BlendConfig()\\n    plan = validate_plan(train, plan, AdapterConfig())\\n    frames = []\\n    for inner_fold, (valid, probabilities, weight) in current_inner(train, features, plan, outer_fold,\\n                                                                    config).items():\\n        frame = pd.DataFrame(probabilities, columns=BLEND)\\n        frame.insert(0, \\"groups_sha256\\", groups_sha256(train))\\n        frame.insert(0, \\"weight\\", weight)\\n        frame.insert(0, \\"inner_fold\\", inner_fold)\\n        frame.insert(0, \\"outer_fold\\", outer_fold)\\n        frame.insert(0, \\"id\\", train[\\"id\\"].to_numpy()[valid])\\n        frames.append(frame)\\n    return pd.concat(frames, ignore_index=True)\\n\\n\\ndef _inner_current(cache: pd.DataFrame, train: pd.DataFrame, plan: pd.DataFrame, outer_fold: int) -> dict:\\n    \\"\\"\\"{inner fold: (row positions, probabilities)}, checked to cover exactly that inner fold.\\"\\"\\"\\n    position = pd.Series(np.arange(len(train)), index=train[\\"id\\"])\\n    column = plan[f\\"inner_{outer_fold}\\"].to_numpy()\\n    result = {}\\n    for inner_fold in sorted(set(column[column >= 0])):\\n        rows = cache[(cache[\\"outer_fold\\"] == outer_fold) & (cache[\\"inner_fold\\"] == inner_fold)]\\n        positions = position.loc[rows[\\"id\\"]].to_numpy()\\n        if len(positions) != (column == inner_fold).sum() or set(positions) != set(np.flatnonzero(column == inner_fold)):\\n            raise ValueError(\\"Cached current-system inner predictions must cover each inner fold exactly\\")\\n        result[inner_fold] = (positions, rows[BLEND].to_numpy())\\n    return result\\n\\n\\ndef evaluate(train: pd.DataFrame, blocks: dict[str, np.ndarray], plan: pd.DataFrame, current_oof: pd.DataFrame,\\n             current_cache: pd.DataFrame, output_dir: str | Path, *, frozen: neural.FrozenConfig | None = None,\\n             samples: int = 10000, seed: int = 2026, provenance: dict | None = None) -> dict:\\n    frozen = frozen or neural.FrozenConfig()\\n    plan = validate_plan(train, plan, AdapterConfig())\\n    output = Path(output_dir)\\n    if (output / \\"comparison.json\\").exists():\\n        raise ValueError(\\"Use a new output directory; an existing comparison must not be overwritten\\")\\n    if set(current_cache[\\"groups_sha256\\"]) != {groups_sha256(train)}:\\n        raise ValueError(\\"Cached current-system inner predictions were built under a different row grouping\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    truth, prompts = ensemble._targets(train), train[\\"prompt_id\\"].to_numpy()\\n    folds = plan[\\"outer_fold\\"].to_numpy()\\n    current = _aligned_current(current_oof, train, folds)\\n    in_group = np.isin(truth, INDEX)\\n    probabilities, selection, specialist_only = {\\"current\\": current}, {}, {}\\n    for name in FEATURE_SETS:\\n        system, specialist, chosen = np.empty_like(current), np.empty((len(train), len(SPECIALIST))), []\\n        for fold in range(5):\\n            column = plan[f\\"inner_{fold}\\"].to_numpy()\\n            scores = {weight: [] for weight in WEIGHTS}\\n            for inner_fold, (valid, inner_current) in _inner_current(current_cache, train, plan, fold).items():\\n                fit = np.flatnonzero((folds != fold) & (column != inner_fold))\\n                inner_specialist = specialist_probabilities(blocks, name, truth, fit, valid, frozen)\\n                for weight in WEIGHTS:\\n                    blended = redistribute(inner_current, inner_specialist, weight)\\n                    scores[weight].append(float(balanced_accuracy_score(truth[valid], blended.argmax(axis=1))))\\n            means = {weight: float(np.mean(values)) for weight, values in scores.items()}\\n            # Ties keep the smaller specialist weight, i.e. stay with the current system.\\n            weight = first_best(WEIGHTS, means.__getitem__)\\n            rows = np.flatnonzero(folds == fold)\\n            specialist[rows] = specialist_probabilities(blocks, name, truth, np.flatnonzero(folds != fold), rows,\\n                                                        frozen)\\n            system[rows] = redistribute(current[rows], specialist[rows], weight)\\n            current_weights = (current_cache[current_cache[\\"outer_fold\\"] == fold]\\n                               .groupby(\\"inner_fold\\")[\\"weight\\"].first().tolist())\\n            chosen.append({\\"outer_fold\\": fold, \\"weight\\": weight, \\"current_inner_weight\\": current_weights,\\n                           \\"inner_balanced_accuracy\\": {str(key): values for key, values in scores.items()}})\\n        probabilities[f\\"current+{name}\\"] = system\\n        selection[name] = chosen\\n        # Seven-way separation on the true Claude and GLM rows, against the current system\'s own.\\n        rows = np.flatnonzero(in_group)\\n        specialist_only[name] = float(balanced_accuracy_score(truth[rows], INDEX[specialist[rows].argmax(axis=1)]))\\n        np.save(output / f\\"specialist-{name}.npy\\", specialist)\\n    group_rows = np.flatnonzero(in_group)\\n    specialist_only[\\"current\\"] = float(balanced_accuracy_score(\\n        truth[group_rows], INDEX[current[group_rows][:, INDEX].argmax(axis=1)]))\\n    predictions = {name: values.argmax(axis=1) for name, values in probabilities.items()}\\n    systems = {name: ensemble._system_metrics(truth, values, folds) for name, values in predictions.items()}\\n    paired = {f\\"{name}_vs_current\\": ensemble.prompt_bootstrap(truth, predictions[\\"current\\"], values, prompts,\\n                                                              samples, seed)\\n              for name, values in predictions.items() if name != \\"current\\"}\\n    paired[\\"current+qwen+styledistance_vs_current+qwen\\"] = ensemble.prompt_bootstrap(\\n        truth, predictions[\\"current+qwen\\"], predictions[\\"current+qwen+styledistance\\"], prompts, samples, seed)\\n    changes = {}\\n    for name, values in predictions.items():\\n        if name == \\"current\\":\\n            continue\\n        gained = (predictions[\\"current\\"] != truth) & (values == truth)\\n        lost = (predictions[\\"current\\"] == truth) & (values != truth)\\n        changes[name] = {\\"gained\\": int(gained.sum()), \\"lost\\": int(lost.sum()),\\n                         \\"by_class\\": {label: {\\"gained\\": int(gained[truth == index].sum()),\\n                                              \\"lost\\": int(lost[truth == index].sum())}\\n                                      for index, label in enumerate(LABELS)\\n                                      if gained[truth == index].any() or lost[truth == index].any()}}\\n    frame = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].rename(columns={\\"model\\": \\"true_model\\"}).assign(outer_fold=folds)\\n    for name, values in predictions.items():\\n        frame[f\\"prediction::{name}\\"] = np.asarray(LABELS)[values]\\n    frame.to_parquet(output / \\"predictions.parquet\\", index=False)\\n    result = {\\"specialist_classes\\": list(SPECIALIST), \\"weights\\": list(WEIGHTS), \\"systems\\": systems,\\n              \\"paired\\": paired, \\"selection\\": selection, \\"changes\\": changes,\\n              \\"seven_class_balanced_accuracy_on_true_rows\\": specialist_only,\\n              \\"groups\\": int(train[\\"prompt_id\\"].nunique()), \\"groups_sha256\\": groups_sha256(train),\\n              \\"train_fingerprint\\": fingerprint_frame(train), \\"provenance\\": provenance or {}}\\n    _json(output / \\"comparison.json\\", result)\\n    return result\\n\\n\\ndef recipe_sha256(recipe: dict) -> str:\\n    \\"\\"\\"Identity of the current system\'s frozen recipe, as parsed JSON.\\"\\"\\"\\n    return hashlib.sha256(json.dumps(recipe, sort_keys=True).encode()).hexdigest()\\n\\n\\ndef freeze_recipe(train: pd.DataFrame, blocks: dict[str, np.ndarray], final_plan: pd.DataFrame,\\n                  current_recipe_sha256: str, *, frozen: neural.FrozenConfig | None = None,\\n                  blend: ensemble.BlendConfig | None = None, validation: dict | None = None) -> dict:\\n    \\"\\"\\"Choose the final specialist weight once, on inner prompt folds over every official row.\\"\\"\\"\\n    frozen, blend = frozen or neural.FrozenConfig(), blend or ensemble.BlendConfig()\\n    inner = _aligned_plan(train, final_plan, AdapterConfig())\\n    truth = ensemble._targets(train)\\n    scores, current_weights = {weight: [] for weight in WEIGHTS}, []\\n    for inner_fold in sorted(set(inner)):\\n        fit, valid = np.flatnonzero(inner != inner_fold), np.flatnonzero(inner == inner_fold)\\n        current, current_weight = current_on_split(train, blocks[\\"qwen\\"], fit, valid, blend,\\n                                                   FINAL_SEED_OFFSET + inner_fold)\\n        current_weights.append(current_weight)\\n        specialist = specialist_probabilities(blocks, FINAL_FEATURE_SET, truth, fit, valid, frozen)\\n        for weight in WEIGHTS:\\n            predictions = redistribute(current, specialist, weight).argmax(axis=1)\\n            scores[weight].append(float(balanced_accuracy_score(truth[valid], predictions)))\\n    means = {weight: float(np.mean(values)) for weight, values in scores.items()}\\n    weight = first_best(WEIGHTS, means.__getitem__)\\n    styledistance = ENCODERS[\\"styledistance\\"]\\n    return {\\"schema_version\\": 1, \\"kind\\": RECIPE_KIND, \\"labels\\": list(LABELS),\\n            \\"specialist_classes\\": list(SPECIALIST), \\"feature_set\\": FINAL_FEATURE_SET, \\"weight\\": weight,\\n            \\"inner_balanced_accuracy_by_weight\\": {str(key): values for key, values in scores.items()},\\n            \\"current_inner_weights\\": current_weights, \\"C\\": frozen.C, \\"seed\\": frozen.seed,\\n            \\"qwen_model_id\\": neural.MODEL_ID, \\"qwen_model_revision\\": neural.MODEL_REVISION,\\n            \\"styledistance_model_id\\": styledistance[\\"model_id\\"],\\n            \\"styledistance_model_revision\\": styledistance[\\"model_revision\\"], \\"styledistance_max_length\\": 512,\\n            \\"current_recipe_sha256\\": current_recipe_sha256, \\"train_fingerprint\\": fingerprint_frame(train),\\n            \\"nested_validation\\": validation or {}}\\n\\n\\ndef _checked_recipe(recipe: dict) -> dict:\\n    styledistance = ENCODERS[\\"styledistance\\"]\\n    if (recipe.get(\\"schema_version\\") != 1 or recipe.get(\\"kind\\") != RECIPE_KIND\\n            or recipe.get(\\"labels\\") != list(LABELS) or recipe.get(\\"specialist_classes\\") != list(SPECIALIST)\\n            or recipe.get(\\"feature_set\\") != FINAL_FEATURE_SET or recipe.get(\\"weight\\") not in WEIGHTS\\n            or recipe.get(\\"styledistance_model_revision\\") != styledistance[\\"model_revision\\"]\\n            or recipe.get(\\"qwen_model_revision\\") != neural.MODEL_REVISION):\\n        raise ValueError(\\"Unsupported Claude-GLM specialist recipe\\")\\n    return recipe\\n\\n\\ndef fit_final(train: pd.DataFrame, blend_dir: str | Path, styledistance_path: str | Path, recipe: dict,\\n              output_dir: str | Path, device: str = \\"auto\\") -> dict:\\n    \\"\\"\\"Fit the specialist on every official Claude and GLM row: the blend fit\'s own Qwen\\n    features, and StyleDistance extracted here.\\"\\"\\"\\n    recipe = _checked_recipe(recipe)\\n    if recipe[\\"train_fingerprint\\"] != fingerprint_frame(train):\\n        raise ValueError(\\"Training data differ from the rows used to freeze the specialist recipe\\")\\n    blend = joblib.load(Path(blend_dir) / \\"blend.joblib\\")  # trusted output of `ntl.ensemble fit` in this run\\n    if recipe_sha256(blend[\\"recipe\\"]) != recipe[\\"current_recipe_sha256\\"]:\\n        raise ValueError(\\"The fitted current system differs from the one the specialist was chosen for\\")\\n    output = Path(output_dir)\\n    if (output / \\"specialist.joblib\\").exists():\\n        raise ValueError(\\"Use a new output directory; a fitted specialist must not be overwritten\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    started = time.perf_counter()\\n    _, _, qwen_config = ensemble.load_recipe(blend[\\"recipe\\"])\\n    qwen, _ = ensemble.load_archive(Path(blend_dir) / \\"train-features.npz\\", train, qwen_config)\\n    styledistance, profile = extract_embeddings(train[\\"text\\"].tolist(), styledistance_path, \\"styledistance\\",\\n                                                device=_device(device))\\n    frozen = neural.FrozenConfig(C=recipe[\\"C\\"], seed=recipe[\\"seed\\"])\\n    blocks = {\\"qwen\\": qwen, \\"styledistance\\": styledistance}\\n    truth = ensemble._targets(train)\\n    names = FEATURE_SETS[FINAL_FEATURE_SET]\\n    rows = np.flatnonzero(np.isin(truth, INDEX))\\n    head = specialist_head(tuple(blocks[name].shape[1] for name in names), frozen)\\n    with threadpool_limits(limits=2):\\n        head.fit(np.hstack([blocks[name] for name in names])[rows], _POSITION[truth[rows]])\\n    if not np.array_equal(head.classes_, np.arange(len(SPECIALIST))):\\n        raise ValueError(\\"Specialist head must contain every specialist class\\")\\n    metadata = {\\"weight\\": recipe[\\"weight\\"], \\"train_fingerprint\\": fingerprint_frame(train),\\n                \\"specialist_rows\\": len(rows), \\"styledistance_profile\\": profile[\\"profile\\"],\\n                \\"fit_seconds\\": time.perf_counter() - started,\\n                \\"versions\\": {package: importlib.metadata.version(package)\\n                             for package in (\\"numpy\\", \\"scikit-learn\\", \\"scipy\\")}}\\n    # joblib is trusted project output; never load a third-party pickle with predict_final.\\n    joblib.dump({\\"head\\": head, \\"recipe\\": recipe, \\"metadata\\": metadata}, output / \\"specialist.joblib\\")\\n    _json(output / \\"fit.json\\", metadata)\\n    return metadata\\n\\n\\ndef _device(device: str) -> str:\\n    if device != \\"auto\\":\\n        return device\\n    import torch\\n\\n    return \\"cuda:0\\" if torch.cuda.is_available() else \\"cpu\\"\\n\\n\\ndef predict_final(specialist_dir: str | Path, blend_dir: str | Path, test: pd.DataFrame, qwen_path: str | Path,\\n                  styledistance_path: str | Path, output: str | Path, device: str = \\"auto\\") -> pd.DataFrame:\\n    \\"\\"\\"Score every current test row: the current blend\'s probabilities, redistributed on the\\n    Claude and GLM classes by the specialist.\\"\\"\\"\\n    specialist = joblib.load(Path(specialist_dir) / \\"specialist.joblib\\")  # trusted output of `fit` in this run\\n    recipe = _checked_recipe(specialist[\\"recipe\\"])\\n    blend = joblib.load(Path(blend_dir) / \\"blend.joblib\\")\\n    if recipe_sha256(blend[\\"recipe\\"]) != recipe[\\"current_recipe_sha256\\"]:\\n        raise ValueError(\\"The fitted current system differs from the one the specialist was chosen for\\")\\n    blend_weight, _, qwen_config = ensemble.load_recipe(blend[\\"recipe\\"])\\n    started = time.perf_counter()\\n    qwen_config = neural.FrozenConfig(**{**asdict(qwen_config), \\"device\\": device})\\n    qwen, qwen_profile = neural.extract_features(test[\\"text\\"].tolist(), qwen_path, qwen_config)\\n    styledistance, styledistance_profile = extract_embeddings(test[\\"text\\"].tolist(), styledistance_path,\\n                                                              \\"styledistance\\", device=_device(device))\\n    with threadpool_limits(limits=2):\\n        cpu = aligned_probabilities(blend[\\"cpu\\"], test[\\"text\\"])\\n        current = ensemble._blend(cpu, neural._probabilities(blend[\\"qwen_head\\"], qwen), blend_weight)\\n        q = specialist[\\"head\\"].predict_proba(np.hstack([qwen, styledistance]))\\n    if not np.array_equal(specialist[\\"head\\"].classes_, np.arange(len(SPECIALIST))):\\n        raise ValueError(\\"Specialist head must contain every specialist class\\")\\n    final = redistribute(current, q, recipe[\\"weight\\"])\\n    if not np.isfinite(final).all() or not np.allclose(final.sum(axis=1), 1.0):\\n        raise ValueError(\\"Final scores must be finite probability vectors\\")\\n    submission = write_submission(test[\\"id\\"], np.asarray(LABELS)[final.argmax(axis=1)].tolist(), output)\\n    output = Path(output)\\n    _json(output.with_name(f\\"{output.stem}_profile.json\\"),\\n          {\\"rows\\": len(test), \\"weight\\": recipe[\\"weight\\"], \\"blend_weight\\": blend_weight,\\n           \\"changed_from_current\\": int((final.argmax(axis=1) != current.argmax(axis=1)).sum()),\\n           \\"elapsed_seconds\\": time.perf_counter() - started, \\"qwen_profile\\": qwen_profile,\\n           \\"styledistance_profile\\": styledistance_profile[\\"profile\\"]})\\n    return submission\\n\\n\\ndef _load_blocks(train: pd.DataFrame, qwen_path: Path, styledistance_path: Path) -> dict[str, np.ndarray]:\\n    qwen, _ = ensemble.load_archive(qwen_path, train, neural.FrozenConfig())\\n    styledistance, metadata = load_feature_archive(styledistance_path, train)\\n    if metadata.get(\\"model_id\\") != \\"StyleDistance/styledistance\\":\\n        raise ValueError(\\"Expected StyleDistance features\\")\\n    return {\\"qwen\\": qwen, \\"styledistance\\": styledistance}\\n\\n\\ndef _sha256(path: Path) -> str:\\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\\n\\n\\ndef _grouped(train: pd.DataFrame, plan: pd.DataFrame, groups: Path | None) -> tuple[pd.DataFrame, pd.DataFrame]:\\n    \\"\\"\\"Training rows and plan under the chosen grouping (prompts unless `groups` is given).\\"\\"\\"\\n    if groups is None:\\n        return train, plan\\n    train = regroup(train, pd.read_csv(groups, dtype={\\"id\\": str, \\"prompt_id\\": str}, keep_default_na=False))\\n    return train, plan.assign(prompt_id=plan[\\"id\\"].map(train.set_index(\\"id\\")[\\"prompt_id\\"]))\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    commands = parser.add_subparsers(dest=\\"command\\", required=True)\\n    features = commands.add_parser(\\"features\\", help=\\"Frozen StyleDistance embeddings for every training row\\")\\n    features.add_argument(\\"--train\\", type=Path, required=True)\\n    features.add_argument(\\"--model-path\\", type=Path, required=True)\\n    features.add_argument(\\"--output\\", type=Path, required=True)\\n    features.add_argument(\\"--device\\", default=\\"cpu\\")\\n    inner = commands.add_parser(\\"current-inner\\", help=\\"Refit the current system on one outer fold\'s inner splits\\")\\n    inner.add_argument(\\"--train\\", type=Path, required=True)\\n    inner.add_argument(\\"--plan\\", type=Path, required=True)\\n    inner.add_argument(\\"--qwen-features\\", type=Path, required=True)\\n    inner.add_argument(\\"--outer-fold\\", type=int, required=True)\\n    inner.add_argument(\\"--output\\", type=Path, required=True)\\n    inner.add_argument(\\"--groups\\", type=Path, help=\\"CSV of id, prompt_id regrouping rows (e.g. by topic)\\")\\n    compare = commands.add_parser(\\"evaluate\\", help=\\"Control and candidate specialists against the current system\\")\\n    compare.add_argument(\\"--train\\", type=Path, required=True)\\n    compare.add_argument(\\"--plan\\", type=Path, required=True)\\n    compare.add_argument(\\"--qwen-features\\", type=Path, required=True)\\n    compare.add_argument(\\"--styledistance-features\\", type=Path, required=True)\\n    compare.add_argument(\\"--current-oof\\", type=Path, required=True)\\n    compare.add_argument(\\"--current-inner\\", type=Path, nargs=\\"+\\", required=True)\\n    compare.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    compare.add_argument(\\"--groups\\", type=Path, help=\\"CSV of id, prompt_id regrouping rows (e.g. by topic)\\")\\n    freeze = commands.add_parser(\\"freeze\\", help=\\"Choose the final specialist weight on full-data inner folds\\")\\n    freeze.add_argument(\\"--train\\", type=Path, required=True)\\n    freeze.add_argument(\\"--final-plan\\", type=Path, required=True)\\n    freeze.add_argument(\\"--qwen-features\\", type=Path, required=True)\\n    freeze.add_argument(\\"--styledistance-features\\", type=Path, required=True)\\n    freeze.add_argument(\\"--current-recipe\\", type=Path, required=True)\\n    freeze.add_argument(\\"--validation\\", type=Path, nargs=\\"+\\", default=[], help=\\"comparison.json files\\")\\n    freeze.add_argument(\\"--output\\", type=Path, required=True)\\n    fit = commands.add_parser(\\"fit\\", help=\\"Fit the specialist on every official Claude and GLM row\\")\\n    fit.add_argument(\\"--train\\", type=Path, required=True)\\n    fit.add_argument(\\"--blend-dir\\", type=Path, required=True)\\n    fit.add_argument(\\"--styledistance-path\\", type=Path, required=True)\\n    fit.add_argument(\\"--recipe\\", type=Path, required=True)\\n    fit.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    fit.add_argument(\\"--device\\", default=\\"auto\\")\\n    predict = commands.add_parser(\\"predict\\", help=\\"Score test.csv with the current system and the specialist\\")\\n    predict.add_argument(\\"--specialist-dir\\", type=Path, required=True)\\n    predict.add_argument(\\"--blend-dir\\", type=Path, required=True)\\n    predict.add_argument(\\"--test\\", type=Path, required=True)\\n    predict.add_argument(\\"--model-path\\", type=Path, required=True)\\n    predict.add_argument(\\"--styledistance-path\\", type=Path, required=True)\\n    predict.add_argument(\\"--output\\", type=Path, required=True)\\n    predict.add_argument(\\"--device\\", default=\\"auto\\")\\n    args = parser.parse_args(argv)\\n    if args.command == \\"predict\\":\\n        predict_final(args.specialist_dir, args.blend_dir, read_test(args.test), args.model_path,\\n                      args.styledistance_path, args.output, args.device)\\n        return 0\\n    train = read_train(args.train)\\n    if args.command == \\"fit\\":\\n        print(json.dumps(fit_final(train, args.blend_dir, args.styledistance_path,\\n                                   json.loads(args.recipe.read_text()), args.output_dir, args.device), indent=2))\\n        return 0\\n    if args.command == \\"freeze\\":\\n        if args.output.exists():\\n            raise ValueError(\\"Use a new output path; a frozen recipe must not be overwritten\\")\\n        blocks = _load_blocks(train, args.qwen_features, args.styledistance_features)\\n        final_plan = pd.read_csv(args.final_plan, dtype={\\"id\\": str, \\"prompt_id\\": str}, keep_default_na=False)\\n        validation = {}\\n        for path in args.validation:\\n            result = json.loads(path.read_text())\\n            paired = result[\\"paired\\"][f\\"current+{FINAL_FEATURE_SET}_vs_current\\"]\\n            validation[str(path)] = {\\n                \\"groups\\": result[\\"groups\\"],\\n                \\"balanced_accuracy\\": {name: values[\\"balanced_accuracy\\"] for name, values in result[\\"systems\\"].items()},\\n                \\"delta\\": paired[\\"balanced_accuracy_delta\\"], \\"delta_95_interval\\": paired[\\"delta_95_interval\\"],\\n                \\"selected_weight_per_outer_fold\\": [item[\\"weight\\"] for item in result[\\"selection\\"][FINAL_FEATURE_SET]]}\\n        recipe = freeze_recipe(train, blocks, final_plan,\\n                               recipe_sha256(json.loads(args.current_recipe.read_text())), validation=validation)\\n        args.output.parent.mkdir(parents=True, exist_ok=True)\\n        _json(args.output, recipe)\\n        print(json.dumps({key: recipe[key] for key in (\\"weight\\", \\"inner_balanced_accuracy_by_weight\\",\\n                                                       \\"current_inner_weights\\")}, indent=2))\\n        return 0\\n    if args.command == \\"features\\":\\n        if args.output.exists():\\n            raise ValueError(\\"Feature output already exists; preserve it or choose another output path\\")\\n        embeddings, metadata = extract_embeddings(train[\\"text\\"].tolist(), args.model_path, \\"styledistance\\",\\n                                                  device=args.device)\\n        metadata[\\"train_fingerprint\\"] = fingerprint_frame(train)\\n        saved = save_feature_archive(args.output, train[\\"id\\"].tolist(), embeddings, metadata,\\n                                     texts=train[\\"text\\"].tolist())\\n        args.output.with_suffix(\\".json\\").write_text(json.dumps(saved, indent=2) + \\"\\\\n\\")\\n        print(json.dumps(metadata[\\"profile\\"], indent=2))\\n        return 0\\n    plan = pd.read_csv(args.plan, dtype={\\"id\\": str, \\"prompt_id\\": str}, keep_default_na=False)\\n    if args.command == \\"current-inner\\":\\n        if args.output.exists():\\n            raise ValueError(\\"Use a new output path; cached inner predictions must not be overwritten\\")\\n        qwen, _ = ensemble.load_archive(args.qwen_features, train, neural.FrozenConfig())\\n        train, plan = _grouped(train, plan, args.groups)\\n        args.output.parent.mkdir(parents=True, exist_ok=True)\\n        build_current_inner(train, qwen, plan, args.outer_fold).to_parquet(args.output, index=False)\\n        return 0\\n    blocks = _load_blocks(train, args.qwen_features, args.styledistance_features)\\n    train, plan = _grouped(train, plan, args.groups)\\n    cache = pd.concat([pd.read_parquet(path) for path in args.current_inner], ignore_index=True)\\n    inputs = [args.plan, args.qwen_features, args.styledistance_features, args.current_oof, *args.current_inner]\\n    provenance = {str(path): _sha256(path) for path in inputs + ([args.groups] if args.groups else [])}\\n    result = evaluate(train, blocks, plan, pd.read_parquet(args.current_oof), cache, args.output_dir,\\n                      provenance={\\"input_sha256\\": provenance})\\n    print(json.dumps({\\"balanced_accuracy\\": {name: round(values[\\"balanced_accuracy\\"], 4)\\n                                            for name, values in result[\\"systems\\"].items()},\\n                      \\"weights\\": {name: [item[\\"weight\\"] for item in items]\\n                                  for name, items in result[\\"selection\\"].items()},\\n                      \\"seven_class\\": {name: round(value, 4) for name, value in\\n                                      result[\\"seven_class_balanced_accuracy_on_true_rows\\"].items()},\\n                      \\"paired\\": {name: [round(values[\\"balanced_accuracy_delta\\"], 4)]\\n                                 + [round(bound, 4) for bound in values[\\"delta_95_interval\\"]]\\n                                 for name, values in result[\\"paired\\"].items()},\\n                      \\"changes\\": {name: [values[\\"gained\\"], values[\\"lost\\"]]\\n                                  for name, values in result[\\"changes\\"].items()}}, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    # Run the imported module, not this __main__ copy, so the fitted head pickles its\\n    # block-weight function as ntl.claude_glm._column_weights and loads from any entry point.\\n    from ntl.claude_glm import main as imported_main\\n\\n    raise SystemExit(imported_main())\\n", "ntl/cli.py": "\\"\\"\\"Command-line entry points; no credentials, network calls or submissions.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport importlib.metadata\\nimport json\\nimport platform\\nimport sys\\nfrom pathlib import Path\\n\\nfrom .constants import DEFAULT_SEED\\n\\n\\ndef _model_arguments(parser: argparse.ArgumentParser) -> None:\\n    parser.add_argument(\\"--train\\", required=True, type=Path)\\n    parser.add_argument(\\"--output-dir\\", required=True, type=Path)\\n    parser.add_argument(\\"--seed\\", type=int, default=DEFAULT_SEED)\\n    parser.add_argument(\\"--char-max-features\\", type=int, default=50000)\\n    parser.add_argument(\\"--word-max-features\\", type=int, default=30000)\\n    parser.add_argument(\\"--C\\", type=float, default=4.0)\\n    parser.add_argument(\\"--max-iter\\", type=int, default=1000)\\n\\n\\ndef parser() -> argparse.ArgumentParser:\\n    result = argparse.ArgumentParser(description=\\"Name That LLM: offline data checks and CPU baseline\\")\\n    subcommands = result.add_subparsers(dest=\\"command\\", required=True)\\n    subcommands.add_parser(\\"doctor\\", help=\\"Report interpreter and dependency versions\\")\\n    check = subcommands.add_parser(\\"data-check\\", help=\\"Validate official data schemas without altering text\\")\\n    check.add_argument(\\"--train\\", required=True, type=Path)\\n    check.add_argument(\\"--test\\", type=Path)\\n    cv = subcommands.add_parser(\\"baseline-cv\\", help=\\"Evaluate with prompt-grouped folds and save OOF predictions\\")\\n    _model_arguments(cv)\\n    cv.add_argument(\\"--folds\\", type=int, default=5)\\n    cv.add_argument(\\"--fold-manifest\\", type=Path)\\n    training = subcommands.add_parser(\\"baseline-train\\", help=\\"Fit on all labeled data and save an inference artifact\\")\\n    _model_arguments(training)\\n    predict = subcommands.add_parser(\\"predict\\", help=\\"Write a validated submission.parquet from a trusted local model\\")\\n    predict.add_argument(\\"--model\\", required=True, type=Path)\\n    predict.add_argument(\\"--test\\", required=True, type=Path)\\n    predict.add_argument(\\"--output\\", required=True, type=Path)\\n    smoke = subcommands.add_parser(\\"smoke\\", help=\\"Exercise the pipeline on clearly synthetic fixtures\\")\\n    smoke.add_argument(\\"--output-dir\\", required=True, type=Path)\\n    return result\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    args = parser().parse_args(argv)\\n    try:\\n        if args.command == \\"doctor\\":\\n            versions = {}\\n            for package in (\\"pandas\\", \\"numpy\\", \\"scipy\\", \\"scikit-learn\\", \\"joblib\\", \\"pyarrow\\"):\\n                try:\\n                    versions[package] = importlib.metadata.version(package)\\n                except importlib.metadata.PackageNotFoundError:\\n                    versions[package] = None\\n            print(json.dumps({\\"python\\": platform.python_version(), \\"executable\\": sys.executable,\\n                              \\"dependencies\\": versions}, indent=2))\\n            return int(any(value is None for value in versions.values()))\\n        if args.command == \\"data-check\\":\\n            from .data import data_quality_report, fingerprint_frame, read_test, read_train\\n            train = read_train(args.train)\\n            summary = {\\"training_rows\\": len(train), \\"prompts\\": int(train[\\"prompt_id\\"].nunique()),\\n                       \\"class_counts\\": train[\\"model\\"].value_counts().sort_index().to_dict(),\\n                       \\"train_fingerprint\\": fingerprint_frame(train),\\n                       \\"data_quality\\": data_quality_report(train)}\\n            if args.test:\\n                test = read_test(args.test)\\n                if set(train[\\"id\\"]) & set(test[\\"id\\"]):\\n                    raise ValueError(\\"Train and test passage ids overlap\\")\\n                summary[\\"test_rows\\"] = len(test)\\n            print(json.dumps(summary, indent=2))\\n            return 0\\n        if args.command == \\"smoke\\":\\n            from .smoke import run_smoke\\n            summary = run_smoke(args.output_dir)\\n        elif args.command in {\\"baseline-cv\\", \\"baseline-train\\"}:\\n            from .data import read_train\\n            from .modeling import BaselineConfig, run_cross_validation, train_baseline\\n            config = BaselineConfig(seed=args.seed, char_max_features=args.char_max_features,\\n                                    word_max_features=args.word_max_features, C=args.C, max_iter=args.max_iter)\\n            train = read_train(args.train)\\n            if args.command == \\"baseline-cv\\":\\n                summary = run_cross_validation(train, args.output_dir, n_splits=args.folds,\\n                                               config=config, fold_manifest=args.fold_manifest)\\n            else:\\n                summary = {\\"model_path\\": str(train_baseline(train, args.output_dir, config=config))}\\n        else:\\n            from .data import read_test\\n            from .modeling import predict_baseline\\n            submission = predict_baseline(args.model, read_test(args.test), args.output)\\n            summary = {\\"submission_path\\": str(args.output), \\"rows\\": len(submission)}\\n        print(json.dumps(summary, indent=2))\\n        return 0\\n    except (ValueError, OSError, ImportError) as error:\\n        print(f\\"Error: {error}\\", file=sys.stderr)\\n        return 2\\n", "ntl/constants.py": "\\"\\"\\"Competition labels, in an explicit order shared by every model artifact.\\"\\"\\"\\n\\nLABELS = (\\n    \\"gpt-5.5\\",\\n    \\"gpt-5.6-luna\\",\\n    \\"gpt-5.6-sol\\",\\n    \\"gpt-5.6-terra\\",\\n    \\"gpt-6-astra\\",\\n    \\"gemini-3.1-pro-preview\\",\\n    \\"gemini-3.6-flash\\",\\n    \\"gemini-3.7-flash\\",\\n    \\"gemini-3.8-flash\\",\\n    \\"claude-fable-5\\",\\n    \\"claude-opus-4-6\\",\\n    \\"claude-opus-4-8\\",\\n    \\"claude-opus-5\\",\\n    \\"claude-sonnet-5\\",\\n    \\"qwen3.8-max\\",\\n    \\"glm-5.3\\",\\n    \\"glm-5.3-flash\\",\\n    \\"kimi-k3\\",\\n)\\n\\nDEFAULT_SEED = 2026\\n", "ntl/cpu_gemini.py": "\\"\\"\\"Four-version CPU specialist, alone and blended with frozen StyleDistance scores.\\n\\nThe control refits the 18-class reference\'s exact character, word and style features\\nand class-weighted logistic regression on the true-Gemini outer-fitting rows only.\\nThe blend averages its probabilities with a StyleDistance readout. The readout\'s C and\\nthe blend weight are chosen jointly by the frozen experiment\'s inner prompt folds;\\neach outer validation partition is predicted once. End-to-end scores substitute the\\nspecialist\'s version into the saved 18-class router predictions on all 2,699 rows.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport importlib.metadata\\nimport json\\nimport time\\nfrom dataclasses import asdict, dataclass, field\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom .constants import LABELS\\nfrom .data import read_train\\nfrom .folds import inner_prompt_splits\\nfrom .frozen_gemini import (\\n    FrozenGeminiConfig,\\n    _aligned_probabilities,\\n    _pipeline,\\n    _validate_frame,\\n    embeddings_sha256,\\n    load_feature_archive,\\n)\\nfrom .gemini import (\\n    GEMINI_LABELS,\\n    OFFICIAL_BASELINE_COUNTS,\\n    _cohort_metrics,\\n    _save_class_reports,\\n    prepare_gemini_evaluation,\\n)\\nfrom .modeling import BaselineConfig, build_pipeline\\nfrom .selection import first_best\\n\\nCONTROL = \\"cpu-specialist\\"\\nBLEND = \\"cpu-styledistance-blend\\"\\n\\n\\n@dataclass(frozen=True)\\nclass SpecialistConfig:\\n    cpu: BaselineConfig = field(default_factory=BaselineConfig)\\n    readout: FrozenGeminiConfig = field(default_factory=FrozenGeminiConfig)\\n    weight_grid: tuple[float, ...] = tuple(step / 10 for step in range(11))\\n    bootstrap_samples: int = 10000\\n\\n    def validate(self) -> None:\\n        self.readout.validate()\\n        if not self.weight_grid or len(set(self.weight_grid)) != len(self.weight_grid):\\n            raise ValueError(\\"Blend weight grid must be nonempty and contain no duplicates\\")\\n        if any(not 0 <= weight <= 1 for weight in self.weight_grid):\\n            raise ValueError(\\"Blend weights must lie in [0, 1]\\")\\n        if self.bootstrap_samples < 1:\\n            raise ValueError(\\"Bootstrap sample count must be positive\\")\\n\\n\\ndef _json(path: Path, value: dict | list) -> None:\\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef _cpu_probabilities(fit_texts: np.ndarray, fit_targets: np.ndarray, texts: np.ndarray,\\n                       config: BaselineConfig) -> np.ndarray:\\n    # The reference\'s features, C and class weighting; only the label set differs.\\n    pipeline = build_pipeline(config)\\n    pipeline.fit(fit_texts, fit_targets)\\n    return _aligned_probabilities(pipeline, texts)\\n\\n\\ndef _blend(cpu: np.ndarray, readout: np.ndarray, weight: float) -> np.ndarray:\\n    return (1 - weight) * cpu + weight * readout\\n\\n\\ndef _select_blend(\\n    texts: np.ndarray, features: np.ndarray, targets: np.ndarray, prompts: np.ndarray,\\n    config: SpecialistConfig, outer_fold: int,\\n) -> tuple[float, float, dict]:\\n    \\"\\"\\"Choose readout C and weight from inner prompt folds; no outer rows are passed in.\\"\\"\\"\\n    splits, split_metadata = inner_prompt_splits(targets, prompts, config.readout.inner_folds,\\n                                                 config.readout.seed + outer_fold)\\n    cpu, readout = [], {C: [] for C in config.readout.C_grid}\\n    for fit, valid in splits:\\n        cpu.append(_cpu_probabilities(texts[fit], targets[fit], texts[valid], config.cpu))\\n        for C in config.readout.C_grid:\\n            estimator = _pipeline(C, config.readout)\\n            estimator.fit(features[fit], targets[fit])\\n            readout[C].append(_aligned_probabilities(estimator, features[valid]))\\n    candidates = []\\n    for C in sorted(config.readout.C_grid):\\n        for weight in sorted(config.weight_grid):\\n            scores = [float(balanced_accuracy_score(targets[valid], _blend(p_cpu, p_readout, weight).argmax(axis=1)))\\n                      for (_, valid), p_cpu, p_readout in zip(splits, cpu, readout[C], strict=True)]\\n            candidates.append({\\"C\\": C, \\"weight\\": weight, \\"inner_balanced_accuracy\\": scores,\\n                               \\"mean_inner_balanced_accuracy\\": float(np.mean(scores))})\\n    # Ties prefer the smaller StyleDistance weight (closer to the control), then smaller C.\\n    best = first_best(sorted(candidates, key=lambda item: (item[\\"weight\\"], item[\\"C\\"])),\\n                      lambda item: item[\\"mean_inner_balanced_accuracy\\"])\\n    control_scores = [float(balanced_accuracy_score(targets[valid], p_cpu.argmax(axis=1)))\\n                      for (_, valid), p_cpu in zip(splits, cpu, strict=True)]\\n    return best[\\"C\\"], best[\\"weight\\"], {\\n        \\"outer_fold\\": outer_fold, \\"selected_C\\": best[\\"C\\"], \\"selected_weight\\": best[\\"weight\\"],\\n        \\"selected_mean_inner_balanced_accuracy\\": best[\\"mean_inner_balanced_accuracy\\"],\\n        \\"control_inner_balanced_accuracy\\": control_scores, \\"candidates\\": candidates,\\n        \\"inner_splits\\": split_metadata,\\n        \\"selection_metric\\": \\"Unweighted mean inner-fold balanced accuracy; ties choose smaller weight, then smaller C.\\",\\n    }\\n\\n\\ndef _paired(frame: pd.DataFrame, first: np.ndarray, second: np.ndarray, samples: int, seed: int) -> dict:\\n    \\"\\"\\"Paired accuracy change, with a bootstrap that resamples whole prompts.\\"\\"\\"\\n    first, second = np.asarray(first, dtype=bool), np.asarray(second, dtype=bool)\\n    _, prompt_index = np.unique(frame[\\"prompt_id\\"].to_numpy(), return_inverse=True)\\n    prompt_delta = np.bincount(prompt_index, weights=second.astype(float) - first.astype(float))\\n    prompt_rows = np.bincount(prompt_index)\\n    draws = np.random.default_rng(seed).integers(0, len(prompt_rows), size=(samples, len(prompt_rows)))\\n    deltas = prompt_delta[draws].sum(axis=1) / prompt_rows[draws].sum(axis=1)\\n    return {\\n        \\"rows\\": len(frame), \\"prompts\\": len(prompt_rows),\\n        \\"first_correct\\": int(first.sum()), \\"second_correct\\": int(second.sum()),\\n        \\"accuracy_delta\\": float(second.mean() - first.mean()),\\n        \\"both_correct\\": int((first & second).sum()), \\"second_only_correct\\": int((~first & second).sum()),\\n        \\"first_only_correct\\": int((first & ~second).sum()), \\"both_wrong\\": int((~first & ~second).sum()),\\n        \\"prompt_bootstrap\\": {\\"samples\\": samples, \\"seed\\": seed,\\n                             \\"accuracy_delta_95_interval\\": [float(np.quantile(deltas, 0.025)),\\n                                                            float(np.quantile(deltas, 0.975))],\\n                             \\"fraction_delta_at_most_zero\\": float((deltas <= 0).mean())},\\n    }\\n\\n\\ndef end_to_end(baseline_oof: pd.DataFrame, versions: dict[str, pd.DataFrame]) -> tuple[dict, pd.DataFrame]:\\n    \\"\\"\\"Score every official row after substituting each specialist\'s Gemini version.\\n\\n    Only the 570 true-Gemini rows routed to Gemini change. A non-Gemini passage routed\\n    to Gemini stays wrong under every Gemini version, and a true-Gemini passage routed\\n    elsewhere stays wrong, so these 18-class recalls are exact for the two-stage system.\\n    \\"\\"\\"\\n    result = baseline_oof[[\\"id\\", \\"prompt_id\\", \\"true_model\\", \\"fold\\"]].copy()\\n    result[\\"reference\\"] = baseline_oof[\\"predicted_model\\"].to_numpy()\\n    true_gemini = result[\\"true_model\\"].isin(GEMINI_LABELS)\\n    routed = result[\\"reference\\"].isin(GEMINI_LABELS)\\n    for name, frame in versions.items():\\n        replacement = frame.loc[frame[\\"selected\\"]].set_index(\\"id\\")[\\"prediction\\"]\\n        result[name] = result[\\"reference\\"]\\n        mask = result[\\"id\\"].isin(replacement.index)\\n        if not (mask == (true_gemini & routed)).all():\\n            raise ValueError(\\"Specialist rows must be exactly the true-Gemini rows routed to Gemini\\")\\n        result.loc[mask, name] = result.loc[mask, \\"id\\"].map(replacement)\\n    systems = {}\\n    for name in [\\"reference\\", *versions]:\\n        recall = recall_score(result[\\"true_model\\"], result[name], labels=list(LABELS), average=None, zero_division=0)\\n        systems[name] = {\\n            \\"balanced_accuracy\\": float(balanced_accuracy_score(result[\\"true_model\\"], result[name])),\\n            \\"accuracy\\": float(accuracy_score(result[\\"true_model\\"], result[name])),\\n            \\"correct\\": int((result[\\"true_model\\"] == result[name]).sum()),\\n            \\"gemini_recall\\": {label: float(recall[LABELS.index(label)]) for label in GEMINI_LABELS},\\n        }\\n    return {\\n        \\"rows\\": len(result),\\n        \\"rows_routed_to_gemini\\": int(routed.sum()),\\n        \\"true_gemini_routed_to_gemini\\": int((true_gemini & routed).sum()),\\n        \\"non_gemini_routed_to_gemini\\": int((~true_gemini & routed).sum()),\\n        \\"true_gemini_routed_elsewhere\\": int((true_gemini & ~routed).sum()),\\n        \\"systems\\": systems,\\n        \\"scoring_note\\": \\"Non-Gemini rows routed to Gemini are scored incorrect, as they are under any Gemini version.\\",\\n    }, result\\n\\n\\ndef _save_variant(frame: pd.DataFrame, probabilities: np.ndarray, directory: Path,\\n                  components: dict[str, np.ndarray] | None = None) -> tuple[pd.DataFrame, dict]:\\n    directory.mkdir(parents=True, exist_ok=True)\\n    result = frame[[\\"id\\", \\"prompt_id\\", \\"model\\", \\"fold\\", \\"baseline_model\\", \\"selected\\", \\"baseline_correct\\"]].copy()\\n    result[\\"prediction\\"] = np.asarray(GEMINI_LABELS)[probabilities.argmax(axis=1)]\\n    result[\\"confidence\\"] = probabilities.max(axis=1)\\n    result[\\"specialist_correct\\"] = result[\\"prediction\\"] == result[\\"model\\"]\\n    for index, label in enumerate(GEMINI_LABELS):\\n        result[f\\"probability::{label}\\"] = probabilities[:, index]\\n    for name, values in (components or {}).items():\\n        for index, label in enumerate(GEMINI_LABELS):\\n            result[f\\"{name}::{label}\\"] = values[:, index]\\n    selected = result.loc[result[\\"selected\\"]].copy()\\n    metrics = {\\"all_true_gemini\\": _cohort_metrics(result), \\"correctly_routed_gemini\\": _cohort_metrics(selected)}\\n    result.to_parquet(directory / \\"oof.parquet\\", index=False)\\n    selected.to_csv(directory / \\"selected_predictions.csv\\", index=False)\\n    _save_class_reports(result, directory, \\"\\")\\n    _save_class_reports(selected, directory, \\"_selected\\")\\n    _json(directory / \\"metrics.json\\", metrics)\\n    return result, metrics\\n\\n\\ndef evaluate_specialists(\\n    gemini_frame: pd.DataFrame, features: np.ndarray, baseline_oof: pd.DataFrame, output_dir: str | Path,\\n    config: SpecialistConfig | None = None, feature_metadata: dict | None = None,\\n    protocol_metadata: dict | None = None,\\n) -> dict:\\n    config = config or SpecialistConfig()\\n    config.validate()\\n    frame = _validate_frame(gemini_frame)\\n    if \\"text\\" not in frame or not frame[\\"text\\"].map(lambda value: isinstance(value, str) and bool(value.strip())).all():\\n        raise ValueError(\\"The CPU specialist needs the exact nonempty passage text of every Gemini row\\")\\n    features = np.asarray(features)\\n    if len(features) != len(frame):\\n        raise ValueError(\\"Features must already be aligned to every Gemini evaluation row\\")\\n    feature_hash = embeddings_sha256(features)\\n    if feature_metadata is not None and feature_metadata.get(\\"aligned_embeddings_sha256\\") != feature_hash:\\n        raise ValueError(\\"Aligned feature hash differs from the provided provenance\\")\\n    output = Path(output_dir)\\n    if output.exists() and any(output.iterdir()):\\n        raise ValueError(\\"Use a new output directory; an existing experiment must not be overwritten\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    texts = frame[\\"text\\"].to_numpy(dtype=object)\\n    targets = frame[\\"model\\"].map({label: index for index, label in enumerate(GEMINI_LABELS)}).to_numpy()\\n    prompts = frame[\\"prompt_id\\"].to_numpy()\\n    control, readout, blended = (np.full((len(frame), 4), np.nan) for _ in range(3))\\n    counts = np.zeros(len(frame), dtype=np.int64)\\n    nested_results = []\\n    started = time.perf_counter()\\n    for outer_fold in range(5):\\n        fit, valid = (frame[\\"fold\\"] != outer_fold).to_numpy(), (frame[\\"fold\\"] == outer_fold).to_numpy()\\n        if set(prompts[fit]) & set(prompts[valid]):\\n            raise ValueError(\\"Prompt leakage detected before the outer specialist fit\\")\\n        if set(targets[fit]) != set(range(4)) or set(targets[valid]) != set(range(4)):\\n            raise ValueError(\\"Every outer fitting/validation partition must contain all four Gemini classes\\")\\n        with threadpool_limits(limits=2):\\n            C, weight, nested = _select_blend(texts[fit], features[fit], targets[fit], prompts[fit], config, outer_fold)\\n            # The control is the blend\'s own CPU component, fitted once on the outer partition.\\n            control[valid] = _cpu_probabilities(texts[fit], targets[fit], texts[valid], config.cpu)\\n            estimator = _pipeline(C, config.readout)\\n            estimator.fit(features[fit], targets[fit])\\n            readout[valid] = _aligned_probabilities(estimator, features[valid])\\n        blended[valid] = _blend(control[valid], readout[valid], weight)\\n        counts[valid] += 1\\n        nested.update({\\n            \\"outer_train_rows\\": int(fit.sum()), \\"outer_validation_rows\\": int(valid.sum()),\\n            \\"outer_train_selected_rows\\": int(frame.loc[fit, \\"selected\\"].sum()),\\n            \\"outer_validation_control_balanced_accuracy\\": float(balanced_accuracy_score(targets[valid], control[valid].argmax(axis=1))),\\n            \\"outer_validation_blend_balanced_accuracy\\": float(balanced_accuracy_score(targets[valid], blended[valid].argmax(axis=1))),\\n        })\\n        nested_results.append(nested)\\n        _json(output / \\"inner_cv.json\\", nested_results)\\n    if not np.all(counts == 1) or not all(np.isfinite(values).all() for values in (control, readout, blended)):\\n        raise ValueError(\\"Every true-Gemini row must have exactly one complete outer OOF prediction\\")\\n    control_result, control_metrics = _save_variant(frame, control, output / CONTROL)\\n    blend_result, blend_metrics = _save_variant(frame, blended, output / BLEND,\\n                                                {\\"cpu\\": control, \\"styledistance\\": readout})\\n    samples, seed = config.bootstrap_samples, config.readout.seed\\n    paired = {}\\n    for cohort, mask in ((\\"correctly_routed_gemini\\", frame[\\"selected\\"].to_numpy()),\\n                         (\\"all_true_gemini\\", np.ones(len(frame), dtype=bool))):\\n        cohort_frame = frame.loc[mask]\\n        reference = frame.loc[mask, \\"baseline_correct\\"].to_numpy()\\n        control_correct = control_result.loc[mask, \\"specialist_correct\\"].to_numpy()\\n        blend_correct = blend_result.loc[mask, \\"specialist_correct\\"].to_numpy()\\n        paired[cohort] = {\\n            \\"control_vs_reference\\": _paired(cohort_frame, reference, control_correct, samples, seed),\\n            \\"blend_vs_control\\": _paired(cohort_frame, control_correct, blend_correct, samples, seed),\\n            \\"blend_vs_reference\\": _paired(cohort_frame, reference, blend_correct, samples, seed),\\n        }\\n    full, full_predictions = end_to_end(baseline_oof, {CONTROL: control_result, BLEND: blend_result})\\n    full_predictions.to_parquet(output / \\"end_to_end_predictions.parquet\\", index=False)\\n    comparison = {\\n        CONTROL: control_metrics, BLEND: blend_metrics, \\"paired\\": paired, \\"end_to_end\\": full,\\n        \\"selected_C_per_outer_fold\\": [item[\\"selected_C\\"] for item in nested_results],\\n        \\"selected_weight_per_outer_fold\\": [item[\\"selected_weight\\"] for item in nested_results],\\n        \\"elapsed_seconds\\": time.perf_counter() - started,\\n        \\"interpretation\\": (\\"Paired comparisons are conditional on true Gemini; end_to_end scores the two-stage \\"\\n                           \\"system on all 2,699 rows with the saved CPU router.\\"),\\n    }\\n    versions = {package: importlib.metadata.version(package) for package in\\n                (\\"numpy\\", \\"pandas\\", \\"scikit-learn\\", \\"scipy\\")}\\n    protocol = None if protocol_metadata is None else dict(protocol_metadata)\\n    if protocol is not None:\\n        protocol[\\"outer_validation_policy\\"] = (\\"Blend C and weight are selected only by inner prompt-grouped \\"\\n                                               \\"balanced accuracy; each outer partition is predicted once.\\")\\n    _json(output / \\"metadata.json\\", {\\n        \\"schema_version\\": 1, \\"kind\\": \\"gemini-four-class-cpu-specialist-and-styledistance-blend\\",\\n        \\"labels\\": list(GEMINI_LABELS), \\"configuration\\": asdict(config), \\"versions\\": versions,\\n        \\"feature_provenance\\": feature_metadata, \\"aligned_embeddings_sha256\\": feature_hash,\\n        \\"protocol\\": protocol,\\n        \\"blend\\": \\"(1 - weight) * CPU probabilities + weight * StyleDistance probabilities; weight 0 is the control.\\",\\n        \\"training_population\\": \\"Every true-Gemini row in outer fitting folds, including CPU routing false negatives.\\",\\n        \\"learned_transforms\\": \\"Fresh TF-IDF vocabularies, scalers and classifiers fit on each inner/outer fitting partition.\\",\\n    })\\n    _json(output / \\"comparison.json\\", comparison)\\n    return comparison\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--baseline-oof\\", type=Path, required=True)\\n    parser.add_argument(\\"--fold-manifest\\", type=Path, required=True)\\n    parser.add_argument(\\"--features\\", type=Path, required=True)\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    parser.add_argument(\\"--bootstrap-samples\\", type=int, default=10000)\\n    args = parser.parse_args(argv)\\n    baseline_oof = pd.read_parquet(args.baseline_oof)\\n    frame, protocol = prepare_gemini_evaluation(read_train(args.train), baseline_oof,\\n                                               args.fold_manifest, OFFICIAL_BASELINE_COUNTS)\\n    features, metadata = load_feature_archive(args.features, frame)\\n    config = SpecialistConfig(bootstrap_samples=args.bootstrap_samples)\\n    comparison = evaluate_specialists(frame, features, baseline_oof, args.output_dir, config, metadata, protocol)\\n    summary = {name: {cohort: comparison[name][cohort][\\"specialist_correct\\"] for cohort in comparison[name]}\\n               for name in (CONTROL, BLEND)}\\n    summary[\\"end_to_end_balanced_accuracy\\"] = {name: values[\\"balanced_accuracy\\"]\\n                                               for name, values in comparison[\\"end_to_end\\"][\\"systems\\"].items()}\\n    summary[\\"selected_weight_per_outer_fold\\"] = comparison[\\"selected_weight_per_outer_fold\\"]\\n    print(json.dumps(summary, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/data.py": "\\"\\"\\"Strict schemas and content fingerprints. Text is never normalized here.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nfrom typing import Sequence\\n\\nimport pandas as pd\\n\\nfrom .constants import LABELS\\n\\nTRAIN_COLUMNS = (\\"id\\", \\"prompt_id\\", \\"text\\", \\"model\\")\\nTEST_COLUMNS = (\\"id\\", \\"text\\")\\n\\n\\ndef _strings(frame: pd.DataFrame, columns: Sequence[str]) -> None:\\n    missing = set(columns) - set(frame.columns)\\n    if missing:\\n        raise ValueError(f\\"Missing columns: {sorted(missing)}\\")\\n    if frame.columns.duplicated().any():\\n        raise ValueError(\\"Duplicate column names are not allowed\\")\\n    for column in columns:\\n        values = frame[column]\\n        if values.isna().any():\\n            raise ValueError(f\\"{column} contains null values\\")\\n        if not values.map(lambda value: isinstance(value, str)).all():\\n            raise ValueError(f\\"{column} must contain strings\\")\\n        if values.map(lambda value: not value.strip()).any():\\n            raise ValueError(f\\"{column} contains empty or whitespace-only strings\\")\\n    if frame[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Duplicate passage ids are not allowed\\")\\n\\n\\ndef validate_train(frame: pd.DataFrame, require_all_labels: bool = True) -> pd.DataFrame:\\n    \\"\\"\\"Validate labeled training rows without changing their text or order.\\"\\"\\"\\n    _strings(frame, TRAIN_COLUMNS)\\n    if frame.empty:\\n        raise ValueError(\\"Training data is empty\\")\\n    observed = set(frame[\\"model\\"])\\n    unknown = observed - set(LABELS)\\n    if unknown:\\n        raise ValueError(f\\"Unknown model labels: {sorted(unknown)}\\")\\n    if require_all_labels and observed != set(LABELS):\\n        raise ValueError(f\\"Training data is missing model labels: {sorted(set(LABELS) - observed)}\\")\\n    return frame.copy()\\n\\n\\ndef validate_test(frame: pd.DataFrame) -> pd.DataFrame:\\n    \\"\\"\\"Validate the hidden-test schema and reject accidental target leakage.\\"\\"\\"\\n    _strings(frame, TEST_COLUMNS)\\n    if frame.empty:\\n        raise ValueError(\\"Test data is empty; expected at least one passage\\")\\n    forbidden = {\\"model\\", \\"prompt_id\\"} & set(frame.columns)\\n    if forbidden:\\n        raise ValueError(f\\"Test data must not contain training-only columns: {sorted(forbidden)}\\")\\n    return frame.copy()\\n\\n\\ndef data_quality_report(train: pd.DataFrame) -> dict:\\n    \\"\\"\\"Describe identical outputs without rejecting legitimate shared responses.\\n\\n    Text shared between prompts can make prompt-grouped validation optimistic;\\n    cross-label identical outputs can also impose an attribution ambiguity.\\n    Neither proves a schema error, so report both rather than removing rows.\\n    \\"\\"\\"\\n    train = validate_train(train)\\n    grouped = train.groupby(\\"text\\", sort=False).agg(\\n        rows=(\\"id\\", \\"size\\"), labels=(\\"model\\", \\"nunique\\"), prompts=(\\"prompt_id\\", \\"nunique\\")\\n    )\\n    repeated = grouped[\\"rows\\"] > 1\\n    ambiguous = grouped[\\"labels\\"] > 1\\n    cross_prompt = grouped[\\"prompts\\"] > 1\\n    warnings = []\\n    if cross_prompt.any():\\n        warnings.append(\\n            \\"Identical text appears under multiple prompt_ids; prompt-grouped validation may be optimistic.\\"\\n        )\\n    if ambiguous.any():\\n        warnings.append(\\"Identical text has multiple model labels; these rows may be intrinsically ambiguous.\\")\\n    return {\\n        \\"unique_texts\\": len(grouped),\\n        \\"exact_duplicate_text_groups\\": int(repeated.sum()),\\n        \\"exact_duplicate_text_rows\\": int(grouped.loc[repeated, \\"rows\\"].sum()),\\n        \\"exact_duplicate_text_excess_rows\\": int((grouped.loc[repeated, \\"rows\\"] - 1).sum()),\\n        \\"cross_label_identical_text_groups\\": int(ambiguous.sum()),\\n        \\"cross_label_identical_text_rows\\": int(grouped.loc[ambiguous, \\"rows\\"].sum()),\\n        \\"cross_prompt_identical_text_groups\\": int(cross_prompt.sum()),\\n        \\"cross_prompt_identical_text_rows\\": int(grouped.loc[cross_prompt, \\"rows\\"].sum()),\\n        \\"warnings\\": warnings,\\n    }\\n\\n\\ndef _read_csv(path: str | Path) -> pd.DataFrame:\\n    # keep_default_na=False preserves legitimate text such as \\"NA\\" or \\"null\\".\\n    # Empty cells are rejected by the schema, rather than silently imputed.\\n    return pd.read_csv(path, dtype=str, keep_default_na=False)\\n\\n\\ndef read_train(path: str | Path, require_all_labels: bool = True) -> pd.DataFrame:\\n    return validate_train(_read_csv(path), require_all_labels=require_all_labels)\\n\\n\\ndef read_test(path: str | Path) -> pd.DataFrame:\\n    return validate_test(_read_csv(path))\\n\\n\\ndef fingerprint_frame(frame: pd.DataFrame) -> str:\\n    \\"\\"\\"SHA-256 of exact required training content, invariant to CSV row order.\\"\\"\\"\\n    validated = validate_train(frame, require_all_labels=False)\\n    digest = hashlib.sha256()\\n    digest.update(json.dumps(TRAIN_COLUMNS).encode(\\"utf-8\\"))\\n    ordered = validated.sort_values(\\"id\\", kind=\\"stable\\")\\n    for row in ordered.loc[:, list(TRAIN_COLUMNS)].itertuples(index=False, name=None):\\n        digest.update(b\\"\\\\n\\")\\n        digest.update(json.dumps(row, ensure_ascii=False, separators=(\\",\\", \\":\\")).encode(\\"utf-8\\"))\\n    return digest.hexdigest()\\n\\n\\ndef validate_submission(\\n    submission: pd.DataFrame,\\n    expected_ids: Sequence[str] | pd.Series | pd.DataFrame,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Check columns, labels and exact id coverage, independent of row order.\\"\\"\\"\\n    if set(submission.columns) != {\\"id\\", \\"model\\"}:\\n        raise ValueError(\\"Submission must contain exactly the id and model columns\\")\\n    _strings(submission, (\\"id\\", \\"model\\"))\\n    if submission.empty:\\n        raise ValueError(\\"Submission is empty\\")\\n    if isinstance(expected_ids, pd.DataFrame):\\n        expected_ids = expected_ids[\\"id\\"]\\n    expected = list(expected_ids)\\n    if any(not isinstance(value, str) or not value.strip() for value in expected):\\n        raise ValueError(\\"Expected ids must be nonempty strings\\")\\n    if len(set(expected)) != len(expected):\\n        raise ValueError(\\"Expected ids contain duplicates\\")\\n    if len(submission) != len(expected) or set(submission[\\"id\\"]) != set(expected):\\n        raise ValueError(\\"Submission ids do not exactly match the test ids\\")\\n    unknown = set(submission[\\"model\\"]) - set(LABELS)\\n    if unknown:\\n        raise ValueError(f\\"Submission contains unknown labels: {sorted(unknown)}\\")\\n    return submission.copy()\\n\\n\\ndef write_submission(\\n    ids: Sequence[str] | pd.Series,\\n    predictions: Sequence[str],\\n    path: str | Path,\\n) -> pd.DataFrame:\\n    ids = list(ids)\\n    predictions = list(predictions)\\n    if len(ids) != len(predictions):\\n        raise ValueError(\\"Prediction count does not match test row count\\")\\n    submission = pd.DataFrame({\\"id\\": ids, \\"model\\": predictions})\\n    validate_submission(submission, ids)\\n    destination = Path(path)\\n    if destination.suffix != \\".parquet\\":\\n        raise ValueError(\\"Competition submissions must be Parquet files\\")\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    submission.to_parquet(destination, index=False)\\n    validate_submission(pd.read_parquet(destination), ids)\\n    return submission\\n", "ntl/deberta_probe.py": "\\"\\"\\"Instrumented small-subset fitting probe of the unchanged DeBERTa training loop.\\n\\nIt calls ``gemini._fit_predict_fold`` with one training-only subset as both the fitting\\nand the scored rows. Wrappers record pre-clipping gradient norms, per-update parameter\\nmovement, learning rates and AMP loss scale without changing the update. Failing to\\nfit warrants investigation; fitting a subset does not validate the full-data recipe.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport contextlib\\nimport json\\nimport math\\nfrom dataclasses import asdict\\nfrom pathlib import Path\\nfrom unittest import mock\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom . import gemini\\nfrom .data import read_train\\n\\nGROUPS = (\\"embeddings\\", \\"encoder\\", \\"pooler\\", \\"classifier\\")\\n\\n\\ndef parameter_group(name: str) -> str:\\n    for prefix, group in ((\\"deberta.embeddings.\\", \\"embeddings\\"), (\\"deberta.encoder.\\", \\"encoder\\"),\\n                          (\\"pooler.\\", \\"pooler\\"), (\\"classifier.\\", \\"classifier\\")):\\n        if name.startswith(prefix):\\n            return group\\n    raise ValueError(f\\"Unexpected DeBERTa sequence-classification parameter: {name}\\")\\n\\n\\ndef select_subset(frame: pd.DataFrame, per_class: int | None, outer_fold: int, seed: int) -> pd.DataFrame:\\n    \\"\\"\\"Sample equally per version from one outer fold\'s fitting rows; None keeps them all.\\"\\"\\"\\n    fitting = frame.loc[frame[\\"fold\\"] != outer_fold]\\n    if per_class is None:\\n        return fitting.sort_values(\\"id\\", kind=\\"stable\\").reset_index(drop=True)\\n    if fitting.groupby(\\"model\\").size().reindex(gemini.GEMINI_LABELS).min() < per_class:\\n        raise ValueError(\\"Too few fitting rows for the requested per-class subset\\")\\n    subset = fitting.groupby(\\"model\\", group_keys=False).sample(n=per_class, random_state=seed)\\n    return subset.sort_values(\\"id\\", kind=\\"stable\\").reset_index(drop=True)\\n\\n\\nclass Recorder:\\n    \\"\\"\\"Observes the training loop through the objects it creates; never alters updates.\\"\\"\\"\\n\\n    def __init__(self, torch):\\n        self.torch = torch\\n        self.model = None\\n        self.scaler = None\\n        self.initial = None\\n        self.previous = None\\n        self.pending = None\\n        self.clip_calls = 0\\n        self.updates = []\\n\\n    def _norms(self, tensors) -> dict[str, float]:\\n        squares = dict.fromkeys(GROUPS, 0.0)\\n        for name, tensor in tensors:\\n            squares[parameter_group(name)] += float(tensor.detach().float().pow(2).sum())\\n        return {group: math.sqrt(value) for group, value in squares.items()}\\n\\n    def before_clip(self) -> None:\\n        # Called after scaler.unscale_, so gradients are in true (unscaled) units.\\n        if self.initial is None:\\n            with self.torch.no_grad():\\n                self.initial = {name: p.detach().clone() for name, p in self.model.named_parameters()}\\n                self.previous = {name: value.clone() for name, value in self.initial.items()}\\n        named = list(self.model.named_parameters())\\n        self.clip_calls += 1\\n        self.pending = {\\n            \\"gradient_norm_by_group\\": self._norms((name, p.grad) for name, p in named if p.grad is not None),\\n            \\"parameters_without_gradient\\": [name for name, p in named if p.grad is None],\\n            \\"nonfinite_gradient_parameters\\": sum(\\n                int(not self.torch.isfinite(p.grad).all()) for _, p in named if p.grad is not None),\\n        }\\n\\n    def after_step(self, optimizer) -> None:\\n        with self.torch.no_grad():\\n            named = list(self.model.named_parameters())\\n            change = self._norms((name, p - self.previous[name]) for name, p in named)\\n            size = self._norms(named)\\n            for name, p in named:\\n                self.previous[name].copy_(p)\\n        record = {\\"update\\": len(self.updates) + 1, \\"learning_rate\\": optimizer.param_groups[0][\\"lr\\"],\\n                  **(self.pending or {}),\\n                  \\"parameter_change_by_group\\": change,\\n                  \\"relative_change_by_group\\": {group: change[group] / size[group] if size[group] else 0.0\\n                                               for group in GROUPS},\\n                  \\"loss_scale\\": self.scaler.get_scale() if self.scaler is not None and self.scaler.is_enabled() else None}\\n        self.pending = None\\n        self.updates.append(record)\\n\\n    def displacement(self) -> dict:\\n        with self.torch.no_grad():\\n            named = list(self.model.named_parameters())\\n            change = self._norms((name, p - self.initial[name]) for name, p in named)\\n            size = self._norms((name, self.initial[name]) for name, _ in named)\\n        return {\\"parameter_change_by_group\\": change,\\n                \\"relative_change_by_group\\": {group: change[group] / size[group] for group in GROUPS}}\\n\\n\\n@contextlib.contextmanager\\ndef instrument():\\n    import torch\\n    import transformers\\n\\n    recorder = Recorder(torch)\\n    load = transformers.AutoModelForSequenceClassification.from_pretrained\\n    clip = torch.nn.utils.clip_grad_norm_\\n    AdamW, GradScaler = torch.optim.AdamW, torch.amp.GradScaler\\n\\n    def from_pretrained(*args, **kwargs):\\n        recorder.model = load(*args, **kwargs)\\n        return recorder.model\\n\\n    def clip_grad_norm_(parameters, max_norm, *args, **kwargs):\\n        recorder.before_clip()\\n        total = clip(parameters, max_norm, *args, **kwargs)\\n        recorder.pending[\\"gradient_norm\\"] = float(total)\\n        recorder.pending[\\"clipped\\"] = bool(float(total) > max_norm)\\n        return total\\n\\n    class ObservedAdamW(AdamW):\\n        def step(self, closure=None):\\n            result = super().step(closure)\\n            recorder.after_step(self)\\n            return result\\n\\n    class ObservedGradScaler(GradScaler):\\n        def __init__(self, *args, **kwargs):\\n            super().__init__(*args, **kwargs)\\n            recorder.scaler = self\\n\\n    with contextlib.ExitStack() as stack:\\n        stack.enter_context(mock.patch.object(transformers.AutoModelForSequenceClassification,\\n                                              \\"from_pretrained\\", from_pretrained))\\n        stack.enter_context(mock.patch.object(torch.nn.utils, \\"clip_grad_norm_\\", clip_grad_norm_))\\n        stack.enter_context(mock.patch.object(torch.optim, \\"AdamW\\", ObservedAdamW))\\n        stack.enter_context(mock.patch.object(torch.amp, \\"GradScaler\\", ObservedGradScaler))\\n        yield recorder\\n\\n\\ndef run_probe(\\n    train: pd.DataFrame, baseline_oof: pd.DataFrame, fold_manifest_path: str | Path, model_path: str | Path,\\n    output_dir: str | Path, config: gemini.GeminiConfig, per_class: int | None = 8, outer_fold: int = 0,\\n) -> dict:\\n    frame, protocol = gemini.prepare_gemini_evaluation(train, baseline_oof, fold_manifest_path,\\n                                                       gemini.OFFICIAL_BASELINE_COUNTS)\\n    gemini.validate_model_snapshot(model_path)\\n    output = Path(output_dir)\\n    if (output / \\"probe.json\\").exists():\\n        raise ValueError(\\"Use a new output directory; an existing probe must not be overwritten\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    subset = select_subset(frame, per_class, outer_fold, config.seed)\\n    with instrument() as recorder:\\n        probabilities, log = gemini._fit_predict_fold(subset, subset, model_path, config, outer_fold)\\n        displacement = recorder.displacement()\\n    targets = subset[\\"target\\"].to_numpy()\\n    predictions = probabilities.argmax(axis=1)\\n    gradient_norms = np.asarray([update[\\"gradient_norm\\"] for update in recorder.updates])\\n    result = {\\n        \\"schema_version\\": 1, \\"kind\\": \\"deberta-small-subset-fitting-probe\\",\\n        \\"configuration\\": asdict(config), \\"per_class\\": per_class, \\"outer_fold\\": outer_fold,\\n        \\"subset_policy\\": (\\"Equal per-version sample\\" if per_class else \\"Every row\\")\\n                         + \\" of the outer fitting partition; fitted and scored on the same rows.\\",\\n        \\"subset_ids\\": subset[\\"id\\"].tolist(), \\"subset_prompts\\": int(subset[\\"prompt_id\\"].nunique()),\\n        \\"baseline_oof_fingerprint\\": protocol[\\"baseline_oof_fingerprint\\"],\\n        \\"subset_fit_after_training\\": {\\n            \\"accuracy\\": float((predictions == targets).mean()),\\n            \\"cross_entropy\\": float(-np.log(probabilities[np.arange(len(targets)), targets]).mean()),\\n            \\"uniform_cross_entropy\\": math.log(len(gemini.GEMINI_LABELS)),\\n            \\"prediction_counts\\": {label: int((predictions == index).sum())\\n                                  for index, label in enumerate(gemini.GEMINI_LABELS)},\\n            \\"mean_max_probability\\": float(probabilities.max(axis=1).mean()),\\n            \\"scoring_mode\\": \\"model.eval() after the final epoch, as in the outer validation pass\\",\\n        },\\n        \\"epoch_training_loss\\": [epoch[\\"mean_training_loss\\"] for epoch in log[\\"epochs\\"]],\\n        \\"clip_calls\\": recorder.clip_calls, \\"optimizer_updates\\": len(recorder.updates),\\n        \\"gradient_norm_summary\\": {\\"first\\": float(gradient_norms[0]), \\"last\\": float(gradient_norms[-1]),\\n                                  \\"median\\": float(np.median(gradient_norms)),\\n                                  \\"fraction_clipped\\": float(np.mean([u[\\"clipped\\"] for u in recorder.updates]))},\\n        \\"parameters_without_gradient\\": sorted({name for update in recorder.updates\\n                                               for name in update[\\"parameters_without_gradient\\"]}),\\n        \\"total_displacement\\": displacement,\\n        \\"training_log\\": log,\\n        \\"updates\\": recorder.updates,\\n    }\\n    (output / \\"probe.json\\").write_text(json.dumps(result, indent=2, allow_nan=False) + \\"\\\\n\\")\\n    return result\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--baseline-oof\\", type=Path, required=True)\\n    parser.add_argument(\\"--fold-manifest\\", type=Path, required=True)\\n    parser.add_argument(\\"--model-path\\", type=Path, required=True)\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    parser.add_argument(\\"--per-class\\", type=int, default=8, help=\\"Rows per version; 0 uses the whole fitting fold\\")\\n    parser.add_argument(\\"--outer-fold\\", type=int, default=0)\\n    parser.add_argument(\\"--epochs\\", type=int, default=40)\\n    parser.add_argument(\\"--learning-rate\\", type=float, default=2e-5)\\n    parser.add_argument(\\"--device\\", default=\\"cpu\\")\\n    args = parser.parse_args(argv)\\n    config = gemini.GeminiConfig(epochs=args.epochs, learning_rate=args.learning_rate, device=args.device)\\n    result = run_probe(read_train(args.train), pd.read_parquet(args.baseline_oof), args.fold_manifest,\\n                       args.model_path, args.output_dir, config, args.per_class or None, args.outer_fold)\\n    summary = {key: result[key] for key in (\\"subset_fit_after_training\\", \\"epoch_training_loss\\",\\n                                            \\"optimizer_updates\\", \\"gradient_norm_summary\\",\\n                                            \\"parameters_without_gradient\\", \\"total_displacement\\")}\\n    print(json.dumps(summary, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/encoders.py": "\\"\\"\\"Frozen, pinned StyleDistance and Qwen3 embeddings for the Gemini diagnostic.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport gc\\nimport importlib.metadata\\nimport json\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom .data import read_train\\nfrom .frozen_gemini import save_feature_archive\\nfrom .gemini import (\\n    OFFICIAL_BASELINE_COUNTS,\\n    _initialize_cuda_memory_stats,\\n    prepare_gemini_evaluation,\\n)\\n\\nENCODERS = {\\n    \\"styledistance\\": {\\n        \\"model_id\\": \\"StyleDistance/styledistance\\",\\n        \\"model_revision\\": \\"b7df5f0b0480773c097ba3121d83ca32b71015ca\\",\\n        \\"model_type\\": \\"roberta\\", \\"layers\\": 12, \\"dimension\\": 768,\\n        \\"pooling\\": \\"masked_mean_including_special_tokens\\", \\"padding_side\\": \\"right\\",\\n        \\"st_modules\\": [\\"Transformer\\", \\"Pooling\\"],\\n    },\\n    \\"qwen3-embedding-0.6b\\": {\\n        \\"model_id\\": \\"Qwen/Qwen3-Embedding-0.6B\\",\\n        \\"model_revision\\": \\"97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3\\",\\n        \\"model_type\\": \\"qwen3\\", \\"layers\\": 28, \\"dimension\\": 1024,\\n        \\"pooling\\": \\"last_non_padding_token\\", \\"padding_side\\": \\"left\\",\\n        \\"st_modules\\": [\\"Transformer\\", \\"Pooling\\", \\"Normalize\\"],\\n    },\\n}\\n\\n\\ndef validate_snapshot(directory: str | Path, model_key: str) -> dict:\\n    spec = ENCODERS[model_key]\\n    directory = Path(directory)\\n    manifest = json.loads((directory / \\"snapshot_manifest.json\\").read_text())\\n    if (manifest.get(\\"schema_version\\") != 1\\n            or manifest.get(\\"repo_id\\") != spec[\\"model_id\\"]\\n            or manifest.get(\\"revision\\") != spec[\\"model_revision\\"]\\n            or manifest.get(\\"includes_weights\\") is not True):\\n        raise ValueError(\\"Expected a full snapshot of the selected immutable encoder revision\\")\\n    entries = manifest.get(\\"files\\", [])\\n    names = set()\\n    if not entries:\\n        raise ValueError(\\"Encoder snapshot has no recorded files\\")\\n    for entry in entries:\\n        relative = Path(entry[\\"path\\"])\\n        if relative.is_absolute() or \\"..\\" in relative.parts or str(relative) in names:\\n            raise ValueError(\\"Unsafe or repeated encoder snapshot path\\")\\n        names.add(str(relative))\\n        path = directory / relative\\n        if path.is_symlink() or not path.is_file() or path.stat().st_size != entry[\\"size\\"]:\\n            raise ValueError(f\\"Missing or changed encoder snapshot file: {relative}\\")\\n    if not any(name.endswith(\\".safetensors\\") for name in names):\\n        raise ValueError(\\"The pinned encoders require safetensors weights\\")\\n    config = json.loads((directory / \\"config.json\\").read_text())\\n    actual = (config.get(\\"model_type\\"), config.get(\\"num_hidden_layers\\"), config.get(\\"hidden_size\\"))\\n    if actual != (spec[\\"model_type\\"], spec[\\"layers\\"], spec[\\"dimension\\"]):\\n        raise ValueError(\\"Encoder architecture differs from the reviewed checkpoint\\")\\n    modules = json.loads((directory / \\"modules.json\\").read_text())\\n    if [item[\\"type\\"].rsplit(\\".\\", 1)[-1] for item in modules] != spec[\\"st_modules\\"]:\\n        raise ValueError(\\"Unexpected Sentence Transformers modules; a learned transform may be missing\\")\\n    pooling = json.loads((directory / \\"1_Pooling/config.json\\").read_text())\\n    intended = \\"pooling_mode_mean_tokens\\" if model_key == \\"styledistance\\" else \\"pooling_mode_lasttoken\\"\\n    if pooling.get(intended) is not True:\\n        raise ValueError(\\"Upstream pooling differs from the reviewed feature recipe\\")\\n    for key, enabled in pooling.items():\\n        if key.startswith(\\"pooling_mode_\\") and key != intended and enabled:\\n            raise ValueError(\\"Upstream uses additional pooling modes that must be implemented explicitly\\")\\n    return manifest\\n\\n\\ndef masked_mean_pool(hidden, attention_mask):\\n    \\"\\"\\"Native StyleDistance pooling counts every unmasked token, including specials.\\"\\"\\"\\n    import torch\\n\\n    mask = attention_mask.unsqueeze(-1).to(torch.float32)\\n    counts = mask.sum(dim=1)\\n    if torch.any(counts == 0):\\n        raise ValueError(\\"Cannot pool an empty token sequence\\")\\n    return (hidden.float() * mask).sum(dim=1) / counts\\n\\n\\ndef last_non_padding_pool(hidden, attention_mask):\\n    \\"\\"\\"Select the last actual token for either left- or right-padded batches.\\"\\"\\"\\n    import torch\\n\\n    if torch.any(attention_mask.sum(dim=1) == 0):\\n        raise ValueError(\\"Cannot pool an empty token sequence\\")\\n    positions = torch.arange(attention_mask.shape[1], device=attention_mask.device)\\n    last = positions.unsqueeze(0).expand_as(attention_mask).masked_fill(attention_mask == 0, -1).max(dim=1).values\\n    return hidden[torch.arange(hidden.shape[0], device=hidden.device), last].float()\\n\\n\\ndef extract_embeddings(\\n    texts: list[str], model_path: str | Path, model_key: str,\\n    *, batch_size: int = 16, max_length: int = 512, device: str = \\"cuda:0\\",\\n) -> tuple[np.ndarray, dict]:\\n    if model_key not in ENCODERS:\\n        raise ValueError(\\"Unknown encoder key\\")\\n    if batch_size < 1 or not 16 <= max_length <= 512:\\n        raise ValueError(\\"Use a positive batch size and a token limit from 16 to 512\\")\\n    if not texts or any(not isinstance(text, str) or not text.strip() for text in texts):\\n        raise ValueError(\\"Encoder inputs must be nonempty passage strings\\")\\n    spec = ENCODERS[model_key]\\n    validate_snapshot(model_path, model_key)\\n    import torch\\n    from transformers import AutoModel, AutoTokenizer\\n\\n    target = torch.device(device)\\n    if target.type not in {\\"cuda\\", \\"cpu\\"}:\\n        raise ValueError(\\"Use CUDA for the experiment or CPU for a small loading check\\")\\n    if target.type == \\"cuda\\":\\n        if not torch.cuda.is_available():\\n            raise ValueError(\\"CUDA unavailable; enable a free Kaggle GPU\\")\\n        _initialize_cuda_memory_stats(torch, target)\\n    started = time.perf_counter()\\n    tokenizer = AutoTokenizer.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False,\\n        padding_side=spec[\\"padding_side\\"], use_fast=True,\\n    )\\n    if tokenizer.pad_token_id is None:\\n        if tokenizer.eos_token_id is None:\\n            raise ValueError(\\"Encoder tokenizer needs a padding or EOS token\\")\\n        tokenizer.pad_token = tokenizer.eos_token\\n    # The complete embedding backbone is used. There is no layer truncation,\\n    # language-model output head, quantization, task instruction, or fine-tuning.\\n    model = AutoModel.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False,\\n        use_safetensors=True, dtype=torch.float32,\\n    ).to(target)\\n    model.config.use_cache = False\\n    model.requires_grad_(False)\\n    model.eval()\\n    if any(parameter.requires_grad for parameter in model.parameters()):\\n        raise ValueError(\\"The encoder must remain completely frozen\\")\\n    chunks, lengths = [], []\\n    try:\\n        with torch.inference_mode():\\n            for offset in range(0, len(texts), batch_size):\\n                passages = texts[offset:offset + batch_size]\\n                lengths.extend(tokenizer(passages, truncation=False, return_length=True)[\\"length\\"])\\n                batch = tokenizer(\\n                    passages, padding=True, truncation=True, max_length=max_length,\\n                    return_tensors=\\"pt\\",\\n                ).to(target)\\n                hidden = model(**batch).last_hidden_state\\n                if spec[\\"pooling\\"] == \\"last_non_padding_token\\":\\n                    vectors = last_non_padding_pool(hidden, batch[\\"attention_mask\\"])\\n                else:\\n                    vectors = masked_mean_pool(hidden, batch[\\"attention_mask\\"])\\n                norms = vectors.norm(p=2, dim=1, keepdim=True)\\n                if not torch.isfinite(vectors).all() or torch.any(norms <= 0):\\n                    raise ValueError(\\"Encoder produced non-finite or zero embeddings\\")\\n                vectors = vectors / norms\\n                chunks.append(vectors.cpu().numpy())\\n                del hidden, vectors, batch\\n        if target.type == \\"cuda\\":\\n            torch.cuda.synchronize(target)\\n        embeddings = np.concatenate(chunks).astype(np.float32, copy=False)\\n        if embeddings.shape != (len(texts), spec[\\"dimension\\"]):\\n            raise ValueError(\\"Unexpected encoder embedding shape\\")\\n        if not np.isfinite(embeddings).all() or not np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-5):\\n            raise ValueError(\\"Frozen embeddings must be finite unit vectors\\")\\n        versions = {package: importlib.metadata.version(package) for package in (\\"torch\\", \\"transformers\\", \\"numpy\\")}\\n        metadata = {\\n            \\"model_id\\": spec[\\"model_id\\"], \\"model_revision\\": spec[\\"model_revision\\"],\\n            \\"encoder_frozen\\": True,\\n            \\"feature_options\\": {\\n                \\"pooling\\": spec[\\"pooling\\"], \\"max_length\\": max_length,\\n                \\"normalization\\": \\"l2\\", \\"padding_side\\": spec[\\"padding_side\\"],\\n                \\"compute_dtype\\": \\"float32\\", \\"instruction_prefix\\": \\"\\",\\n                \\"special_tokens_included\\": True, \\"quantization\\": \\"none\\",\\n                \\"full_backbone\\": True,\\n                \\"use_cache\\": False,\\n            },\\n            \\"profile\\": {\\n                \\"rows\\": len(texts), \\"dimension\\": embeddings.shape[1],\\n                \\"elapsed_seconds\\": time.perf_counter() - started,\\n                \\"device\\": str(target),\\n                \\"gpu\\": torch.cuda.get_device_name(target) if target.type == \\"cuda\\" else None,\\n                \\"peak_cuda_allocated_bytes\\": torch.cuda.max_memory_allocated(target) if target.type == \\"cuda\\" else None,\\n                \\"truncated_rows\\": int(sum(length > max_length for length in lengths)),\\n                \\"maximum_input_tokens\\": int(max(lengths)),\\n                \\"token_length_quantiles\\": {str(q): float(np.quantile(lengths, q)) for q in (0.5, 0.9, 0.99)},\\n                \\"versions\\": versions,\\n            },\\n        }\\n        return embeddings, metadata\\n    finally:\\n        del model\\n        gc.collect()\\n        if target.type == \\"cuda\\":\\n            torch.cuda.empty_cache()\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--baseline-oof\\", type=Path, required=True)\\n    parser.add_argument(\\"--fold-manifest\\", type=Path, required=True)\\n    parser.add_argument(\\"--model-key\\", choices=tuple(ENCODERS), required=True)\\n    parser.add_argument(\\"--model-path\\", type=Path, required=True)\\n    parser.add_argument(\\"--output\\", type=Path, required=True)\\n    parser.add_argument(\\"--device\\", default=\\"cuda:0\\")\\n    parser.add_argument(\\"--batch-size\\", type=int, default=16)\\n    parser.add_argument(\\"--max-length\\", type=int, default=512)\\n    args = parser.parse_args(argv)\\n    if args.output.exists():\\n        raise ValueError(\\"Feature output already exists; preserve it or choose another output path\\")\\n    frame, protocol = prepare_gemini_evaluation(\\n        read_train(args.train), pd.read_parquet(args.baseline_oof), args.fold_manifest,\\n        OFFICIAL_BASELINE_COUNTS,\\n    )\\n    embeddings, metadata = extract_embeddings(\\n        frame[\\"text\\"].tolist(), args.model_path, args.model_key,\\n        batch_size=args.batch_size, max_length=args.max_length, device=args.device,\\n    )\\n    metadata[\\"train_fingerprint\\"] = protocol[\\"train_fingerprint\\"]\\n    metadata[\\"baseline_oof_fingerprint\\"] = protocol[\\"baseline_oof_fingerprint\\"]\\n    saved = save_feature_archive(\\n        args.output, frame[\\"id\\"].tolist(), embeddings, metadata, texts=frame[\\"text\\"].tolist(),\\n    )\\n    args.output.with_suffix(\\".json\\").write_text(json.dumps(saved, indent=2) + \\"\\\\n\\")\\n    print(json.dumps({\\"feature_archive\\": str(args.output), **metadata[\\"profile\\"]}, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/ensemble.py": "\\"\\"\\"Frozen Qwen3-8B-Base features, their 18-class OOF head, and a nested CPU blend.\\n\\nFrozen extraction is pointwise and label-free, so every official passage is embedded\\nonce and cached with exact ID, text and array hashes. Every TF-IDF vocabulary, scaler\\nand classifier is fit only on its inner or outer training partition. The blend\\n``(1 - w) * CPU + w * Qwen`` takes ``w`` from inner prompt folds; each outer\\npartition is predicted once, and both models score every passage.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport importlib.metadata\\nimport json\\nimport time\\nfrom dataclasses import asdict, dataclass, field\\nfrom pathlib import Path\\n\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom . import neural\\nfrom .constants import LABELS\\nfrom .data import fingerprint_frame, read_train, write_submission\\nfrom .folds import inner_prompt_splits, load_fold_manifest\\nfrom .frozen_gemini import load_feature_archive, save_feature_archive\\nfrom .modeling import BaselineConfig, aligned_probabilities, build_pipeline\\nfrom .selection import first_best\\n\\nRECORDED_REFERENCE_BALANCED_ACCURACY = 0.359667\\n# neural.validate_snapshot pins the backbone\'s hidden width.\\nHIDDEN_SIZE = 4096\\n\\n\\n@dataclass(frozen=True)\\nclass BlendConfig:\\n    cpu: BaselineConfig = field(default_factory=BaselineConfig)\\n    qwen: neural.FrozenConfig = field(default_factory=neural.FrozenConfig)\\n    weight_grid: tuple[float, ...] = (0.0, 0.25, 0.5, 0.75, 1.0)\\n    inner_folds: int = 3\\n    seed: int = 2026\\n    bootstrap_samples: int = 10000\\n    # Fixed before the first blend result: the user\'s practical target and fold support.\\n    minimum_gain_over_recorded_reference: float = 0.01\\n    minimum_folds_beating_cpu: int = 4\\n    maximum_zero_weight_folds: int = 1\\n\\n    def validate(self) -> None:\\n        self.qwen.validate()\\n        if not self.weight_grid or len(set(self.weight_grid)) != len(self.weight_grid):\\n            raise ValueError(\\"Blend weight grid must be nonempty and contain no duplicates\\")\\n        if any(not 0 <= weight <= 1 for weight in self.weight_grid):\\n            raise ValueError(\\"Blend weights must lie in [0, 1]\\")\\n        if self.inner_folds < 2 or self.bootstrap_samples < 1:\\n            raise ValueError(\\"Use at least two inner folds and one bootstrap sample\\")\\n\\n\\ndef _json(path: Path, value: dict | list) -> None:\\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef feature_options(config: neural.FrozenConfig) -> dict:\\n    return {\\n        \\"pooling\\": \\"masked token mean of each listed decoder block output, concatenated\\",\\n        \\"pool_layers\\": list(config.pool_layers), \\"retained_layers\\": config.retained_layers,\\n        \\"quantization\\": config.quantization, \\"max_length\\": config.max_length,\\n        \\"batch_size\\": config.batch_size, \\"normalization\\": \\"none\\", \\"padding_side\\": \\"right\\",\\n        \\"language_model_head\\": False, \\"use_cache\\": False,\\n    }\\n\\n\\ndef extract_archive(frame: pd.DataFrame, model_path: str | Path, output: str | Path,\\n                    config: neural.FrozenConfig) -> dict:\\n    \\"\\"\\"Embed every row once with the frozen recipe; the archive pins ids, text and bytes.\\"\\"\\"\\n    output = Path(output)\\n    if output.exists():\\n        raise ValueError(\\"Feature output already exists; preserve it or choose another output path\\")\\n    features, profile = neural.extract_features(frame[\\"text\\"].tolist(), model_path, config)\\n    if features.shape != (len(frame), HIDDEN_SIZE * len(config.pool_layers)):\\n        raise ValueError(\\"Unexpected frozen Qwen feature shape\\")\\n    metadata = {\\"model_id\\": neural.MODEL_ID, \\"model_revision\\": neural.MODEL_REVISION,\\n                \\"encoder_frozen\\": True, \\"feature_options\\": feature_options(config), \\"profile\\": profile}\\n    if {\\"prompt_id\\", \\"model\\"}.issubset(frame.columns):\\n        metadata[\\"train_fingerprint\\"] = fingerprint_frame(frame)\\n    saved = save_feature_archive(output, frame[\\"id\\"].tolist(), features, metadata, texts=frame[\\"text\\"].tolist())\\n    output.with_suffix(\\".json\\").write_text(json.dumps(saved, indent=2) + \\"\\\\n\\")\\n    return saved\\n\\n\\ndef load_archive(path: str | Path, frame: pd.DataFrame, config: neural.FrozenConfig) -> tuple[np.ndarray, dict]:\\n    \\"\\"\\"Align a verified archive to the frame and require the configured frozen recipe.\\"\\"\\"\\n    features, metadata = load_feature_archive(path, frame)\\n    if metadata.get(\\"model_id\\") != neural.MODEL_ID or metadata.get(\\"model_revision\\") != neural.MODEL_REVISION:\\n        raise ValueError(\\"Features were not extracted from the pinned Qwen3-8B-Base revision\\")\\n    expected = feature_options(config)\\n    stored = metadata[\\"feature_options\\"]\\n    # Batch size and padding change fp16 numerics slightly, so they are part of the recipe too.\\n    for key in (\\"pool_layers\\", \\"retained_layers\\", \\"quantization\\", \\"max_length\\", \\"normalization\\",\\n                \\"batch_size\\", \\"padding_side\\"):\\n        if stored.get(key) != expected[key]:\\n            raise ValueError(f\\"Feature archive {key} differs from the configured frozen recipe\\")\\n    return features, metadata\\n\\n\\ndef _targets(frame: pd.DataFrame) -> np.ndarray:\\n    return frame[\\"model\\"].map({label: index for index, label in enumerate(LABELS)}).to_numpy()\\n\\n\\ndef _cpu_probabilities(fit_texts: np.ndarray, fit_labels: np.ndarray, texts: np.ndarray,\\n                       config: BaselineConfig) -> np.ndarray:\\n    pipeline = build_pipeline(config)\\n    with threadpool_limits(limits=2):\\n        pipeline.fit(fit_texts, fit_labels)\\n        return aligned_probabilities(pipeline, texts)\\n\\n\\ndef _qwen_probabilities(fit_features: np.ndarray, fit_targets: np.ndarray, features: np.ndarray,\\n                        config: neural.FrozenConfig) -> np.ndarray:\\n    # The existing recipe\'s fresh StandardScaler and C=0.005 logistic head.\\n    head = neural._head(config)\\n    with threadpool_limits(limits=2):\\n        head.fit(fit_features, fit_targets)\\n        probabilities = neural._probabilities(head, features)\\n    if not np.allclose(probabilities.sum(axis=1), 1.0):\\n        raise ValueError(\\"Qwen head probabilities must sum to one\\")\\n    return probabilities\\n\\n\\ndef _blend(cpu: np.ndarray, qwen: np.ndarray, weight: float) -> np.ndarray:\\n    return (1 - weight) * cpu + weight * qwen\\n\\n\\n\\ndef _outer_assignments(train: pd.DataFrame, manifest_path: str | Path) -> tuple[np.ndarray, dict]:\\n    manifest, metadata = load_fold_manifest(train, manifest_path)\\n    if metadata[\\"n_splits\\"] != 5:\\n        raise ValueError(\\"This comparison requires the original five prompt-grouped folds\\")\\n    return train[\\"id\\"].map(manifest.set_index(\\"id\\")[\\"fold\\"]).to_numpy(), metadata\\n\\n\\ndef _system_metrics(truth: np.ndarray, predictions: np.ndarray, folds: np.ndarray) -> dict:\\n    recall = recall_score(truth, predictions, labels=np.arange(len(LABELS)), average=None, zero_division=0)\\n    return {\\n        \\"balanced_accuracy\\": float(balanced_accuracy_score(truth, predictions)),\\n        \\"accuracy\\": float(accuracy_score(truth, predictions)),\\n        \\"fold_balanced_accuracy\\": [float(balanced_accuracy_score(truth[folds == fold], predictions[folds == fold]))\\n                                   for fold in range(5)],\\n        \\"recall\\": {label: float(recall[index]) for index, label in enumerate(LABELS)},\\n    }\\n\\n\\ndef prompt_bootstrap(truth: np.ndarray, first: np.ndarray, second: np.ndarray, prompts: np.ndarray,\\n                     samples: int, seed: int) -> dict:\\n    \\"\\"\\"Balanced-accuracy change (second - first) when whole prompts are resampled.\\"\\"\\"\\n    _, prompt_index = np.unique(prompts, return_inverse=True)\\n    shape = (prompt_index.max() + 1, len(LABELS))\\n    totals, first_correct, second_correct = (np.zeros(shape) for _ in range(3))\\n    np.add.at(totals, (prompt_index, truth), 1)\\n    np.add.at(first_correct, (prompt_index, truth), first == truth)\\n    np.add.at(second_correct, (prompt_index, truth), second == truth)\\n    draws = np.random.default_rng(seed).integers(0, shape[0], size=(samples, shape[0]))\\n    weights = np.zeros((samples, shape[0]))\\n    np.add.at(weights, (np.repeat(np.arange(samples), shape[0]), draws.ravel()), 1)\\n    denominator = weights @ totals\\n    with np.errstate(invalid=\\"ignore\\", divide=\\"ignore\\"):\\n        deltas = np.nanmean((weights @ second_correct - weights @ first_correct) / denominator, axis=1)\\n    observed = balanced_accuracy_score(truth, second) - balanced_accuracy_score(truth, first)\\n    return {\\"balanced_accuracy_delta\\": float(observed), \\"samples\\": samples, \\"seed\\": seed,\\n            \\"prompts\\": int(shape[0]),\\n            \\"delta_95_interval\\": [float(np.quantile(deltas, 0.025)), float(np.quantile(deltas, 0.975))],\\n            \\"fraction_delta_at_most_zero\\": float((deltas <= 0).mean()),\\n            \\"second_only_correct\\": int(((second == truth) & (first != truth)).sum()),\\n            \\"first_only_correct\\": int(((first == truth) & (second != truth)).sum())}\\n\\n\\ndef qwen_oof(train: pd.DataFrame, features: np.ndarray, manifest_path: str | Path, output_dir: str | Path,\\n             config: neural.FrozenConfig) -> dict:\\n    \\"\\"\\"Step 1: one held-out 18-class Qwen score vector for every official row.\\"\\"\\"\\n    output = Path(output_dir)\\n    if (output / \\"oof.parquet\\").exists():\\n        raise ValueError(\\"Use a new output directory; an existing OOF must not be overwritten\\")\\n    folds, fold_metadata = _outer_assignments(train, manifest_path)\\n    truth = _targets(train)\\n    probabilities = np.full((len(train), len(LABELS)), np.nan)\\n    for fold in range(5):\\n        fit, valid = folds != fold, folds == fold\\n        if set(train.loc[fit, \\"prompt_id\\"]) & set(train.loc[valid, \\"prompt_id\\"]):\\n            raise ValueError(\\"Prompt leakage detected before the Qwen head fit\\")\\n        probabilities[valid] = _qwen_probabilities(features[fit], truth[fit], features[valid], config)\\n    if not np.isfinite(probabilities).all():\\n        raise ValueError(\\"Every official row must receive exactly one finite OOF score vector\\")\\n    predictions = probabilities.argmax(axis=1)\\n    output.mkdir(parents=True, exist_ok=True)\\n    result = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].rename(columns={\\"model\\": \\"true_model\\"})\\n    result[\\"fold\\"] = folds\\n    result[\\"predicted_model\\"] = np.asarray(LABELS)[predictions]\\n    for index, label in enumerate(LABELS):\\n        result[f\\"probability::{label}\\"] = probabilities[:, index]\\n    result.to_parquet(output / \\"oof.parquet\\", index=False)\\n    metrics = {**_system_metrics(truth, predictions, folds), \\"n_rows\\": len(train),\\n               \\"configuration\\": asdict(config), \\"validation\\": fold_metadata,\\n               \\"head_policy\\": \\"Fresh StandardScaler and logistic head fit on each outer training partition.\\"}\\n    _json(output / \\"metrics.json\\", metrics)\\n    return metrics\\n\\n\\ndef _select_weight(texts, labels, features, targets, prompts, config: BlendConfig, outer_fold: int):\\n    \\"\\"\\"Both classifiers are refit inside every inner split; no outer row is passed in.\\"\\"\\"\\n    splits, split_metadata = inner_prompt_splits(targets, prompts, config.inner_folds, config.seed + outer_fold)\\n    scores = {weight: [] for weight in config.weight_grid}\\n    for fit, valid in splits:\\n        cpu = _cpu_probabilities(texts[fit], labels[fit], texts[valid], config.cpu)\\n        qwen = _qwen_probabilities(features[fit], targets[fit], features[valid], config.qwen)\\n        for weight in config.weight_grid:\\n            scores[weight].append(float(balanced_accuracy_score(targets[valid], _blend(cpu, qwen, weight).argmax(axis=1))))\\n    candidates = [{\\"weight\\": weight, \\"inner_balanced_accuracy\\": values,\\n                   \\"mean_inner_balanced_accuracy\\": float(np.mean(values))}\\n                  for weight, values in sorted(scores.items())]\\n    # Ties keep the smaller Qwen weight, i.e. stay closer to the CPU-only system.\\n    best = first_best(candidates, lambda item: item[\\"mean_inner_balanced_accuracy\\"])\\n    return best[\\"weight\\"], {\\"outer_fold\\": outer_fold, \\"selected_weight\\": best[\\"weight\\"],\\n                            \\"candidates\\": candidates, \\"inner_splits\\": split_metadata,\\n                            \\"selection_metric\\": \\"Mean inner prompt-fold balanced accuracy; ties choose smaller weight.\\"}\\n\\n\\ndef evaluate_blend(\\n    train: pd.DataFrame, features: np.ndarray, manifest_path: str | Path, output_dir: str | Path,\\n    config: BlendConfig | None = None, reference_oof: pd.DataFrame | None = None,\\n    feature_metadata: dict | None = None,\\n) -> dict:\\n    \\"\\"\\"Steps 2-3: nested CPU-Qwen blend on all rows, compared with both CPU references.\\"\\"\\"\\n    config = config or BlendConfig()\\n    config.validate()\\n    output = Path(output_dir)\\n    if output.exists() and any(output.iterdir()):\\n        raise ValueError(\\"Use a new output directory; an existing experiment must not be overwritten\\")\\n    if len(features) != len(train) or not np.isfinite(features).all():\\n        raise ValueError(\\"Features must be finite and aligned to every training row\\")\\n    folds, fold_metadata = _outer_assignments(train, manifest_path)\\n    texts = train[\\"text\\"].to_numpy(dtype=object)\\n    labels = train[\\"model\\"].to_numpy(dtype=object)\\n    truth = _targets(train)\\n    prompts = train[\\"prompt_id\\"].to_numpy()\\n    cpu, qwen, blended = (np.full((len(train), len(LABELS)), np.nan) for _ in range(3))\\n    weights, nested = np.full(len(train), np.nan), []\\n    output.mkdir(parents=True, exist_ok=True)\\n    started = time.perf_counter()\\n    for fold in range(5):\\n        fit, valid = folds != fold, folds == fold\\n        if set(prompts[fit]) & set(prompts[valid]):\\n            raise ValueError(\\"Prompt leakage detected before the outer fits\\")\\n        weight, record = _select_weight(texts[fit], labels[fit], features[fit], truth[fit], prompts[fit],\\n                                        config, fold)\\n        cpu[valid] = _cpu_probabilities(texts[fit], labels[fit], texts[valid], config.cpu)\\n        qwen[valid] = _qwen_probabilities(features[fit], truth[fit], features[valid], config.qwen)\\n        blended[valid] = _blend(cpu[valid], qwen[valid], weight)\\n        weights[valid] = weight\\n        record[\\"outer_balanced_accuracy\\"] = {\\n            name: float(balanced_accuracy_score(truth[valid], values[valid].argmax(axis=1)))\\n            for name, values in ((\\"cpu\\", cpu), (\\"qwen\\", qwen), (\\"blend\\", blended))}\\n        nested.append(record)\\n        _json(output / \\"inner_cv.json\\", nested)\\n        print(json.dumps({\\"outer_fold\\": fold, \\"selected_weight\\": weight, **record[\\"outer_balanced_accuracy\\"]}),\\n              flush=True)\\n    if not all(np.isfinite(values).all() for values in (cpu, qwen, blended)):\\n        raise ValueError(\\"Every official row must receive one finite outer score vector per system\\")\\n    predictions = {\\"cpu\\": cpu.argmax(axis=1), \\"qwen\\": qwen.argmax(axis=1), \\"blend\\": blended.argmax(axis=1)}\\n    if reference_oof is not None:\\n        reference = train[[\\"id\\"]].merge(reference_oof, on=\\"id\\", how=\\"left\\", validate=\\"one_to_one\\")\\n        if (reference[\\"true_model\\"].to_numpy() != labels).any() or (reference[\\"fold\\"].to_numpy() != folds).any():\\n            raise ValueError(\\"Reference OOF labels or folds differ from the official rows and manifest\\")\\n        predictions[\\"recorded_reference\\"] = reference[\\"predicted_model\\"].map(LABELS.index).to_numpy()\\n    result = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].rename(columns={\\"model\\": \\"true_model\\"})\\n    result[\\"fold\\"] = folds\\n    result[\\"blend_weight\\"] = weights\\n    for name, values in predictions.items():\\n        result[f\\"prediction::{name}\\"] = np.asarray(LABELS)[values]\\n    for name, values in ((\\"cpu\\", cpu), (\\"qwen\\", qwen), (\\"blend\\", blended)):\\n        for index, label in enumerate(LABELS):\\n            result[f\\"{name}::{label}\\"] = values[:, index]\\n    result.to_parquet(output / \\"oof.parquet\\", index=False)\\n    systems = {name: _system_metrics(truth, values, folds) for name, values in predictions.items()}\\n    pairs = [(\\"cpu\\", \\"blend\\"), (\\"cpu\\", \\"qwen\\")]\\n    if \\"recorded_reference\\" in predictions:\\n        pairs += [(\\"recorded_reference\\", \\"blend\\"), (\\"recorded_reference\\", \\"cpu\\")]\\n    paired = {f\\"{second}_vs_{first}\\": prompt_bootstrap(truth, predictions[first], predictions[second], prompts,\\n                                                       config.bootstrap_samples, config.seed)\\n              for first, second in pairs}\\n    selected = [record[\\"selected_weight\\"] for record in nested]\\n    blend_folds, cpu_folds = systems[\\"blend\\"][\\"fold_balanced_accuracy\\"], systems[\\"cpu\\"][\\"fold_balanced_accuracy\\"]\\n    gain = systems[\\"blend\\"][\\"balanced_accuracy\\"] - RECORDED_REFERENCE_BALANCED_ACCURACY\\n    gate = {\\n        \\"gain_over_recorded_reference\\": gain,\\n        \\"gain_over_same_environment_cpu\\": systems[\\"blend\\"][\\"balanced_accuracy\\"] - systems[\\"cpu\\"][\\"balanced_accuracy\\"],\\n        \\"folds_blend_beats_same_environment_cpu\\": int(sum(b > c for b, c in zip(blend_folds, cpu_folds, strict=True))),\\n        \\"zero_weight_folds\\": int(sum(weight == 0 for weight in selected)),\\n    }\\n    gate[\\"passed\\"] = bool(gain >= config.minimum_gain_over_recorded_reference\\n                          and gate[\\"folds_blend_beats_same_environment_cpu\\"] >= config.minimum_folds_beating_cpu\\n                          and gate[\\"zero_weight_folds\\"] <= config.maximum_zero_weight_folds)\\n    versions = {package: importlib.metadata.version(package) for package in (\\"numpy\\", \\"scikit-learn\\", \\"scipy\\")}\\n    comparison = {\\"systems\\": systems, \\"paired\\": paired, \\"selected_weight_per_outer_fold\\": selected,\\n                  \\"gate\\": gate, \\"recorded_reference_balanced_accuracy\\": RECORDED_REFERENCE_BALANCED_ACCURACY,\\n                  \\"elapsed_seconds\\": time.perf_counter() - started}\\n    _json(output / \\"comparison.json\\", comparison)\\n    _json(output / \\"metadata.json\\", {\\n        \\"schema_version\\": 1, \\"kind\\": \\"cpu-qwen-nested-18-class-blend\\", \\"labels\\": list(LABELS),\\n        \\"configuration\\": asdict(config), \\"versions\\": versions, \\"validation\\": fold_metadata,\\n        \\"train_fingerprint\\": fingerprint_frame(train), \\"feature_provenance\\": feature_metadata,\\n        \\"blend\\": \\"(1 - w) * CPU probabilities + w * Qwen probabilities, aligned by id and canonical label order.\\",\\n        \\"selection_policy\\": \\"w from inner prompt folds of each outer training partition; both models refit per split.\\",\\n        \\"same_environment_cpu\\": \\"The CPU system refit here; the recorded reference is the submitted Kaggle OOF.\\",\\n    })\\n    return comparison\\n\\n\\n# Outer folds used seed offsets 0-4; the one full-data selection gets its own.\\nFINAL_SELECTION_SEED_OFFSET = 5\\n\\n\\ndef freeze_recipe(train: pd.DataFrame, features: np.ndarray, config: BlendConfig | None = None,\\n                  validation: dict | None = None) -> dict:\\n    \\"\\"\\"Apply the nested-validated weight selection once, to all official rows.\\"\\"\\"\\n    config = config or BlendConfig()\\n    config.validate()\\n    weight, record = _select_weight(\\n        train[\\"text\\"].to_numpy(dtype=object), train[\\"model\\"].to_numpy(dtype=object), features,\\n        _targets(train), train[\\"prompt_id\\"].to_numpy(), config, FINAL_SELECTION_SEED_OFFSET)\\n    summary = None if validation is None else {\\n        \\"balanced_accuracy\\": {name: values[\\"balanced_accuracy\\"] for name, values in validation[\\"systems\\"].items()},\\n        \\"selected_weight_per_outer_fold\\": validation[\\"selected_weight_per_outer_fold\\"], \\"gate\\": validation[\\"gate\\"]}\\n    return {\\"schema_version\\": 1, \\"kind\\": \\"cpu-qwen-blend-recipe\\", \\"labels\\": list(LABELS), \\"weight\\": weight,\\n            \\"cpu\\": asdict(config.cpu), \\"qwen\\": asdict(config.qwen), \\"model_id\\": neural.MODEL_ID,\\n            \\"model_revision\\": neural.MODEL_REVISION, \\"feature_options\\": feature_options(config.qwen),\\n            \\"train_fingerprint\\": fingerprint_frame(train), \\"weight_selection\\": record,\\n            \\"nested_validation\\": summary}\\n\\n\\ndef load_recipe(recipe: dict) -> tuple[float, BaselineConfig, neural.FrozenConfig]:\\n    if (recipe.get(\\"schema_version\\") != 1 or recipe.get(\\"kind\\") != \\"cpu-qwen-blend-recipe\\"\\n            or recipe.get(\\"labels\\") != list(LABELS) or recipe.get(\\"model_id\\") != neural.MODEL_ID\\n            or recipe.get(\\"model_revision\\") != neural.MODEL_REVISION):\\n        raise ValueError(\\"Unsupported blend recipe, label order or backbone revision\\")\\n    weight = recipe.get(\\"weight\\")\\n    if not isinstance(weight, (int, float)) or not 0 <= weight <= 1:\\n        raise ValueError(\\"Blend recipe weight must lie in [0, 1]\\")\\n    qwen = dict(recipe[\\"qwen\\"], pool_layers=tuple(recipe[\\"qwen\\"][\\"pool_layers\\"]))\\n    qwen_config = neural.FrozenConfig(**qwen)\\n    if recipe.get(\\"feature_options\\") != feature_options(qwen_config):\\n        raise ValueError(\\"Blend recipe feature options differ from its frozen configuration\\")\\n    return float(weight), BaselineConfig(**recipe[\\"cpu\\"]), qwen_config\\n\\n\\ndef fit_final(train: pd.DataFrame, model_path: str | Path, recipe: dict, output_dir: str | Path,\\n              device: str = \\"auto\\") -> dict:\\n    \\"\\"\\"Refit both classifiers on every official row for the frozen recipe.\\"\\"\\"\\n    weight, cpu_config, qwen_config = load_recipe(recipe)\\n    if recipe[\\"train_fingerprint\\"] != fingerprint_frame(train):\\n        raise ValueError(\\"Training data differ from the rows used to freeze the recipe\\")\\n    output = Path(output_dir)\\n    if (output / \\"blend.joblib\\").exists():\\n        raise ValueError(\\"Use a new output directory; an existing fitted blend must not be overwritten\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    qwen_config = neural.FrozenConfig(**{**asdict(qwen_config), \\"device\\": device})\\n    started = time.perf_counter()\\n    saved = extract_archive(train, model_path, output / \\"train-features.npz\\", qwen_config)\\n    features, _ = load_archive(output / \\"train-features.npz\\", train, qwen_config)\\n    cpu = build_pipeline(cpu_config)\\n    head = neural._head(qwen_config)\\n    with threadpool_limits(limits=2):\\n        cpu.fit(train[\\"text\\"], train[\\"model\\"])\\n        head.fit(features, _targets(train))\\n    # Fail early if either classifier lost a label or its canonical alignment.\\n    aligned_probabilities(cpu, train[\\"text\\"].iloc[:1])\\n    neural._probabilities(head, features[:1])\\n    metadata = {\\"weight\\": weight, \\"train_fingerprint\\": fingerprint_frame(train), \\"n_train\\": len(train),\\n                \\"feature_profile\\": saved[\\"profile\\"], \\"fit_seconds\\": time.perf_counter() - started,\\n                \\"versions\\": {package: importlib.metadata.version(package)\\n                             for package in (\\"numpy\\", \\"scikit-learn\\", \\"scipy\\")}}\\n    # joblib is trusted project output; never load a third-party pickle with predict_final.\\n    joblib.dump({\\"cpu\\": cpu, \\"qwen_head\\": head, \\"recipe\\": recipe, \\"metadata\\": metadata}, output / \\"blend.joblib\\")\\n    _json(output / \\"fit.json\\", metadata)\\n    return metadata\\n\\n\\ndef predict_final(model_dir: str | Path, test: pd.DataFrame, model_path: str | Path, output: str | Path,\\n                  device: str = \\"auto\\") -> pd.DataFrame:\\n    \\"\\"\\"Score every current test row with both models and the frozen blend weight.\\"\\"\\"\\n    artifact = joblib.load(Path(model_dir) / \\"blend.joblib\\")\\n    weight, _, qwen_config = load_recipe(artifact[\\"recipe\\"])\\n    if artifact[\\"metadata\\"][\\"weight\\"] != weight:\\n        raise ValueError(\\"Fitted blend weight differs from its recipe\\")\\n    started = time.perf_counter()\\n    qwen_config = neural.FrozenConfig(**{**asdict(qwen_config), \\"device\\": device})\\n    features, profile = neural.extract_features(test[\\"text\\"].tolist(), model_path, qwen_config)\\n    with threadpool_limits(limits=2):\\n        cpu = aligned_probabilities(artifact[\\"cpu\\"], test[\\"text\\"])\\n        qwen = neural._probabilities(artifact[\\"qwen_head\\"], features)\\n    blended = _blend(cpu, qwen, weight)\\n    if not np.isfinite(blended).all() or not np.allclose(blended.sum(axis=1), 1.0):\\n        raise ValueError(\\"Blended scores must be finite probability vectors\\")\\n    submission = write_submission(test[\\"id\\"], np.asarray(LABELS)[blended.argmax(axis=1)].tolist(), output)\\n    output = Path(output)\\n    _json(output.with_name(f\\"{output.stem}_profile.json\\"),\\n          {\\"rows\\": len(test), \\"weight\\": weight, \\"elapsed_seconds\\": time.perf_counter() - started,\\n           \\"feature_profile\\": profile})\\n    return submission\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    commands = parser.add_subparsers(dest=\\"command\\", required=True)\\n    extract = commands.add_parser(\\"extract\\", help=\\"Embed every row of a CSV with the frozen recipe\\")\\n    extract.add_argument(\\"--input\\", type=Path, required=True, help=\\"train.csv or test.csv\\")\\n    extract.add_argument(\\"--model-path\\", type=Path, required=True)\\n    extract.add_argument(\\"--output\\", type=Path, required=True)\\n    extract.add_argument(\\"--device\\", default=\\"auto\\")\\n    for name in (\\"qwen-oof\\", \\"blend-cv\\"):\\n        sub = commands.add_parser(name)\\n        sub.add_argument(\\"--train\\", type=Path, required=True)\\n        sub.add_argument(\\"--fold-manifest\\", type=Path, required=True)\\n        sub.add_argument(\\"--features\\", type=Path, required=True)\\n        sub.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    commands.choices[\\"blend-cv\\"].add_argument(\\"--reference-oof\\", type=Path)\\n    freeze = commands.add_parser(\\"freeze\\", help=\\"Select the final weight on all rows and write the recipe\\")\\n    freeze.add_argument(\\"--train\\", type=Path, required=True)\\n    freeze.add_argument(\\"--features\\", type=Path, required=True)\\n    freeze.add_argument(\\"--validation\\", type=Path, help=\\"comparison.json from blend-cv\\")\\n    freeze.add_argument(\\"--output\\", type=Path, required=True)\\n    fit = commands.add_parser(\\"fit\\", help=\\"Refit both classifiers on all rows for a frozen recipe\\")\\n    fit.add_argument(\\"--train\\", type=Path, required=True)\\n    fit.add_argument(\\"--model-path\\", type=Path, required=True)\\n    fit.add_argument(\\"--recipe\\", type=Path, required=True)\\n    fit.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    fit.add_argument(\\"--device\\", default=\\"auto\\")\\n    predict = commands.add_parser(\\"predict\\", help=\\"Write submission.parquet from a fitted blend\\")\\n    predict.add_argument(\\"--model-dir\\", type=Path, required=True)\\n    predict.add_argument(\\"--test\\", type=Path, required=True)\\n    predict.add_argument(\\"--model-path\\", type=Path, required=True)\\n    predict.add_argument(\\"--output\\", type=Path, required=True)\\n    predict.add_argument(\\"--device\\", default=\\"auto\\")\\n    args = parser.parse_args(argv)\\n    if args.command == \\"fit\\":\\n        metadata = fit_final(read_train(args.train), args.model_path, json.loads(args.recipe.read_text()),\\n                             args.output_dir, args.device)\\n        print(json.dumps({key: metadata[key] for key in (\\"weight\\", \\"n_train\\", \\"fit_seconds\\")}, indent=2))\\n        return 0\\n    if args.command == \\"predict\\":\\n        from .data import read_test\\n        submission = predict_final(args.model_dir, read_test(args.test), args.model_path, args.output, args.device)\\n        print(json.dumps({\\"submission\\": str(args.output), \\"rows\\": len(submission)}, indent=2))\\n        return 0\\n    if args.command == \\"extract\\":\\n        from .data import read_test\\n        frame = pd.read_csv(args.input, dtype=str, keep_default_na=False)\\n        frame = read_train(args.input) if \\"model\\" in frame.columns else read_test(args.input)\\n        saved = extract_archive(frame, args.model_path, args.output, neural.FrozenConfig(device=args.device))\\n        print(json.dumps({\\"feature_archive\\": str(args.output), **saved[\\"profile\\"]}, indent=2))\\n        return 0\\n    train = read_train(args.train)\\n    config = BlendConfig()\\n    features, metadata = load_archive(args.features, train, config.qwen)\\n    if args.command == \\"freeze\\":\\n        if args.output.exists():\\n            raise ValueError(\\"Recipe output already exists; preserve it or choose another path\\")\\n        validation = None if args.validation is None else json.loads(args.validation.read_text())\\n        recipe = freeze_recipe(train, features, config, validation)\\n        _json(args.output, recipe)\\n        print(json.dumps({\\"weight\\": recipe[\\"weight\\"], \\"candidates\\": [\\n            (item[\\"weight\\"], item[\\"mean_inner_balanced_accuracy\\"]) for item in recipe[\\"weight_selection\\"][\\"candidates\\"]]},\\n            indent=2))\\n        return 0\\n    if args.command == \\"qwen-oof\\":\\n        metrics = qwen_oof(train, features, args.fold_manifest, args.output_dir, config.qwen)\\n        print(json.dumps({key: metrics[key] for key in (\\"balanced_accuracy\\", \\"fold_balanced_accuracy\\")}, indent=2))\\n        return 0\\n    reference = None if args.reference_oof is None else pd.read_parquet(args.reference_oof)\\n    comparison = evaluate_blend(train, features, args.fold_manifest, args.output_dir, config, reference, metadata)\\n    print(json.dumps({\\"balanced_accuracy\\": {name: values[\\"balanced_accuracy\\"]\\n                                            for name, values in comparison[\\"systems\\"].items()},\\n                      \\"selected_weight_per_outer_fold\\": comparison[\\"selected_weight_per_outer_fold\\"],\\n                      \\"gate\\": comparison[\\"gate\\"]}, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/features.py": "\\"\\"\\"Case-sensitive lexical features and transparent punctuation/style statistics.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, TransformerMixin\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\n\\n\\nclass StylometryTransformer(TransformerMixin, BaseEstimator):\\n    \\"\\"\\"Small numeric feature block; preserve the original punctuation/case.\\"\\"\\"\\n\\n    FEATURE_NAMES = (\\n        \\"log_char_count\\", \\"log_word_count\\", \\"mean_word_length\\", \\"mean_sentence_words\\",\\n        \\"uppercase_ratio\\", \\"digit_ratio\\", \\"whitespace_ratio\\", \\"newline_ratio\\",\\n        \\"comma_ratio\\", \\"semicolon_ratio\\", \\"colon_ratio\\", \\"exclamation_ratio\\",\\n        \\"question_ratio\\", \\"dash_ratio\\", \\"em_dash_ratio\\", \\"curly_quote_ratio\\",\\n        \\"straight_quote_ratio\\", \\"parenthesis_ratio\\", \\"markdown_ratio\\", \\"unique_word_ratio\\",\\n    )\\n\\n    def fit(self, X, y=None):\\n        return self\\n\\n    def transform(self, X):\\n        rows = []\\n        for text in X:\\n            words = re.findall(r\\"\\\\b\\\\w+\\\\b\\", text, flags=re.UNICODE)\\n            chars = max(len(text), 1)\\n            count = max(len(words), 1)\\n            sentences = max(len(re.findall(r\\"[.!?]+(?:\\\\s|$)\\", text)), 1)\\n            rows.append([\\n                np.log1p(len(text)), np.log1p(len(words)),\\n                sum(map(len, words)) / count, len(words) / sentences,\\n                sum(char.isupper() for char in text) / chars,\\n                sum(char.isdigit() for char in text) / chars,\\n                sum(char.isspace() for char in text) / chars,\\n                text.count(\\"\\\\n\\") / chars, text.count(\\",\\") / chars,\\n                text.count(\\";\\") / chars, text.count(\\":\\") / chars,\\n                text.count(\\"!\\") / chars, text.count(\\"?\\") / chars,\\n                text.count(\\"-\\") / chars, text.count(\\"\\u2014\\") / chars,\\n                sum(text.count(char) for char in \\"\\u2018\\u2019\\u201c\\u201d\\") / chars,\\n                sum(text.count(char) for char in \\"\\\\\\"\'\\") / chars,\\n                sum(text.count(char) for char in \\"()[]{}\\") / chars,\\n                sum(text.count(char) for char in \\"*#`_\\") / chars,\\n                len(set(words)) / count,\\n            ])\\n        return np.asarray(rows, dtype=np.float64).reshape(-1, len(self.FEATURE_NAMES))\\n\\n    def get_feature_names_out(self, input_features=None):\\n        return np.asarray(self.FEATURE_NAMES, dtype=object)\\n\\n\\ndef build_features(char_max_features: int = 50000, word_max_features: int = 30000) -> FeatureUnion:\\n    return FeatureUnion(\\n        [\\n            (\\"char\\", TfidfVectorizer(\\n                analyzer=\\"char\\", ngram_range=(3, 5), lowercase=False, strip_accents=None,\\n                max_features=char_max_features, min_df=2, sublinear_tf=True, dtype=np.float64,\\n            )),\\n            (\\"word\\", TfidfVectorizer(\\n                analyzer=\\"word\\", ngram_range=(1, 2), lowercase=False, strip_accents=None,\\n                token_pattern=r\\"(?u)\\\\b\\\\w+\\\\b\\", max_features=word_max_features,\\n                min_df=2, sublinear_tf=True, dtype=np.float64,\\n            )),\\n            (\\"style\\", Pipeline([\\n                (\\"extract\\", StylometryTransformer()),\\n                (\\"scale\\", StandardScaler(with_mean=False)),\\n            ])),\\n        ],\\n        transformer_weights={\\"char\\": 1.0, \\"word\\": 1.0, \\"style\\": 0.1},\\n        n_jobs=1,\\n    )\\n", "ntl/folds.py": "\\"\\"\\"Persisted, fingerprinted prompt-grouped cross-validation assignments.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import StratifiedGroupKFold\\n\\nfrom .constants import DEFAULT_SEED, LABELS\\nfrom .data import fingerprint_frame, validate_train\\n\\n\\ndef build_fold_manifest(\\n    train: pd.DataFrame, n_splits: int = 5, seed: int = DEFAULT_SEED\\n) -> tuple[pd.DataFrame, dict]:\\n    train = validate_train(train)\\n    if n_splits < 2:\\n        raise ValueError(\\"Cross-validation requires at least two folds\\")\\n    if train[\\"prompt_id\\"].nunique() < n_splits:\\n        raise ValueError(\\"There are fewer unique prompts than requested folds\\")\\n    prompt_counts = train.groupby(\\"model\\")[\\"prompt_id\\"].nunique()\\n    if (prompt_counts < n_splits).any():\\n        raise ValueError(\\"Each model must appear in at least n_splits different prompts\\")\\n    # Sorting makes this stable even if a CSV is loaded in a different row order.\\n    ordered = train.sort_values([\\"prompt_id\\", \\"id\\"], kind=\\"stable\\").reset_index(drop=True)\\n    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)\\n    assignments = np.full(len(ordered), -1, dtype=np.int64)\\n    for fold, (_, valid_index) in enumerate(\\n        splitter.split(ordered[\\"text\\"], ordered[\\"model\\"], groups=ordered[\\"prompt_id\\"])\\n    ):\\n        assignments[valid_index] = fold\\n    manifest = ordered[[\\"id\\", \\"prompt_id\\"]].assign(fold=assignments).sort_values(\\"id\\").reset_index(drop=True)\\n    metadata = {\\n        \\"schema_version\\": 1,\\n        \\"algorithm\\": \\"StratifiedGroupKFold\\",\\n        \\"group_column\\": \\"prompt_id\\",\\n        \\"n_splits\\": n_splits,\\n        \\"seed\\": seed,\\n        \\"train_fingerprint\\": fingerprint_frame(train),\\n        \\"n_rows\\": len(train),\\n        \\"n_prompts\\": int(train[\\"prompt_id\\"].nunique()),\\n    }\\n    validate_fold_manifest(train, manifest, metadata)\\n    return manifest, metadata\\n\\n\\ndef validate_fold_manifest(train: pd.DataFrame, manifest: pd.DataFrame, metadata: dict) -> None:\\n    train = validate_train(train)\\n    if metadata.get(\\"schema_version\\") != 1 or metadata.get(\\"group_column\\") != \\"prompt_id\\":\\n        raise ValueError(\\"Unsupported fold manifest schema or grouping column\\")\\n    if metadata.get(\\"train_fingerprint\\") != fingerprint_frame(train):\\n        raise ValueError(\\"Fold manifest fingerprint differs from the training data\\")\\n    if set(manifest.columns) != {\\"id\\", \\"prompt_id\\", \\"fold\\"}:\\n        raise ValueError(\\"Fold manifest must have id, prompt_id and fold columns\\")\\n    if manifest.isna().any().any() or manifest[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Fold manifest contains nulls or duplicate ids\\")\\n    if len(manifest) != len(train) or set(manifest[\\"id\\"]) != set(train[\\"id\\"]):\\n        raise ValueError(\\"Fold manifest must cover every training id exactly once\\")\\n    n_splits = metadata.get(\\"n_splits\\")\\n    if not isinstance(n_splits, int) or n_splits < 2:\\n        raise ValueError(\\"Invalid number of folds in manifest metadata\\")\\n    fold_values = manifest[\\"fold\\"]\\n    if not pd.api.types.is_integer_dtype(fold_values):\\n        raise ValueError(\\"Fold assignments must be integers\\")\\n    if set(fold_values) != set(range(n_splits)):\\n        raise ValueError(\\"Fold assignments must cover exactly 0 through n_splits - 1\\")\\n    aligned = train.merge(manifest, on=\\"id\\", suffixes=(\\"\\", \\"_manifest\\"), validate=\\"one_to_one\\")\\n    if not (aligned[\\"prompt_id\\"] == aligned[\\"prompt_id_manifest\\"]).all():\\n        raise ValueError(\\"Fold manifest prompt ids do not match the training data\\")\\n    if aligned.groupby(\\"prompt_id\\")[\\"fold\\"].nunique().max() != 1:\\n        raise ValueError(\\"Prompt leakage: a prompt occurs in more than one fold\\")\\n    for fold in range(n_splits):\\n        fit = aligned.loc[aligned[\\"fold\\"] != fold]\\n        valid = aligned.loc[aligned[\\"fold\\"] == fold]\\n        if set(fit[\\"prompt_id\\"]) & set(valid[\\"prompt_id\\"]):\\n            raise ValueError(\\"Prompt leakage between fitting and validation rows\\")\\n        if set(fit[\\"model\\"]) != set(LABELS) or set(valid[\\"model\\"]) != set(LABELS):\\n            raise ValueError(f\\"Fold {fold} does not include all labels in both training and validation\\")\\n\\n\\ndef inner_prompt_splits(\\n    targets: np.ndarray, prompts: np.ndarray, n_splits: int, seed: int,\\n) -> tuple[list[tuple[np.ndarray, np.ndarray]], list[dict]]:\\n    \\"\\"\\"Prompt-grouped folds inside one outer fitting partition; shared by every inner selection.\\"\\"\\"\\n    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)\\n    # The splitter reads only the sample count from X, so readouts of any features share splits.\\n    splits = list(splitter.split(np.zeros((len(targets), 1)), targets, groups=prompts))\\n    classes = set(targets.tolist())\\n    split_metadata = []\\n    for inner_fold, (fit, valid) in enumerate(splits):\\n        fit_groups, valid_groups = set(prompts[fit]), set(prompts[valid])\\n        if fit_groups & valid_groups:\\n            raise ValueError(\\"Prompt leakage in inner selection folds\\")\\n        if set(targets[fit].tolist()) != classes or set(targets[valid].tolist()) != classes:\\n            raise ValueError(\\"Every inner fitting/validation partition must contain every class\\")\\n        split_metadata.append({\\"inner_fold\\": inner_fold, \\"training_rows\\": len(fit),\\n                               \\"validation_rows\\": len(valid), \\"training_prompts\\": sorted(fit_groups),\\n                               \\"validation_prompts\\": sorted(valid_groups)})\\n    return splits, split_metadata\\n\\n\\ndef save_fold_manifest(manifest: pd.DataFrame, metadata: dict, directory: str | Path) -> Path:\\n    directory = Path(directory)\\n    directory.mkdir(parents=True, exist_ok=True)\\n    manifest.to_csv(directory / \\"folds.csv\\", index=False)\\n    (directory / \\"folds.json\\").write_text(json.dumps(metadata, indent=2) + \\"\\\\n\\")\\n    return directory / \\"folds.csv\\"\\n\\n\\ndef load_fold_manifest(train: pd.DataFrame, path: str | Path) -> tuple[pd.DataFrame, dict]:\\n    path = Path(path)\\n    if path.is_dir():\\n        path = path / \\"folds.csv\\"\\n    manifest = pd.read_csv(path, dtype={\\"id\\": str, \\"prompt_id\\": str}, keep_default_na=False)\\n    metadata = json.loads(path.with_suffix(\\".json\\").read_text())\\n    validate_fold_manifest(train, manifest, metadata)\\n    return manifest, metadata\\n", "ntl/frozen_gemini.py": "\\"\\"\\"Nested prompt-grouped linear readouts of immutable frozen Gemini embeddings.\\n\\nEncoder extraction is independent of labels. All learned scaling and regularized\\nclassification are fit within the inner/outer training partitions. The original\\n570-row CPU-routed cohort changes reporting only; all 600 true-Gemini rows enter\\nthe original five outer prompt folds.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport re\\nimport time\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\nfrom typing import Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.metrics import balanced_accuracy_score\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom .data import read_train\\nfrom .folds import inner_prompt_splits\\nfrom .gemini import (\\n    GEMINI_LABELS,\\n    OFFICIAL_BASELINE_COUNTS,\\n    _cohort_metrics,\\n    _save_class_reports,\\n    prepare_gemini_evaluation,\\n)\\nfrom .selection import first_best\\n\\n\\n@dataclass(frozen=True)\\nclass FrozenGeminiConfig:\\n    C_grid: tuple[float, ...] = (0.001, 0.01, 0.1, 1.0, 10.0)\\n    inner_folds: int = 3\\n    max_iter: int = 2000\\n    seed: int = 2026\\n\\n    def validate(self) -> None:\\n        if not self.C_grid or any(not np.isfinite(c) or c <= 0 for c in self.C_grid):\\n            raise ValueError(\\"Regularization grid must contain finite positive C values\\")\\n        if len(set(self.C_grid)) != len(self.C_grid):\\n            raise ValueError(\\"Regularization grid contains duplicate C values\\")\\n        if self.inner_folds < 2 or self.max_iter < 1:\\n            raise ValueError(\\"Inner fold count must be at least two and max_iter must be positive\\")\\n\\n\\ndef _json(path: Path, value: dict | list) -> None:\\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef _string_ids(ids: Sequence[str]) -> list[str]:\\n    values = list(ids)\\n    if not values or any(not isinstance(value, str) or not value.strip() for value in values):\\n        raise ValueError(\\"Embedding ids must be nonempty strings\\")\\n    if len(set(values)) != len(values):\\n        raise ValueError(\\"Embedding ids must be unique\\")\\n    return values\\n\\n\\ndef ids_sha256(ids: Sequence[str]) -> str:\\n    \\"\\"\\"SHA-256 of UTF-8 compact JSON list, retaining the archive\'s row order.\\"\\"\\"\\n    return hashlib.sha256(json.dumps(_string_ids(ids), ensure_ascii=False, separators=(\\",\\", \\":\\")).encode()).hexdigest()\\n\\n\\ndef embeddings_sha256(features: np.ndarray) -> str:\\n    \\"\\"\\"Hash sorted-key JSON shape/dtype.str followed by C-order raw array bytes.\\n\\n    Archive dtype and byte order are preserved; this does not round, cast or\\n    normalize values. Import this helper instead of duplicating the contract.\\n    \\"\\"\\"\\n    array = np.asarray(features)\\n    if array.ndim != 2 or not np.issubdtype(array.dtype, np.floating):\\n        raise ValueError(\\"Embeddings must be a two-dimensional floating-point array\\")\\n    if min(array.shape) < 1 or not np.isfinite(array).all():\\n        raise ValueError(\\"Embeddings must be nonempty and finite\\")\\n    digest = hashlib.sha256()\\n    digest.update(json.dumps({\\"shape\\": list(array.shape), \\"dtype\\": array.dtype.str}, sort_keys=True).encode())\\n    digest.update(np.ascontiguousarray(array).tobytes(order=\\"C\\"))\\n    return digest.hexdigest()\\n\\n\\ndef input_text_sha256(ids: Sequence[str], texts: Sequence[str]) -> str:\\n    \\"\\"\\"Hash compact UTF-8 JSON [id, exact_text] lines in archive row order.\\"\\"\\"\\n    ids = _string_ids(ids)\\n    texts = list(texts)\\n    if len(ids) != len(texts) or any(not isinstance(text, str) or not text.strip() for text in texts):\\n        raise ValueError(\\"Each embedding id needs its exact nonempty input text\\")\\n    digest = hashlib.sha256()\\n    for pair in zip(ids, texts, strict=True):\\n        digest.update(json.dumps(pair, ensure_ascii=False, separators=(\\",\\", \\":\\")).encode())\\n        digest.update(b\\"\\\\n\\")\\n    return digest.hexdigest()\\n\\n\\ndef _validate_provenance(metadata: dict) -> None:\\n    if metadata.get(\\"schema_version\\") != 1 or metadata.get(\\"encoder_frozen\\") is not True:\\n        raise ValueError(\\"Features require schema_version 1 and encoder_frozen=true provenance\\")\\n    if not isinstance(metadata.get(\\"model_id\\"), str) or not metadata[\\"model_id\\"].strip():\\n        raise ValueError(\\"Feature provenance is missing model_id\\")\\n    if not isinstance(metadata.get(\\"model_revision\\"), str) or not re.fullmatch(r\\"[0-9a-f]{40}\\", metadata[\\"model_revision\\"]):\\n        raise ValueError(\\"Feature provenance requires an immutable model revision\\")\\n    options = metadata.get(\\"feature_options\\")\\n    if not isinstance(options, dict) or not {\\"pooling\\", \\"max_length\\", \\"normalization\\"}.issubset(options):\\n        raise ValueError(\\"Feature provenance must describe pooling, max_length and normalization\\")\\n    if not isinstance(options[\\"pooling\\"], str) or not options[\\"pooling\\"].strip():\\n        raise ValueError(\\"Feature pooling must be explicit\\")\\n    if not isinstance(options[\\"max_length\\"], int) or options[\\"max_length\\"] < 1:\\n        raise ValueError(\\"Feature max_length must be a positive integer\\")\\n    if not isinstance(options[\\"normalization\\"], (str, bool)):\\n        raise ValueError(\\"Fixed feature normalization must be explicit\\")\\n\\n\\ndef save_feature_archive(\\n    path: str | Path, ids: Sequence[str], embeddings: np.ndarray, metadata: dict,\\n    *, texts: Sequence[str],\\n) -> dict:\\n    \\"\\"\\"Write a pickle-free NPZ with exact row/text/array identity and source pins.\\"\\"\\"\\n    ids = _string_ids(ids)\\n    array = np.asarray(embeddings)\\n    array_hash = embeddings_sha256(array)\\n    if len(ids) != len(array):\\n        raise ValueError(\\"Embedding row count does not match ids\\")\\n    value = dict(metadata)\\n    value[\\"schema_version\\"] = 1\\n    value[\\"ids_sha256\\"] = ids_sha256(ids)\\n    value[\\"embeddings_sha256\\"] = array_hash\\n    value[\\"input_text_sha256\\"] = input_text_sha256(ids, texts)\\n    _validate_provenance(value)\\n    destination = Path(path)\\n    if destination.suffix != \\".npz\\":\\n        raise ValueError(\\"Embedding archive filename must end in .npz\\")\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    np.savez_compressed(destination, ids=np.asarray(ids, dtype=str), embeddings=array,\\n                        metadata=np.asarray(json.dumps(value, sort_keys=True)))\\n    return value\\n\\n\\ndef load_feature_archive(path: str | Path, gemini_frame: pd.DataFrame) -> tuple[np.ndarray, dict]:\\n    \\"\\"\\"Verify stored row order before aligning features to the evaluation frame.\\"\\"\\"\\n    path = Path(path)\\n    with np.load(path, allow_pickle=False) as archive:\\n        if set(archive.files) != {\\"ids\\", \\"embeddings\\", \\"metadata\\"}:\\n            raise ValueError(\\"Feature archive must contain exactly ids, embeddings and metadata\\")\\n        ids_array = archive[\\"ids\\"]\\n        if ids_array.ndim != 1 or ids_array.dtype.kind != \\"U\\":\\n            raise ValueError(\\"Feature ids must be a one-dimensional Unicode array\\")\\n        ids = _string_ids(ids_array.tolist())\\n        features = archive[\\"embeddings\\"]\\n        encoded_metadata = archive[\\"metadata\\"]\\n        if encoded_metadata.ndim != 0 or encoded_metadata.dtype.kind != \\"U\\":\\n            raise ValueError(\\"Feature metadata must be a scalar JSON string\\")\\n        metadata = json.loads(str(encoded_metadata.item()))\\n    if not isinstance(metadata, dict):\\n        raise ValueError(\\"Feature metadata must be a JSON object\\")\\n    _validate_provenance(metadata)\\n    if metadata.get(\\"ids_sha256\\") != ids_sha256(ids):\\n        raise ValueError(\\"Feature id hash mismatch\\")\\n    if metadata.get(\\"embeddings_sha256\\") != embeddings_sha256(features):\\n        raise ValueError(\\"Feature array hash mismatch\\")\\n    if len(features) != len(ids):\\n        raise ValueError(\\"Embedding rows and ids differ in length\\")\\n    expected_ids = _string_ids(gemini_frame[\\"id\\"].tolist())\\n    if len(ids) != len(expected_ids) or set(ids) != set(expected_ids):\\n        raise ValueError(\\"Feature ids must exactly cover all true-Gemini evaluation rows\\")\\n    texts = gemini_frame.set_index(\\"id\\")[\\"text\\"]\\n    if metadata.get(\\"input_text_sha256\\") != input_text_sha256(ids, texts.loc[ids].tolist()):\\n        raise ValueError(\\"Feature input-text hash mismatch\\")\\n    row_lookup = {value: index for index, value in enumerate(ids)}\\n    aligned = features[[row_lookup[value] for value in expected_ids]]\\n    provenance = dict(metadata)\\n    provenance[\\"archive_sha256\\"] = hashlib.sha256(path.read_bytes()).hexdigest()\\n    provenance[\\"aligned_embeddings_sha256\\"] = embeddings_sha256(aligned)\\n    return aligned, provenance\\n\\n\\ndef _validate_frame(frame: pd.DataFrame) -> pd.DataFrame:\\n    required = {\\"id\\", \\"prompt_id\\", \\"model\\", \\"fold\\", \\"baseline_model\\", \\"selected\\", \\"baseline_correct\\"}\\n    if not required.issubset(frame.columns) or frame.empty:\\n        raise ValueError(\\"Gemini evaluation frame is missing required data\\")\\n    if frame[list(required)].isna().any().any() or frame[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Gemini evaluation rows contain nulls or repeated ids\\")\\n    _string_ids(frame[\\"id\\"].tolist())\\n    if set(frame[\\"model\\"]) != set(GEMINI_LABELS):\\n        raise ValueError(\\"Gemini evaluation frame must contain all four canonical Gemini labels\\")\\n    if not pd.api.types.is_integer_dtype(frame[\\"fold\\"]) or set(frame[\\"fold\\"]) != set(range(5)):\\n        raise ValueError(\\"Gemini evaluation requires the original five outer folds\\")\\n    if frame.groupby(\\"prompt_id\\")[\\"fold\\"].nunique().max() != 1:\\n        raise ValueError(\\"Prompt leakage between outer folds\\")\\n    if not pd.api.types.is_bool_dtype(frame[\\"selected\\"]) or not pd.api.types.is_bool_dtype(frame[\\"baseline_correct\\"]):\\n        raise ValueError(\\"Cohort and baseline correctness masks must be boolean\\")\\n    if not frame[\\"selected\\"].equals(frame[\\"baseline_model\\"].isin(GEMINI_LABELS)):\\n        raise ValueError(\\"Selected cohort differs from the saved Gemini-routing condition\\")\\n    if not frame[\\"baseline_correct\\"].equals(frame[\\"baseline_model\\"] == frame[\\"model\\"]):\\n        raise ValueError(\\"Baseline correctness mask differs from the true labels\\")\\n    if not frame[\\"selected\\"].any():\\n        raise ValueError(\\"The selected Gemini reporting cohort is empty\\")\\n    return frame.reset_index(drop=True).copy()\\n\\n\\ndef _pipeline(C: float, config: FrozenGeminiConfig) -> Pipeline:\\n    return Pipeline([\\n        (\\"scale\\", StandardScaler()),\\n        (\\"classifier\\", LogisticRegression(C=C, penalty=\\"l2\\", solver=\\"lbfgs\\",\\n                                           max_iter=config.max_iter, random_state=config.seed)),\\n    ])\\n\\n\\ndef _aligned_probabilities(pipeline: Pipeline, features: np.ndarray) -> np.ndarray:\\n    classes = list(pipeline.named_steps[\\"classifier\\"].classes_)\\n    if len(classes) != 4 or set(classes) != set(range(4)):\\n        raise ValueError(\\"Classifier did not fit the four canonical Gemini label indices\\")\\n    values = pipeline.predict_proba(features)[:, [classes.index(index) for index in range(4)]]\\n    if not np.isfinite(values).all() or (values < 0).any() or not np.allclose(values.sum(axis=1), 1.0):\\n        raise ValueError(\\"Classifier probabilities are non-finite or invalid\\")\\n    return values\\n\\n\\ndef _select_regularization(\\n    fitting_features: np.ndarray, fitting_targets: np.ndarray, fitting_prompts: np.ndarray,\\n    config: FrozenGeminiConfig, outer_fold: int,\\n) -> tuple[float, dict]:\\n    \\"\\"\\"No outer-validation features, labels or scores enter this selection API.\\"\\"\\"\\n    splits, split_metadata = inner_prompt_splits(fitting_targets, fitting_prompts,\\n                                                 config.inner_folds, config.seed + outer_fold)\\n    candidates = []\\n    for C in sorted(config.C_grid):\\n        scores = []\\n        for fit, valid in splits:\\n            estimator = _pipeline(C, config)\\n            estimator.fit(fitting_features[fit], fitting_targets[fit])\\n            probabilities = _aligned_probabilities(estimator, fitting_features[valid])\\n            scores.append(float(balanced_accuracy_score(fitting_targets[valid], probabilities.argmax(axis=1))))\\n        candidates.append({\\"C\\": C, \\"inner_balanced_accuracy\\": scores,\\n                           \\"mean_inner_balanced_accuracy\\": float(np.mean(scores))})\\n    best = first_best(candidates, lambda item: item[\\"mean_inner_balanced_accuracy\\"])  # ascending C\\n    return best[\\"C\\"], {\\"outer_fold\\": outer_fold, \\"selected_C\\": best[\\"C\\"], \\"candidates\\": candidates,\\n                       \\"inner_splits\\": split_metadata,\\n                       \\"selection_metric\\": \\"Unweighted mean inner-fold balanced accuracy; ties choose smaller C.\\"}\\n\\n\\ndef evaluate_embeddings(\\n    gemini_frame: pd.DataFrame, features: np.ndarray, output_dir: str | Path,\\n    config: FrozenGeminiConfig | None = None, feature_metadata: dict | None = None,\\n    protocol_metadata: dict | None = None,\\n) -> dict:\\n    config = config or FrozenGeminiConfig()\\n    config.validate()\\n    frame = _validate_frame(gemini_frame)\\n    features = np.asarray(features)\\n    feature_hash = embeddings_sha256(features)\\n    if len(features) != len(frame):\\n        raise ValueError(\\"Features must already be aligned to every Gemini evaluation row\\")\\n    if feature_metadata is not None:\\n        _validate_provenance(feature_metadata)\\n        if feature_metadata.get(\\"aligned_embeddings_sha256\\", feature_metadata.get(\\"embeddings_sha256\\")) != feature_hash:\\n            raise ValueError(\\"Aligned feature hash differs from the provided provenance\\")\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    if (output / \\"metrics.json\\").exists() or (output / \\"oof.parquet\\").exists():\\n        raise ValueError(\\"Use a new output directory for each frozen-encoder evaluation\\")\\n    targets = frame[\\"model\\"].map({label: index for index, label in enumerate(GEMINI_LABELS)}).to_numpy()\\n    probabilities = np.full((len(frame), 4), np.nan)\\n    counts = np.zeros(len(frame), dtype=np.int64)\\n    nested_results = []\\n    started = time.perf_counter()\\n    for outer_fold in range(5):\\n        fit, valid = (frame[\\"fold\\"] != outer_fold).to_numpy(), (frame[\\"fold\\"] == outer_fold).to_numpy()\\n        if set(frame.loc[fit, \\"prompt_id\\"]) & set(frame.loc[valid, \\"prompt_id\\"]):\\n            raise ValueError(\\"Prompt leakage detected before the outer classifier fit\\")\\n        if set(targets[fit]) != set(range(4)) or set(targets[valid]) != set(range(4)):\\n            raise ValueError(\\"Every outer fitting/validation partition must contain all four Gemini classes\\")\\n        with threadpool_limits(limits=2):\\n            C, nested = _select_regularization(features[fit], targets[fit], frame.loc[fit, \\"prompt_id\\"].to_numpy(),\\n                                               config, outer_fold)\\n            estimator = _pipeline(C, config)\\n            estimator.fit(features[fit], targets[fit])\\n            probabilities[valid] = _aligned_probabilities(estimator, features[valid])\\n        counts[valid] += 1\\n        nested[\\"outer_train_rows\\"] = int(fit.sum())\\n        nested[\\"outer_validation_rows\\"] = int(valid.sum())\\n        nested[\\"outer_train_selected_rows\\"] = int(frame.loc[fit, \\"selected\\"].sum())\\n        nested[\\"outer_validation_balanced_accuracy\\"] = float(balanced_accuracy_score(targets[valid], probabilities[valid].argmax(axis=1)))\\n        nested_results.append(nested)\\n        _json(output / \\"inner_cv.json\\", nested_results)\\n    if not np.all(counts == 1) or not np.isfinite(probabilities).all():\\n        raise ValueError(\\"Every true-Gemini row must have exactly one complete outer OOF prediction\\")\\n    result = frame[[\\"id\\", \\"prompt_id\\", \\"model\\", \\"fold\\", \\"baseline_model\\", \\"selected\\", \\"baseline_correct\\"]].copy()\\n    result[\\"prediction\\"] = np.asarray(GEMINI_LABELS)[probabilities.argmax(axis=1)]\\n    result[\\"confidence\\"] = probabilities.max(axis=1)\\n    result[\\"specialist_correct\\"] = result[\\"prediction\\"] == result[\\"model\\"]\\n    for index, label in enumerate(GEMINI_LABELS):\\n        result[f\\"probability::{label}\\"] = probabilities[:, index]\\n    selected = result.loc[result[\\"selected\\"]].copy()\\n    metrics = {\\"all_true_gemini\\": _cohort_metrics(result), \\"correctly_routed_gemini\\": _cohort_metrics(selected),\\n               \\"selected_C_per_outer_fold\\": [item[\\"selected_C\\"] for item in nested_results],\\n               \\"elapsed_seconds\\": time.perf_counter() - started,\\n               \\"interpretation\\": \\"Nested prompt-grouped frozen-feature comparison; 570-row selection is for reporting only, and is not a hidden-test or end-to-end routing score.\\"}\\n    versions = {package: importlib.metadata.version(package) for package in\\n                (\\"numpy\\", \\"pandas\\", \\"scikit-learn\\", \\"scipy\\")}\\n    protocol = None if protocol_metadata is None else dict(protocol_metadata)\\n    if protocol is not None:\\n        protocol[\\"outer_validation_policy\\"] = \\"C is selected only by inner prompt-grouped balanced accuracy; each outer validation partition is predicted once.\\"\\n    metadata = {\\"schema_version\\": 1, \\"kind\\": \\"frozen-encoder-nested-gemini-linear-readout\\",\\n                \\"labels\\": list(GEMINI_LABELS), \\"configuration\\": asdict(config), \\"versions\\": versions,\\n                \\"feature_provenance\\": feature_metadata,\\n                \\"aligned_embeddings_sha256\\": feature_hash,\\n                \\"aligned_ids_sha256\\": ids_sha256(frame[\\"id\\"].tolist()),\\n                \\"protocol\\": protocol,\\n                \\"selection_policy\\": \\"Inner prompt-grouped folds only; outer validation is predicted once after choosing C.\\",\\n                \\"training_population\\": \\"Every true-Gemini row in outer fitting folds, including CPU routing false negatives.\\",\\n                \\"learned_transforms\\": \\"Fresh StandardScaler and L2 logistic regression fit on each inner/outer fitting partition.\\"}\\n    result.to_parquet(output / \\"oof.parquet\\", index=False)\\n    selected.to_parquet(output / \\"selected_oof.parquet\\", index=False)\\n    selected.to_csv(output / \\"selected_predictions.csv\\", index=False)\\n    _save_class_reports(result, output, \\"\\")\\n    _save_class_reports(selected, output, \\"_selected\\")\\n    _json(output / \\"metadata.json\\", metadata)\\n    _json(output / \\"metrics.json\\", metrics)\\n    return metrics\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--baseline-oof\\", type=Path, required=True)\\n    parser.add_argument(\\"--fold-manifest\\", type=Path, required=True)\\n    parser.add_argument(\\"--features\\", type=Path, required=True)\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    parser.add_argument(\\"--C-grid\\", type=float, nargs=\\"+\\", default=[0.001, 0.01, 0.1, 1.0, 10.0])\\n    parser.add_argument(\\"--inner-folds\\", type=int, default=3)\\n    parser.add_argument(\\"--max-iter\\", type=int, default=2000)\\n    parser.add_argument(\\"--seed\\", type=int, default=2026)\\n    args = parser.parse_args(argv)\\n    frame, protocol = prepare_gemini_evaluation(read_train(args.train), pd.read_parquet(args.baseline_oof),\\n                                               args.fold_manifest, OFFICIAL_BASELINE_COUNTS)\\n    features, metadata = load_feature_archive(args.features, frame)\\n    config = FrozenGeminiConfig(C_grid=tuple(args.C_grid), inner_folds=args.inner_folds,\\n                                max_iter=args.max_iter, seed=args.seed)\\n    metrics = evaluate_embeddings(frame, features, args.output_dir, config, metadata, protocol)\\n    print(json.dumps(metrics, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/gemini.py": "\\"\\"\\"A fixed, prompt-grouped DeBERTa experiment for exact Gemini version attribution.\\n\\nTrain each fresh four-class head/backbone on every true-Gemini training-fold row.\\nEvaluate all 600 held-out Gemini rows and the preselected 570 correctly routed by\\nthe saved CPU out-of-fold predictions. Outer validation never selects an epoch.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport gc\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport math\\nimport random\\nimport time\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import (\\n    accuracy_score,\\n    balanced_accuracy_score,\\n    confusion_matrix,\\n    precision_recall_fscore_support,\\n)\\n\\nfrom .constants import LABELS\\nfrom .data import fingerprint_frame, read_train, validate_train\\nfrom .folds import load_fold_manifest, save_fold_manifest\\n\\nMODEL_ID = \\"microsoft/deberta-v3-base\\"\\nMODEL_REVISION = \\"8ccc9b6f36199bec6961081d44eb72fb3f7353f3\\"\\nGEMINI_LABELS = tuple(label for label in LABELS if label.startswith(\\"gemini-\\"))\\nOFFICIAL_BASELINE_COUNTS = {\\n    \\"train_rows\\": 2699,\\n    \\"gemini_rows\\": 600,\\n    \\"selected_rows\\": 570,\\n    \\"baseline_selected_correct\\": 291,\\n}\\n\\n\\n@dataclass(frozen=True)\\nclass GeminiConfig:\\n    epochs: int = 3\\n    learning_rate: float = 2e-5\\n    max_length: int = 512\\n    batch_size: int = 8\\n    gradient_accumulation: int = 2\\n    warmup_ratio: float = 0.1\\n    weight_decay: float = 0.01\\n    max_grad_norm: float = 1.0\\n    amp_initial_scale: float = 1024.0\\n    max_consecutive_amp_skips: int = 20\\n    seed: int = 2026\\n    device: str = \\"cuda:0\\"\\n\\n    def validate(self) -> None:\\n        if self.epochs < 1 or self.batch_size < 1 or self.gradient_accumulation < 1:\\n            raise ValueError(\\"Epochs, microbatch size and accumulation must be positive\\")\\n        if not 16 <= self.max_length <= 512:\\n            raise ValueError(\\"The selected DeBERTa protocol requires a token limit from 16 to 512\\")\\n        if self.learning_rate <= 0 or self.max_grad_norm <= 0 or self.weight_decay < 0:\\n            raise ValueError(\\"Invalid optimizer configuration\\")\\n        if not 0 <= self.warmup_ratio < 1:\\n            raise ValueError(\\"Warmup ratio must be in [0, 1)\\")\\n        if self.amp_initial_scale <= 0 or self.max_consecutive_amp_skips < 1:\\n            raise ValueError(\\"Invalid AMP loss-scale recovery settings\\")\\n\\n\\ndef _json(path: Path, value: dict | list) -> None:\\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef _frame_digest(frame: pd.DataFrame, columns: list[str]) -> str:\\n    digest = hashlib.sha256()\\n    for row in frame.sort_values(\\"id\\").loc[:, columns].itertuples(index=False, name=None):\\n        digest.update(json.dumps(row, ensure_ascii=False, separators=(\\",\\", \\":\\")).encode())\\n        digest.update(b\\"\\\\n\\")\\n    return digest.hexdigest()\\n\\n\\ndef prepare_gemini_evaluation(\\n    train: pd.DataFrame,\\n    baseline_oof: pd.DataFrame,\\n    fold_manifest_path: str | Path,\\n    expected_counts: dict[str, int] | None = None,\\n) -> tuple[pd.DataFrame, dict]:\\n    \\"\\"\\"Join exact baseline OOF truth/IDs/folds before selecting the fixed cohort.\\n\\n    Selection changes the reporting cohort only, never the training population.\\n    The fingerprinted fold manifest always belongs to the complete 18-class train.\\n    \\"\\"\\"\\n    train = validate_train(train)\\n    manifest, fold_metadata = load_fold_manifest(train, fold_manifest_path)\\n    if fold_metadata[\\"n_splits\\"] != 5:\\n        raise ValueError(\\"This fixed comparison requires the original five prompt-grouped folds\\")\\n    required = {\\"id\\", \\"prompt_id\\", \\"true_model\\", \\"fold\\", \\"predicted_model\\"}\\n    if not required.issubset(baseline_oof.columns):\\n        raise ValueError(f\\"Baseline OOF is missing columns: {sorted(required - set(baseline_oof.columns))}\\")\\n    baseline = baseline_oof.loc[:, sorted(required)].copy()\\n    if baseline.isna().any().any() or baseline[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Baseline OOF contains null values or duplicate ids\\")\\n    for column in (\\"id\\", \\"prompt_id\\", \\"true_model\\", \\"predicted_model\\"):\\n        if not baseline[column].map(lambda value: isinstance(value, str) and bool(value.strip())).all():\\n            raise ValueError(f\\"Baseline OOF {column} must contain nonempty strings\\")\\n    if not pd.api.types.is_integer_dtype(baseline[\\"fold\\"]):\\n        raise ValueError(\\"Baseline OOF fold assignments must be integers\\")\\n    if len(baseline) != len(train) or set(baseline[\\"id\\"]) != set(train[\\"id\\"]):\\n        raise ValueError(\\"Baseline OOF must contain every official training id exactly once\\")\\n    if not set(baseline[\\"predicted_model\\"]).issubset(LABELS):\\n        raise ValueError(\\"Baseline OOF contains predictions outside the 18 valid labels\\")\\n    merged = train.merge(baseline, on=\\"id\\", suffixes=(\\"\\", \\"_baseline\\"), validate=\\"one_to_one\\")\\n    expected_folds = merged[\\"id\\"].map(manifest.set_index(\\"id\\")[\\"fold\\"])\\n    if not (merged[\\"prompt_id\\"] == merged[\\"prompt_id_baseline\\"]).all():\\n        raise ValueError(\\"Baseline OOF prompt ids differ from official training rows\\")\\n    if not (merged[\\"model\\"] == merged[\\"true_model\\"]).all():\\n        raise ValueError(\\"Baseline OOF true model labels differ from official training rows\\")\\n    if not (merged[\\"fold\\"] == expected_folds).all():\\n        raise ValueError(\\"Baseline OOF folds differ from the saved prompt-grouped manifest\\")\\n    gemini = merged.loc[merged[\\"model\\"].isin(GEMINI_LABELS)].copy()\\n    gemini = gemini.rename(columns={\\"predicted_model\\": \\"baseline_model\\"})\\n    gemini[\\"selected\\"] = gemini[\\"baseline_model\\"].isin(GEMINI_LABELS)\\n    gemini[\\"baseline_correct\\"] = gemini[\\"baseline_model\\"] == gemini[\\"model\\"]\\n    gemini[\\"target\\"] = gemini[\\"model\\"].map({label: index for index, label in enumerate(GEMINI_LABELS)})\\n    gemini = gemini.sort_values(\\"id\\", kind=\\"stable\\").reset_index(drop=True)\\n    for fold in range(5):\\n        fitting = gemini.loc[gemini[\\"fold\\"] != fold]\\n        validation = gemini.loc[gemini[\\"fold\\"] == fold]\\n        if set(fitting[\\"prompt_id\\"]) & set(validation[\\"prompt_id\\"]):\\n            raise ValueError(\\"Prompt leakage in Gemini fold assignments\\")\\n        if set(fitting[\\"model\\"]) != set(GEMINI_LABELS) or set(validation[\\"model\\"]) != set(GEMINI_LABELS):\\n            raise ValueError(\\"Every Gemini class must occur in fitting and validation for every fold\\")\\n    counts = {\\n        \\"train_rows\\": len(train), \\"gemini_rows\\": len(gemini),\\n        \\"selected_rows\\": int(gemini[\\"selected\\"].sum()),\\n        \\"baseline_selected_correct\\": int((gemini[\\"selected\\"] & gemini[\\"baseline_correct\\"]).sum()),\\n    }\\n    if expected_counts is not None:\\n        for key, expected in expected_counts.items():\\n            if counts.get(key) != expected:\\n                raise ValueError(f\\"Saved baseline cohort mismatch for {key}: expected {expected}, found {counts.get(key)}\\")\\n    protocol = {\\n        \\"counts\\": counts,\\n        \\"train_fingerprint\\": fingerprint_frame(train),\\n        \\"baseline_oof_fingerprint\\": _frame_digest(baseline, [\\"id\\", \\"prompt_id\\", \\"true_model\\", \\"fold\\", \\"predicted_model\\"]),\\n        \\"fold_metadata\\": fold_metadata,\\n        \\"labels\\": list(GEMINI_LABELS),\\n        \\"cohort_definition\\": \\"True model is Gemini AND saved CPU OOF prediction is Gemini; selection is for reporting only.\\",\\n        \\"training_population\\": \\"All true-Gemini rows in the fitting prompt folds, including CPU family false negatives.\\",\\n        \\"outer_validation_policy\\": \\"Fixed epochs; no checkpoint, epoch, hyperparameter or threshold selection on outer validation.\\",\\n    }\\n    return gemini, protocol\\n\\n\\ndef validate_model_snapshot(directory: str | Path) -> dict:\\n    directory = Path(directory)\\n    manifest = json.loads((directory / \\"snapshot_manifest.json\\").read_text())\\n    if (manifest.get(\\"schema_version\\") != 1 or manifest.get(\\"repo_id\\") != MODEL_ID\\n            or manifest.get(\\"revision\\") != MODEL_REVISION or manifest.get(\\"includes_weights\\") is not True):\\n        raise ValueError(\\"Expected a local snapshot of the pinned microsoft/deberta-v3-base revision\\")\\n    entries = manifest.get(\\"files\\", [])\\n    declared = set()\\n    if not entries:\\n        raise ValueError(\\"Snapshot manifest has no file inventory\\")\\n    for entry in entries:\\n        relative = Path(entry[\\"path\\"])\\n        if relative.is_absolute() or \\"..\\" in relative.parts or str(relative) in declared:\\n            raise ValueError(\\"Snapshot manifest contains an unsafe or repeated file path\\")\\n        declared.add(str(relative))\\n        target = directory / relative\\n        if target.is_symlink() or not target.is_file() or target.stat().st_size != entry[\\"size\\"]:\\n            raise ValueError(f\\"Missing or changed pinned snapshot file: {relative}\\")\\n    if not {\\"config.json\\", \\"spm.model\\", \\"pytorch_model.bin\\"}.issubset(declared):\\n        raise ValueError(\\"Pinned DeBERTa snapshot needs config.json, spm.model and pytorch_model.bin\\")\\n    architecture = json.loads((directory / \\"config.json\\").read_text())\\n    if (architecture.get(\\"model_type\\"), architecture.get(\\"hidden_size\\"),\\n            architecture.get(\\"num_hidden_layers\\")) != (\\"deberta-v2\\", 768, 12):\\n        raise ValueError(\\"Snapshot architecture does not match DeBERTa-v3-base\\")\\n    return manifest\\n\\n\\ndef validate_experiment_inputs(\\n    train_path: str | Path, baseline_oof_path: str | Path, fold_manifest_path: str | Path,\\n) -> dict:\\n    \\"\\"\\"CPU-only preflight before downloading a model or allocating GPU memory.\\"\\"\\"\\n    _, protocol = prepare_gemini_evaluation(\\n        read_train(train_path), pd.read_parquet(baseline_oof_path), fold_manifest_path,\\n        OFFICIAL_BASELINE_COUNTS,\\n    )\\n    return protocol\\n\\n\\ndef _seed_everything(seed: int, torch) -> None:\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    if torch.cuda.is_available():\\n        torch.cuda.manual_seed_all(seed)\\n        torch.backends.cudnn.benchmark = False\\n        torch.backends.cuda.matmul.allow_tf32 = False\\n\\n\\ndef _initialize_cuda_memory_stats(torch, device) -> None:\\n    \\"\\"\\"Initialize this subprocess\'s caching allocator before resetting counters.\\"\\"\\"\\n    torch.cuda.init()\\n    torch.cuda.set_device(device)\\n    transient = torch.empty(1, device=device)\\n    del transient\\n    torch.cuda.empty_cache()\\n    torch.cuda.reset_peak_memory_stats(device)\\n\\n\\ndef _fit_predict_fold(\\n    fitting: pd.DataFrame, validation: pd.DataFrame, model_path: str | Path,\\n    config: GeminiConfig, fold: int,\\n) -> tuple[np.ndarray, dict]:\\n    \\"\\"\\"Full fine-tuning; a new pretrained backbone and random head per outer fold.\\"\\"\\"\\n    import torch\\n    from torch.utils.data import DataLoader\\n    from transformers import (\\n        AutoModelForSequenceClassification,\\n        AutoTokenizer,\\n        get_linear_schedule_with_warmup,\\n    )\\n\\n    config.validate()\\n    device = torch.device(config.device)\\n    cuda = device.type == \\"cuda\\"\\n    if cuda and not torch.cuda.is_available():\\n        raise ValueError(\\"This experiment requires a CUDA GPU; select a free Kaggle T4\\")\\n    if device.type not in {\\"cuda\\", \\"cpu\\"}:\\n        raise ValueError(\\"Use CUDA for the experiment or CPU for a tiny diagnostic\\")\\n    fold_seed = config.seed + fold\\n    _seed_everything(fold_seed, torch)\\n    started = time.perf_counter()\\n    if cuda:\\n        _initialize_cuda_memory_stats(torch, device)\\n    tokenizer = AutoTokenizer.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False, use_fast=False,\\n    )\\n    model = AutoModelForSequenceClassification.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False,\\n        use_safetensors=False, dtype=torch.float32, num_labels=len(GEMINI_LABELS),\\n        id2label=dict(enumerate(GEMINI_LABELS)),\\n        label2id={label: index for index, label in enumerate(GEMINI_LABELS)},\\n    ).to(device)\\n    if not all(parameter.requires_grad for parameter in model.parameters()):\\n        raise ValueError(\\"The experiment requires a fully trainable pretrained backbone and classifier\\")\\n\\n    def rows(frame):\\n        encoded = tokenizer(frame[\\"text\\"].tolist(), truncation=True, max_length=config.max_length)\\n        return [{**{name: values[index] for name, values in encoded.items()},\\n                 \\"labels\\": int(frame.iloc[index][\\"target\\"])} for index in range(len(frame))]\\n\\n    def collate(items):\\n        labels = torch.tensor([item[\\"labels\\"] for item in items], dtype=torch.long)\\n        batch = tokenizer.pad(\\n            [{key: value for key, value in item.items() if key != \\"labels\\"} for item in items],\\n            padding=True, return_tensors=\\"pt\\",\\n        )\\n        batch[\\"labels\\"] = labels\\n        return batch\\n\\n    fitting_rows, validation_rows = rows(fitting), rows(validation)\\n    generator = torch.Generator().manual_seed(fold_seed)\\n    train_loader = DataLoader(fitting_rows, batch_size=config.batch_size, shuffle=True,\\n                              collate_fn=collate, generator=generator, num_workers=0)\\n    valid_loader = DataLoader(validation_rows, batch_size=config.batch_size, shuffle=False,\\n                              collate_fn=collate, num_workers=0)\\n    decay, no_decay = [], []\\n    for name, parameter in model.named_parameters():\\n        (no_decay if parameter.ndim == 1 or name.endswith(\\".bias\\") else decay).append(parameter)\\n    optimizer = torch.optim.AdamW([\\n        {\\"params\\": decay, \\"weight_decay\\": config.weight_decay},\\n        {\\"params\\": no_decay, \\"weight_decay\\": 0.0},\\n    ], lr=config.learning_rate)\\n    updates_per_epoch = math.ceil(len(train_loader) / config.gradient_accumulation)\\n    total_updates = updates_per_epoch * config.epochs\\n    warmup_updates = math.ceil(config.warmup_ratio * total_updates)\\n    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_updates, total_updates)\\n    scaler = torch.amp.GradScaler(\\"cuda\\", enabled=cuda, init_scale=config.amp_initial_scale)\\n    epoch_logs = []\\n    optimizer_updates = 0\\n    skipped_amp_updates = 0\\n    consecutive_amp_skips = 0\\n    try:\\n        for epoch in range(config.epochs):\\n            epoch_started = time.perf_counter()\\n            model.train()\\n            optimizer.zero_grad(set_to_none=True)\\n            total_loss = 0.0\\n            for step, batch in enumerate(train_loader):\\n                labels = batch.pop(\\"labels\\").to(device)\\n                inputs = {key: value.to(device) for key, value in batch.items()}\\n                group_start = (step // config.gradient_accumulation) * config.gradient_accumulation * config.batch_size\\n                group_rows = min(config.gradient_accumulation * config.batch_size, len(fitting_rows) - group_start)\\n                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=cuda):\\n                    logits = model(**inputs).logits\\n                    summed_loss = torch.nn.functional.cross_entropy(logits.float(), labels, reduction=\\"sum\\")\\n                    loss = summed_loss / group_rows\\n                if not torch.isfinite(loss):\\n                    raise ValueError(f\\"Non-finite fitting loss in fold {fold}, epoch {epoch + 1}\\")\\n                total_loss += float(summed_loss.detach().cpu())\\n                scaler.scale(loss).backward()\\n                if (step + 1) % config.gradient_accumulation == 0 or step + 1 == len(train_loader):\\n                    scaler.unscale_(optimizer)\\n                    # Dynamic fp16 loss scaling may overflow scaled gradients even\\n                    # with a finite loss. GradScaler records that at unscale_,\\n                    # skips the unsafe update, and lowers its scale for retry.\\n                    torch.nn.utils.clip_grad_norm_(\\n                        model.parameters(), config.max_grad_norm, error_if_nonfinite=not cuda,\\n                    )\\n                    previous_scale = scaler.get_scale()\\n                    scaler.step(optimizer)\\n                    scaler.update()\\n                    if scaler.get_scale() < previous_scale:\\n                        skipped_amp_updates += 1\\n                        consecutive_amp_skips += 1\\n                        if consecutive_amp_skips >= config.max_consecutive_amp_skips:\\n                            raise ValueError(\\"Persistent non-finite CUDA gradients despite AMP scale backoff\\")\\n                    else:\\n                        scheduler.step()\\n                        optimizer_updates += 1\\n                        consecutive_amp_skips = 0\\n                    optimizer.zero_grad(set_to_none=True)\\n            epoch_logs.append({\\"epoch\\": epoch + 1, \\"mean_training_loss\\": total_loss / len(fitting_rows),\\n                               \\"elapsed_seconds\\": time.perf_counter() - epoch_started,\\n                               \\"learning_rate_after_epoch\\": scheduler.get_last_lr()[0]})\\n            print(json.dumps({\\"fold\\": fold, **epoch_logs[-1]}), flush=True)\\n        if optimizer_updates == 0:\\n            raise ValueError(\\"Fine-tuning completed without a successful optimizer update\\")\\n        # One validation pass after exactly the predeclared number of epochs.\\n        validation_started = time.perf_counter()\\n        model.eval()\\n        chunks = []\\n        with torch.inference_mode():\\n            for batch in valid_loader:\\n                batch.pop(\\"labels\\")\\n                inputs = {key: value.to(device) for key, value in batch.items()}\\n                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=cuda):\\n                    logits = model(**inputs).logits\\n                chunks.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())\\n        probabilities = np.concatenate(chunks)\\n        if not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5):\\n            raise ValueError(\\"Non-finite or invalid DeBERTa validation probabilities\\")\\n        if cuda:\\n            torch.cuda.synchronize(device)\\n        log = {\\n            \\"fold\\": fold, \\"seed\\": fold_seed, \\"train_rows\\": len(fitting), \\"validation_rows\\": len(validation),\\n            \\"train_prompts\\": int(fitting[\\"prompt_id\\"].nunique()),\\n            \\"validation_prompts\\": int(validation[\\"prompt_id\\"].nunique()),\\n            \\"training_selected_rows\\": int(fitting[\\"selected\\"].sum()),\\n            \\"validation_selected_rows\\": int(validation[\\"selected\\"].sum()),\\n            \\"optimizer_updates\\": optimizer_updates, \\"skipped_amp_updates\\": skipped_amp_updates,\\n            \\"amp_initial_scale\\": config.amp_initial_scale if cuda else None,\\n            \\"amp_final_scale\\": scaler.get_scale() if cuda else None,\\n            \\"epochs\\": epoch_logs,\\n            \\"validation_elapsed_seconds\\": time.perf_counter() - validation_started,\\n            \\"elapsed_seconds\\": time.perf_counter() - started,\\n            \\"device\\": str(device), \\"gpu\\": torch.cuda.get_device_name(device) if cuda else None,\\n            \\"precision\\": \\"float32 parameters with CUDA float16 autocast/GradScaler\\" if cuda else \\"float32 CPU diagnostic\\",\\n            \\"peak_cuda_allocated_bytes\\": torch.cuda.max_memory_allocated(device) if cuda else None,\\n            \\"peak_cuda_reserved_bytes\\": torch.cuda.max_memory_reserved(device) if cuda else None,\\n        }\\n        return probabilities, log\\n    finally:\\n        del model, optimizer, scheduler, scaler\\n        gc.collect()\\n        if cuda:\\n            torch.cuda.empty_cache()\\n\\n\\ndef _cohort_metrics(frame: pd.DataFrame) -> dict:\\n    truth, prediction = frame[\\"model\\"], frame[\\"prediction\\"]\\n    baseline_correct = frame[\\"baseline_correct\\"].to_numpy(dtype=bool)\\n    specialist_correct = (truth == prediction).to_numpy(dtype=bool)\\n    return {\\n        \\"rows\\": len(frame), \\"baseline_correct\\": int(baseline_correct.sum()),\\n        \\"specialist_correct\\": int(specialist_correct.sum()),\\n        \\"baseline_accuracy\\": float(baseline_correct.mean()),\\n        \\"specialist_accuracy\\": float(accuracy_score(truth, prediction)),\\n        \\"specialist_balanced_accuracy\\": float(balanced_accuracy_score(truth, prediction)),\\n        \\"accuracy_delta\\": float(specialist_correct.mean() - baseline_correct.mean()),\\n        \\"paired_transitions\\": {\\n            \\"both_correct\\": int((baseline_correct & specialist_correct).sum()),\\n            \\"baseline_wrong_specialist_correct\\": int((~baseline_correct & specialist_correct).sum()),\\n            \\"baseline_correct_specialist_wrong\\": int((baseline_correct & ~specialist_correct).sum()),\\n            \\"both_wrong\\": int((~baseline_correct & ~specialist_correct).sum()),\\n        },\\n    }\\n\\n\\ndef _save_class_reports(frame: pd.DataFrame, directory: Path, suffix: str) -> None:\\n    precision, recall, f1, support = precision_recall_fscore_support(\\n        frame[\\"model\\"], frame[\\"prediction\\"], labels=list(GEMINI_LABELS), zero_division=0,\\n    )\\n    baseline_recall = frame.groupby(\\"model\\")[\\"baseline_correct\\"].mean().reindex(GEMINI_LABELS)\\n    pd.DataFrame({\\"model\\": GEMINI_LABELS, \\"precision\\": precision, \\"recall\\": recall,\\n                  \\"f1\\": f1, \\"support\\": support,\\n                  \\"baseline_recall\\": baseline_recall.to_numpy(),\\n                  \\"recall_delta\\": recall - baseline_recall.to_numpy()}).to_csv(\\n                      directory / f\\"class_metrics{suffix}.csv\\", index=False)\\n    pd.DataFrame(confusion_matrix(frame[\\"model\\"], frame[\\"prediction\\"], labels=list(GEMINI_LABELS)),\\n                 index=pd.Index(GEMINI_LABELS, name=\\"true_model\\"), columns=GEMINI_LABELS).to_csv(\\n                     directory / f\\"confusion_matrix{suffix}.csv\\")\\n\\n\\ndef run_experiment(\\n    train: pd.DataFrame, baseline_oof: pd.DataFrame, fold_manifest_path: str | Path,\\n    model_path: str | Path, output_dir: str | Path, config: GeminiConfig | None = None,\\n    expected_counts: dict[str, int] | None = OFFICIAL_BASELINE_COUNTS,\\n) -> dict:\\n    config = config or GeminiConfig()\\n    config.validate()\\n    gemini, protocol = prepare_gemini_evaluation(train, baseline_oof, fold_manifest_path, expected_counts)\\n    manifest = validate_model_snapshot(model_path)\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    if (output / \\"metrics.json\\").exists() or (output / \\"oof.parquet\\").exists():\\n        raise ValueError(\\"Use a new output directory; an existing experiment must not be overwritten\\")\\n    versions = {}\\n    for package in (\\"torch\\", \\"transformers\\", \\"numpy\\", \\"pandas\\", \\"scikit-learn\\", \\"sentencepiece\\"):\\n        try:\\n            versions[package] = importlib.metadata.version(package)\\n        except importlib.metadata.PackageNotFoundError:\\n            versions[package] = None\\n    metadata = {\\"schema_version\\": 1, \\"kind\\": \\"gemini-four-class-full-finetuning-oof\\",\\n                \\"model_id\\": MODEL_ID, \\"model_revision\\": MODEL_REVISION,\\n                \\"configuration\\": asdict(config), \\"protocol\\": protocol, \\"versions\\": versions,\\n                \\"snapshot_manifest_sha256\\": hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest(),\\n                \\"checkpoint_policy\\": \\"No full checkpoints saved; diagnostic OOF results only.\\",\\n                \\"seed_policy\\": \\"Fresh pretrained backbone/head each fold, seed + fold; bitwise CUDA determinism is not promised.\\"}\\n    _json(output / \\"metadata.json\\", metadata)\\n    full_manifest, fold_metadata = load_fold_manifest(train, fold_manifest_path)\\n    save_fold_manifest(full_manifest, fold_metadata, output)\\n    probabilities = np.full((len(gemini), len(GEMINI_LABELS)), np.nan)\\n    predicted_count = np.zeros(len(gemini), dtype=np.int64)\\n    training_logs = []\\n    started = time.perf_counter()\\n    for fold in range(5):\\n        fitting = gemini.loc[gemini[\\"fold\\"] != fold].copy()\\n        valid_mask = gemini[\\"fold\\"] == fold\\n        validation = gemini.loc[valid_mask].copy()\\n        # Fitting rows are deliberately not filtered by `selected`.\\n        if set(fitting[\\"prompt_id\\"]) & set(validation[\\"prompt_id\\"]):\\n            raise ValueError(\\"Prompt leakage detected before fine-tuning\\")\\n        values, log = _fit_predict_fold(fitting, validation, model_path, config, fold)\\n        if values.shape != (len(validation), len(GEMINI_LABELS)) or not np.isfinite(values).all():\\n            raise ValueError(\\"Fold prediction shape or finite-value validation failed\\")\\n        if (values < 0).any() or not np.allclose(values.sum(axis=1), 1.0, atol=1e-5):\\n            raise ValueError(\\"Fold probabilities must be nonnegative and sum to one\\")\\n        probabilities[valid_mask] = values\\n        predicted_count[valid_mask] += 1\\n        training_logs.append(log)\\n        _json(output / \\"training_log.json\\", training_logs)\\n    if not np.all(predicted_count == 1) or not np.isfinite(probabilities).all():\\n        raise ValueError(\\"Every true-Gemini row must receive exactly one OOF prediction\\")\\n    result = gemini[[\\"id\\", \\"prompt_id\\", \\"model\\", \\"fold\\", \\"baseline_model\\", \\"selected\\", \\"baseline_correct\\"]].copy()\\n    result[\\"prediction\\"] = np.asarray(GEMINI_LABELS)[probabilities.argmax(axis=1)]\\n    for index, label in enumerate(GEMINI_LABELS):\\n        result[f\\"probability::{label}\\"] = probabilities[:, index]\\n    selected = result.loc[result[\\"selected\\"]].copy()\\n    metrics = {\\"all_true_gemini\\": _cohort_metrics(result),\\n               \\"correctly_routed_gemini\\": _cohort_metrics(selected),\\n               \\"elapsed_seconds\\": time.perf_counter() - started,\\n               \\"interpretation\\": \\"Conditional exact-version attribution on saved CPU-routed true Gemini; not an end-to-end family router or hidden-test score.\\"}\\n    result.to_parquet(output / \\"oof.parquet\\", index=False)\\n    selected.to_parquet(output / \\"selected_oof.parquet\\", index=False)\\n    _save_class_reports(result, output, \\"\\")\\n    _save_class_reports(selected, output, \\"_selected\\")\\n    _json(output / \\"metrics.json\\", metrics)\\n    return metrics\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--baseline-oof\\", type=Path, required=True)\\n    parser.add_argument(\\"--fold-manifest\\", type=Path, required=True)\\n    parser.add_argument(\\"--model-path\\", type=Path, required=True)\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    parser.add_argument(\\"--epochs\\", type=int, default=3)\\n    parser.add_argument(\\"--learning-rate\\", type=float, default=2e-5)\\n    parser.add_argument(\\"--max-length\\", type=int, default=512)\\n    parser.add_argument(\\"--batch-size\\", type=int, default=8)\\n    parser.add_argument(\\"--gradient-accumulation\\", type=int, default=2)\\n    parser.add_argument(\\"--warmup-ratio\\", type=float, default=0.1)\\n    parser.add_argument(\\"--seed\\", type=int, default=2026)\\n    parser.add_argument(\\"--device\\", default=\\"cuda:0\\")\\n    args = parser.parse_args(argv)\\n    config = GeminiConfig(epochs=args.epochs, learning_rate=args.learning_rate,\\n                          max_length=args.max_length, batch_size=args.batch_size,\\n                          gradient_accumulation=args.gradient_accumulation,\\n                          warmup_ratio=args.warmup_ratio, seed=args.seed, device=args.device)\\n    metrics = run_experiment(read_train(args.train), pd.read_parquet(args.baseline_oof),\\n                             args.fold_manifest, args.model_path, args.output_dir, config)\\n    print(json.dumps(metrics, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/lora.py": "\\"\\"\\"QLoRA adaptation of the frozen Qwen3-8B-Base recipe for 18-class attribution.\\n\\nThe pinned NF4 backbone, tokenizer, 512-token limit, 24 retained blocks and masked mean\\npooling of blocks 7 and 22 are unchanged. The head is the frozen pipeline itself: the\\ntraining partition\'s standardization and a linear layer initialised from the C=0.005\\nlogistic fit, trained against that same L2-penalised objective. Rank-16 adapters then train\\njointly with it (linear probe, then fine-tune); with adapters disabled the identical loop is\\nthe control. The number of epochs, including zero, is chosen by inner prompt folds of each\\nouter training partition, so outer validation rows never select anything.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport time\\nfrom dataclasses import asdict, dataclass, replace\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import balanced_accuracy_score\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom . import ensemble, neural\\nfrom .constants import LABELS\\nfrom .data import fingerprint_frame, read_train\\nfrom .folds import inner_prompt_splits, load_fold_manifest\\nfrom .selection import first_best\\n\\n\\n@dataclass(frozen=True)\\nclass AdapterConfig:\\n    rank: int = 16\\n    alpha: int = 32\\n    dropout: float = 0.05\\n    target_modules: tuple[str, ...] = (\\"q_proj\\", \\"k_proj\\", \\"v_proj\\", \\"o_proj\\", \\"gate_proj\\", \\"up_proj\\", \\"down_proj\\")\\n    # Blocks after the last pooled block cannot change the pooled features.\\n    adapted_layers: int = 22\\n    # Gentler than the first run (1e-4, whole epochs), which overfit within its first epoch.\\n    learning_rates: tuple[float, ...] = (1e-5, 3e-5)\\n    # Training durations in epochs; 0 is the frozen head itself, for every learning rate.\\n    durations: tuple[float, ...] = (0.0, 0.25, 0.5, 1.0, 2.0)\\n    warmup_steps: int = 10\\n    micro_batch: int = 4\\n    accumulation: int = 4\\n    max_grad_norm: float = 1.0\\n    eval_batch: int = 8\\n    inner_folds: int = 2\\n    seed: int = 2026\\n\\n    def validate(self) -> None:\\n        if min(self.rank, self.alpha, self.micro_batch, self.accumulation, self.eval_batch, self.warmup_steps) < 1:\\n            raise ValueError(\\"Adapter sizes, batches and warmup must be positive\\")\\n        if 0 not in self.durations or min(self.durations) < 0 or len(set(self.durations)) != len(self.durations):\\n            raise ValueError(\\"Durations must be distinct, nonnegative and include 0 (the frozen head)\\")\\n        if not self.learning_rates or min(self.learning_rates) <= 0 or not 0 <= self.dropout < 1 or self.inner_folds < 2:\\n            raise ValueError(\\"Invalid learning rates, dropout or inner fold count\\")\\n\\n    def settings(self) -> list[tuple[float, float]]:\\n        \\"\\"\\"(learning rate, duration) candidates; the frozen head is one candidate, not one per rate.\\"\\"\\"\\n        return [(0.0, 0.0)] + [(rate, duration) for rate in sorted(self.learning_rates)\\n                               for duration in sorted(self.durations) if duration > 0]\\n\\n    @property\\n    def trained_durations(self) -> list[float]:\\n        return sorted(duration for duration in self.durations if duration > 0)\\n\\n\\ndef setting_name(rate: float, duration: float) -> str:\\n    return \\"frozen\\" if duration == 0 else f\\"lr{rate:g}-d{duration:g}\\"\\n\\n\\ndef _json(path: Path, value: dict | list) -> None:\\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef split_plan(train: pd.DataFrame, manifest_path: str | Path, config: AdapterConfig) -> pd.DataFrame:\\n    \\"\\"\\"Original outer prompt folds plus the inner prompt folds of every outer training partition.\\"\\"\\"\\n    manifest, metadata = load_fold_manifest(train, manifest_path)\\n    if metadata[\\"n_splits\\"] != 5:\\n        raise ValueError(\\"The adaptation experiment uses the original five prompt folds\\")\\n    plan = train[[\\"id\\", \\"prompt_id\\"]].merge(manifest[[\\"id\\", \\"fold\\"]], on=\\"id\\", validate=\\"one_to_one\\")\\n    plan = plan.rename(columns={\\"fold\\": \\"outer_fold\\"})\\n    targets, prompts = ensemble._targets(train), plan[\\"prompt_id\\"].to_numpy()\\n    for fold in range(5):\\n        fit = np.flatnonzero(plan[\\"outer_fold\\"].to_numpy() != fold)\\n        splits, _ = inner_prompt_splits(targets[fit], prompts[fit], config.inner_folds, config.seed + fold)\\n        inner = np.full(len(plan), -1)\\n        for inner_fold, (_, valid) in enumerate(splits):\\n            inner[fit[valid]] = inner_fold\\n        plan[f\\"inner_{fold}\\"] = inner\\n    return plan\\n\\n\\ndef validate_plan(train: pd.DataFrame, plan: pd.DataFrame, config: AdapterConfig) -> pd.DataFrame:\\n    plan = train[[\\"id\\"]].merge(plan, on=\\"id\\", how=\\"left\\", validate=\\"one_to_one\\")\\n    if plan.isna().any().any() or (plan[\\"prompt_id\\"] != train[\\"prompt_id\\"]).any():\\n        raise ValueError(\\"Split plan must cover every training row with its own prompt\\")\\n    if set(plan[\\"outer_fold\\"]) != set(range(5)):\\n        raise ValueError(\\"Outer folds must be exactly the five folds 0-4\\")\\n    if plan.groupby(\\"prompt_id\\")[\\"outer_fold\\"].nunique().max() != 1:\\n        raise ValueError(\\"Outer folds must keep prompts together\\")\\n    for fold in range(5):\\n        column = plan[f\\"inner_{fold}\\"].to_numpy()\\n        outer = plan[\\"outer_fold\\"].to_numpy() == fold\\n        if (column[outer] != -1).any() or set(column[~outer]) != set(range(config.inner_folds)):\\n            raise ValueError(\\"Inner folds must partition exactly the outer training rows\\")\\n        if plan.loc[~outer].groupby(\\"prompt_id\\")[f\\"inner_{fold}\\"].nunique().max() != 1:\\n            raise ValueError(\\"Inner folds must keep prompts together\\")\\n    return plan\\n\\n\\ndef fit_probe(features: np.ndarray, targets: np.ndarray, frozen: neural.FrozenConfig) -> dict:\\n    \\"\\"\\"The frozen recipe\'s head, fit on a training partition: the head\'s starting point.\\"\\"\\"\\n    head = neural._head(frozen)\\n    with threadpool_limits(limits=2):\\n        head.fit(features, targets)\\n    scaler, classifier = head.steps[0][1], head.steps[-1][1]\\n    if not np.array_equal(classifier.classes_, np.arange(len(LABELS))):\\n        raise ValueError(\\"Probe head must contain every canonical label\\")\\n    return {\\"mean\\": scaler.mean_, \\"scale\\": scaler.scale_, \\"weight\\": classifier.coef_,\\n            \\"bias\\": classifier.intercept_, \\"pipeline\\": head}\\n\\n\\ndef head_module(torch, probe: dict, device):\\n    class StandardizedLinear(torch.nn.Module):\\n        def __init__(self):\\n            super().__init__()\\n            self.register_buffer(\\"mean\\", torch.as_tensor(probe[\\"mean\\"], dtype=torch.float32))\\n            self.register_buffer(\\"scale\\", torch.as_tensor(probe[\\"scale\\"], dtype=torch.float32))\\n            self.linear = torch.nn.Linear(probe[\\"weight\\"].shape[1], probe[\\"weight\\"].shape[0])\\n            with torch.no_grad():\\n                self.linear.weight.copy_(torch.as_tensor(probe[\\"weight\\"], dtype=torch.float32))\\n                self.linear.bias.copy_(torch.as_tensor(probe[\\"bias\\"], dtype=torch.float32))\\n\\n        def forward(self, pooled):\\n            return self.linear((pooled.float() - self.mean) / self.scale)\\n\\n    return StandardizedLinear().to(device)\\n\\n\\ndef _batches(lengths: np.ndarray, size: int, seed: int) -> list[np.ndarray]:\\n    \\"\\"\\"Shuffle, sort within chunks of 16 batches by length (less padding), shuffle batches.\\"\\"\\"\\n    rng = np.random.default_rng(seed)\\n    order = rng.permutation(len(lengths))\\n    batches = []\\n    for start in range(0, len(order), size * 16):\\n        chunk = order[start:start + size * 16]\\n        chunk = chunk[np.argsort(lengths[chunk], kind=\\"stable\\")]\\n        batches += [chunk[index:index + size] for index in range(0, len(chunk), size)]\\n    return [batches[index] for index in rng.permutation(len(batches))]\\n\\n\\nclass Experiment:\\n    \\"\\"\\"One process: the backbone (if adapters are on) is loaded once and reset between runs.\\"\\"\\"\\n\\n    def __init__(self, train: pd.DataFrame, cached: np.ndarray, *, adapters: bool, model_path=None,\\n                 frozen: neural.FrozenConfig | None = None, config: AdapterConfig | None = None,\\n                 device: str = \\"cpu\\"):\\n        import torch\\n\\n        self.torch, self.adapters = torch, adapters\\n        self.frozen = replace(frozen or neural.FrozenConfig(), device=device)\\n        self.config = config or AdapterConfig()\\n        self.config.validate()\\n        self.texts = train[\\"text\\"].tolist()\\n        self.targets = ensemble._targets(train)\\n        self.cached = np.asarray(cached, dtype=np.float32)\\n        if self.cached.shape[0] != len(train):\\n            raise ValueError(\\"Cached frozen features must align with the training rows\\")\\n        self.device = device if not adapters else None\\n        if model_path is None:\\n            raise ValueError(\\"model_path is required: the control also needs the backbone\'s tokenizer\\")\\n        if adapters:\\n            from peft import LoraConfig, get_peft_model\\n\\n            tokenizer, model, device, _, self.manifest = neural.load_backbone(model_path, self.frozen)\\n            model.config.use_cache = False\\n            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\\"use_reentrant\\": False})\\n            model.enable_input_require_grads()\\n            # The bare Qwen3Model names blocks \\"layers.N\\" with no prefix, which peft\'s\\n            # layers_to_transform cannot match; a full-name pattern selects the same modules.\\n            layers = \\"|\\".join(str(index) for index in range(self.config.adapted_layers))\\n            modules = \\"|\\".join(self.config.target_modules)\\n            self.model = get_peft_model(model, LoraConfig(\\n                r=self.config.rank, lora_alpha=self.config.alpha, lora_dropout=self.config.dropout, bias=\\"none\\",\\n                target_modules=rf\\"(?:.*\\\\.)?layers\\\\.(?:{layers})\\\\.(?:self_attn|mlp)\\\\.(?:{modules})\\"))\\n            self.tokenizer, self.device = tokenizer, device\\n            # Every run restarts from the same adapter initialisation (B = 0: the frozen backbone).\\n            self.initial = {name: value.detach().clone() for name, value in self.model.named_parameters()\\n                            if value.requires_grad}\\n        else:\\n            self.manifest = neural.validate_snapshot(model_path, weights=False)\\n            tokenizer = neural.load_tokenizer(model_path)\\n        # Batches are bucketed by token length, so the control needs the same lengths to train\\n        # on exactly the adapted model\'s batches.\\n        self.lengths = np.asarray(tokenizer(self.texts, truncation=True, max_length=self.frozen.max_length,\\n                                            return_length=True)[\\"length\\"])\\n        self.cuda = str(self.device).startswith(\\"cuda\\")\\n\\n    def reset(self) -> None:\\n        \\"\\"\\"Restore the adapter initialisation (B = 0: the frozen backbone); no-op for the control.\\"\\"\\"\\n        if self.adapters:\\n            with self.torch.no_grad():\\n                for name, value in self.model.named_parameters():\\n                    if name in self.initial:\\n                        value.copy_(self.initial[name])\\n\\n    def trainable_parameters(self) -> int:\\n        return sum(p.numel() for p in self.model.parameters() if p.requires_grad) if self.adapters else 0\\n\\n    def _pooled(self, rows: np.ndarray):\\n        if not self.adapters:\\n            return self.torch.as_tensor(self.cached[rows], device=self.device)\\n        encoded = self.tokenizer([self.texts[index] for index in rows], padding=True, truncation=True,\\n                                 max_length=self.frozen.max_length, return_tensors=\\"pt\\").to(self.device)\\n        output = self.model(**encoded, output_hidden_states=True, use_cache=False)\\n        return neural.masked_layer_mean(output.hidden_states, encoded[\\"attention_mask\\"], self.frozen.pool_layers)\\n\\n    def predict(self, head, rows: np.ndarray) -> np.ndarray:\\n        torch = self.torch\\n        probabilities = np.empty((len(rows), len(LABELS)))\\n        order = np.argsort(self.lengths[rows], kind=\\"stable\\")\\n        if self.adapters:\\n            self.model.eval()\\n        head.eval()\\n        with torch.inference_mode():\\n            for start in range(0, len(rows), self.config.eval_batch):\\n                positions = order[start:start + self.config.eval_batch]\\n                logits = head(self._pooled(rows[positions]))\\n                probabilities[positions] = torch.softmax(logits.float(), dim=-1).cpu().numpy()\\n        if not np.isfinite(probabilities).all():\\n            raise ValueError(\\"Non-finite adapted probabilities\\")\\n        return probabilities\\n\\n    def run(self, fit_rows: np.ndarray, eval_rows: np.ndarray, learning_rate: float,\\n            checkpoints: list[float]) -> dict:\\n        \\"\\"\\"Probe head on the fit rows, then train adapters and head at one learning rate.\\n\\n        Returns eval probabilities at duration 0 (the frozen head) and at every checkpoint, in\\n        epochs. The learning rate is constant after warmup, so the checkpoint at d epochs is the\\n        model trained for exactly d epochs.\\n        \\"\\"\\"\\n        torch = self.torch\\n        probe = fit_probe(self.cached[fit_rows], self.targets[fit_rows], self.frozen)\\n        head = head_module(torch, probe, self.device)\\n        result = {\\"probabilities\\": {}, \\"checkpoints\\": [], \\"head\\": head, \\"probe\\": probe}\\n        if len(eval_rows):\\n            result[\\"probabilities\\"][0.0] = probe[\\"pipeline\\"].predict_proba(self.cached[eval_rows])\\n        # Reset even when nothing trains, so a zero-duration run never keeps an earlier run\'s adapters.\\n        self.reset()\\n        if not checkpoints:\\n            return result\\n        torch.manual_seed(self.config.seed)\\n        if self.adapters:\\n            self.model.train()\\n        head.train()\\n        parameters = list(head.parameters())\\n        if self.adapters:\\n            parameters += [p for p in self.model.parameters() if p.requires_grad]\\n        optimizer = torch.optim.AdamW(parameters, lr=learning_rate, weight_decay=0.0)\\n        scheduler = torch.optim.lr_scheduler.LambdaLR(\\n            optimizer, lambda step: min(1.0, (step + 1) / self.config.warmup_steps))\\n        scaler = torch.amp.GradScaler(\\"cuda\\", enabled=self.cuda)\\n        # Mean cross-entropy + ||W||^2 / (2 C n): the frozen logistic objective per row.\\n        l2 = 1.0 / (self.frozen.C * len(fit_rows))\\n        groups_per_epoch = -(-len(_batches(self.lengths[fit_rows], self.config.micro_batch, 0))\\n                             // self.config.accumulation)\\n        targets = {}\\n        for duration in sorted(checkpoints):\\n            targets.setdefault(max(1, round(duration * groups_per_epoch)), []).append(duration)\\n        last = max(targets)\\n        if self.cuda:\\n            torch.cuda.reset_peak_memory_stats(self.device)\\n        step, epoch, started, objective, since, skipped = 0, 0, time.perf_counter(), 0.0, 0, 0\\n        while step < last:\\n            epoch += 1\\n            batches = _batches(self.lengths[fit_rows], self.config.micro_batch, self.config.seed + epoch)\\n            for first in range(0, len(batches), self.config.accumulation):\\n                members = batches[first:first + self.config.accumulation]\\n                rows_in_group = sum(len(batch) for batch in members)\\n                for position, batch in enumerate(members):\\n                    rows = fit_rows[batch]\\n                    logits = head(self._pooled(rows))\\n                    labels = torch.as_tensor(self.targets[rows], device=self.device)\\n                    loss = torch.nn.functional.cross_entropy(logits.float(), labels, reduction=\\"sum\\") / rows_in_group\\n                    if position == 0:\\n                        loss = loss + 0.5 * l2 * head.linear.weight.pow(2).sum()\\n                    if not torch.isfinite(loss):\\n                        raise ValueError(f\\"Non-finite adaptation loss at step {step + 1}\\")\\n                    objective += float(loss.detach())\\n                    scaler.scale(loss).backward()\\n                scaler.unscale_(optimizer)\\n                torch.nn.utils.clip_grad_norm_(parameters, self.config.max_grad_norm)\\n                scale = scaler.get_scale()\\n                scaler.step(optimizer)\\n                scaler.update()\\n                skipped += int(scaler.get_scale() < scale)\\n                scheduler.step()\\n                optimizer.zero_grad(set_to_none=True)\\n                step, since = step + 1, since + 1\\n                if step in targets:\\n                    record = {\\"durations\\": targets[step], \\"optimizer_steps\\": step, \\"learning_rate\\": learning_rate,\\n                              \\"mean_objective\\": objective / since, \\"skipped_amp_steps\\": skipped,\\n                              \\"train_seconds\\": time.perf_counter() - started, \\"train_rows\\": len(fit_rows)}\\n                    if len(eval_rows):\\n                        evaluated = time.perf_counter()\\n                        probabilities = self.predict(head, eval_rows)\\n                        for duration in targets[step]:\\n                            result[\\"probabilities\\"][duration] = probabilities\\n                        record[\\"eval_seconds\\"] = time.perf_counter() - evaluated\\n                        if self.adapters:\\n                            self.model.train()\\n                        head.train()\\n                    if self.cuda:\\n                        record[\\"peak_cuda_allocated_bytes\\"] = torch.cuda.max_memory_allocated(self.device)\\n                    result[\\"checkpoints\\"].append(record)\\n                    print(json.dumps(record), flush=True)\\n                    started, objective, since = time.perf_counter(), 0.0, 0\\n                if step >= last:\\n                    break\\n        return result\\n\\n\\ndef run_fold(train: pd.DataFrame, plan: pd.DataFrame, experiment: Experiment, outer_fold: int,\\n             output_dir: str | Path) -> dict:\\n    \\"\\"\\"Inner folds choose the (learning rate, duration); then train once on the outer partition.\\"\\"\\"\\n    config = experiment.config\\n    plan = validate_plan(train, plan, config)\\n    output = Path(output_dir)\\n    if (output / \\"outer.parquet\\").exists():\\n        raise ValueError(\\"Use a new output directory; an existing fold must not be overwritten\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    outer = plan[\\"outer_fold\\"].to_numpy()\\n    inner = plan[f\\"inner_{outer_fold}\\"].to_numpy()\\n    settings = config.settings()\\n    started = time.perf_counter()\\n    inner_rows, inner_logs, scores = [], [], {setting: [] for setting in settings}\\n    for inner_fold in range(config.inner_folds):\\n        fit_rows = np.flatnonzero((outer != outer_fold) & (inner != inner_fold))\\n        eval_rows = np.flatnonzero(inner == inner_fold)\\n        probabilities = {}\\n        for rate in sorted(config.learning_rates):\\n            result = experiment.run(fit_rows, eval_rows, rate, config.trained_durations)\\n            probabilities[(0.0, 0.0)] = result[\\"probabilities\\"][0.0]\\n            probabilities.update({(rate, duration): result[\\"probabilities\\"][duration]\\n                                  for duration in config.trained_durations})\\n            inner_logs.append({\\"inner_fold\\": inner_fold, \\"learning_rate\\": rate, \\"fit_rows\\": len(fit_rows),\\n                               \\"eval_rows\\": len(eval_rows), \\"checkpoints\\": result[\\"checkpoints\\"]})\\n        for setting in settings:\\n            values = probabilities[setting]\\n            scores[setting].append(float(balanced_accuracy_score(experiment.targets[eval_rows],\\n                                                                 values.argmax(axis=1))))\\n            frame = pd.DataFrame(values, columns=[f\\"probability::{label}\\" for label in LABELS])\\n            frame.insert(0, \\"duration\\", setting[1])\\n            frame.insert(0, \\"learning_rate\\", setting[0])\\n            frame.insert(0, \\"setting\\", setting_name(*setting))\\n            frame.insert(0, \\"inner_fold\\", inner_fold)\\n            frame.insert(0, \\"id\\", train[\\"id\\"].to_numpy()[eval_rows])\\n            inner_rows.append(frame)\\n    means = {setting: float(np.mean(values)) for setting, values in scores.items()}\\n    # Ties keep the shorter duration (zero is the frozen head), then the smaller learning rate.\\n    rate, duration = first_best(sorted(settings, key=lambda setting: (setting[1], setting[0])), means.__getitem__)\\n    fit_rows, eval_rows = np.flatnonzero(outer != outer_fold), np.flatnonzero(outer == outer_fold)\\n    result = experiment.run(fit_rows, eval_rows, rate, [duration] if duration > 0 else [])\\n    probabilities = result[\\"probabilities\\"][duration]\\n    outer_frame = pd.DataFrame(probabilities, columns=[f\\"probability::{label}\\" for label in LABELS])\\n    outer_frame.insert(0, \\"duration\\", duration)\\n    outer_frame.insert(0, \\"learning_rate\\", rate)\\n    outer_frame.insert(0, \\"setting\\", setting_name(rate, duration))\\n    outer_frame.insert(0, \\"id\\", train[\\"id\\"].to_numpy()[eval_rows])\\n    pd.concat(inner_rows, ignore_index=True).to_parquet(output / \\"inner.parquet\\", index=False)\\n    outer_frame.to_parquet(output / \\"outer.parquet\\", index=False)\\n    log = {\\"outer_fold\\": outer_fold, \\"adapters\\": experiment.adapters,\\n           \\"model_revision\\": experiment.manifest.get(\\"revision\\"),\\n           \\"selected_setting\\": setting_name(rate, duration), \\"selected_learning_rate\\": rate,\\n           \\"selected_duration\\": duration,\\n           \\"inner_balanced_accuracy\\": {setting_name(*key): values for key, values in scores.items()},\\n           \\"inner_mean_balanced_accuracy\\": {setting_name(*key): value for key, value in means.items()},\\n           \\"outer_balanced_accuracy\\": float(balanced_accuracy_score(experiment.targets[eval_rows],\\n                                                                    probabilities.argmax(axis=1))),\\n           \\"inner_runs\\": inner_logs, \\"outer_run\\": result[\\"checkpoints\\"],\\n           \\"elapsed_seconds\\": time.perf_counter() - started,\\n           \\"trainable_parameters\\": experiment.trainable_parameters(),\\n           \\"configuration\\": asdict(config), \\"frozen\\": asdict(experiment.frozen)}\\n    _json(output / \\"log.json\\", log)\\n    print(json.dumps({key: log[key] for key in (\\"outer_fold\\", \\"selected_setting\\", \\"inner_mean_balanced_accuracy\\",\\n                                                \\"outer_balanced_accuracy\\", \\"elapsed_seconds\\")}), flush=True)\\n    return log\\n\\n\\ndef probe_training(train: pd.DataFrame, plan: pd.DataFrame, experiment: Experiment, output_dir: str | Path,\\n                   per_class: int = 4, epochs: int = 6) -> dict:\\n    \\"\\"\\"Small check: init features match the frozen archive; memory, speed, and fitting a subset.\\"\\"\\"\\n    torch = experiment.torch\\n    plan = validate_plan(train, plan, experiment.config)\\n    subset = (plan.loc[plan[\\"outer_fold\\"] != 0].assign(model=train[\\"model\\"].to_numpy()[plan[\\"outer_fold\\"] != 0])\\n              .groupby(\\"model\\", group_keys=False).sample(n=per_class, random_state=experiment.config.seed))\\n    rows = np.sort(train.index.get_indexer(subset.index))\\n    report = {\\"rows\\": len(rows), \\"trainable_parameters\\": experiment.trainable_parameters()}\\n    if experiment.adapters:\\n        experiment.reset()  # the initial-feature check must not see an earlier run\'s adapters\\n        experiment.model.eval()\\n        with torch.inference_mode():\\n            start = time.perf_counter()\\n            pooled = torch.cat([experiment._pooled(rows[index:index + 4]) for index in range(0, len(rows), 4)])\\n            report[\\"init_feature_seconds_per_row\\"] = (time.perf_counter() - start) / len(rows)\\n        difference = np.abs(pooled.cpu().numpy() - experiment.cached[rows])\\n        report[\\"init_vs_frozen_archive\\"] = {\\"max_abs\\": float(difference.max()),\\n                                            \\"mean_abs\\": float(difference.mean()),\\n                                            \\"relative\\": float(difference.mean() / np.abs(experiment.cached[rows]).mean())}\\n    rate = max(experiment.config.learning_rates)\\n    result = experiment.run(rows, rows, rate, [float(epoch) for epoch in range(1, epochs + 1)])\\n    truth = experiment.targets[rows]\\n    report[\\"learning_rate\\"] = rate\\n    report[\\"fit_by_epoch\\"] = [\\n        {\\"epoch\\": epoch, \\"accuracy\\": float((probabilities.argmax(axis=1) == truth).mean()),\\n         \\"cross_entropy\\": float(-np.log(probabilities[np.arange(len(rows)), truth] + 1e-12).mean())}\\n        for epoch, probabilities in sorted(result[\\"probabilities\\"].items())]\\n    report[\\"epochs\\"] = result[\\"checkpoints\\"]\\n    train_seconds = sum(record[\\"train_seconds\\"] for record in result[\\"checkpoints\\"])\\n    report[\\"train_seconds_per_row_epoch\\"] = train_seconds / (len(rows) * epochs)\\n    report[\\"eval_seconds_per_row\\"] = float(np.mean([record[\\"eval_seconds\\"] for record in result[\\"checkpoints\\"]])) / len(rows)\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    _json(output / (\\"probe.json\\" if experiment.adapters else \\"probe_control.json\\"), report)\\n    print(json.dumps({key: value for key, value in report.items() if key != \\"epochs\\"}, indent=2), flush=True)\\n    return report\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    commands = parser.add_subparsers(dest=\\"command\\", required=True)\\n    plan_command = commands.add_parser(\\"plan\\", help=\\"Write the outer/inner prompt-fold plan\\")\\n    plan_command.add_argument(\\"--train\\", type=Path, required=True)\\n    plan_command.add_argument(\\"--fold-manifest\\", type=Path, required=True)\\n    plan_command.add_argument(\\"--output\\", type=Path, required=True)\\n    for name in (\\"probe\\", \\"fold\\"):\\n        sub = commands.add_parser(name)\\n        sub.add_argument(\\"--train\\", type=Path, required=True)\\n        sub.add_argument(\\"--plan\\", type=Path, required=True)\\n        sub.add_argument(\\"--features\\", type=Path, required=True)\\n        sub.add_argument(\\"--output-dir\\", type=Path, required=True)\\n        sub.add_argument(\\"--model-path\\", type=Path, required=True,\\n                         help=\\"Pinned snapshot; the control reads only its tokenizer\\")\\n        sub.add_argument(\\"--device\\", default=\\"cuda:0\\")\\n        sub.add_argument(\\"--control\\", action=\\"store_true\\", help=\\"Adapters disabled; head only, on cached features\\")\\n    commands.choices[\\"fold\\"].add_argument(\\"--outer-fold\\", type=int, nargs=\\"+\\", required=True)\\n    args = parser.parse_args(argv)\\n    train = read_train(args.train)\\n    if args.command == \\"plan\\":\\n        plan = split_plan(train, args.fold_manifest, AdapterConfig())\\n        plan.to_csv(args.output, index=False)\\n        print(json.dumps({\\"rows\\": len(plan), \\"train_fingerprint\\": fingerprint_frame(train)}))\\n        return 0\\n    plan = pd.read_csv(args.plan, dtype={\\"id\\": str, \\"prompt_id\\": str}, keep_default_na=False)\\n    cached, _ = ensemble.load_archive(args.features, train, neural.FrozenConfig())\\n    experiment = Experiment(train, cached, adapters=not args.control, model_path=args.model_path,\\n                            device=\\"cpu\\" if args.control else args.device)\\n    if args.command == \\"probe\\":\\n        probe_training(train, plan, experiment, args.output_dir)\\n        return 0\\n    for fold in args.outer_fold:\\n        run_fold(train, plan, experiment, fold, Path(args.output_dir) / f\\"fold{fold}\\")\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/lora_compare.py": "\\"\\"\\"Compare QLoRA-adapted Qwen (and its adapters-disabled control) with the validated system.\\n\\nThe current CPU-Qwen system\'s outer scores come from its nested validation. On each outer\\nfold\'s inner folds (those that chose the adapter setting) the current system is refit with\\nits own nested weight selection, so the weight v of (1 - v) * current + v * adapted is\\nchosen from inner rows only; v = 0 keeps the current system. Every system scores all 2,699\\nrows and is compared with the current system by a paired prompt bootstrap.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import balanced_accuracy_score\\n\\nfrom . import ensemble, neural\\nfrom .constants import LABELS\\nfrom .data import read_train\\nfrom .lora import AdapterConfig, validate_plan\\nfrom .selection import first_best\\n\\nPROBABILITIES = [f\\"probability::{label}\\" for label in LABELS]\\nWEIGHTS = (0.0, 0.25, 0.5, 0.75, 1.0)\\n\\n\\ndef load_runs(directory: str | Path, train: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame, list[dict]]:\\n    \\"\\"\\"Outer probabilities aligned to train rows, inner probabilities, and per-fold logs.\\"\\"\\"\\n    directory = Path(directory)\\n    outer = pd.concat([pd.read_parquet(directory / f\\"fold{fold}\\" / \\"outer.parquet\\") for fold in range(5)])\\n    inner = pd.concat([pd.read_parquet(directory / f\\"fold{fold}\\" / \\"inner.parquet\\").assign(outer_fold=fold)\\n                       for fold in range(5)])\\n    logs = [json.loads((directory / f\\"fold{fold}\\" / \\"log.json\\").read_text()) for fold in range(5)]\\n    if len(outer) != len(train) or set(outer[\\"id\\"]) != set(train[\\"id\\"]) or outer[\\"id\\"].duplicated().any():\\n        raise ValueError(f\\"{directory} must score every official row exactly once\\")\\n    aligned = outer.set_index(\\"id\\").loc[train[\\"id\\"], PROBABILITIES].to_numpy()\\n    return aligned, inner, logs\\n\\n\\ndef current_on_split(train: pd.DataFrame, features: np.ndarray, fit: np.ndarray, valid: np.ndarray,\\n                     config: ensemble.BlendConfig, seed_offset: int) -> tuple[np.ndarray, float]:\\n    \\"\\"\\"The current system refit on `fit` rows, with its own nested weight choice, scoring `valid`.\\"\\"\\"\\n    texts, labels = train[\\"text\\"].to_numpy(dtype=object), train[\\"model\\"].to_numpy(dtype=object)\\n    truth, prompts = ensemble._targets(train), train[\\"prompt_id\\"].to_numpy()\\n    weight, _ = ensemble._select_weight(texts[fit], labels[fit], features[fit], truth[fit], prompts[fit],\\n                                        config, seed_offset)\\n    cpu = ensemble._cpu_probabilities(texts[fit], labels[fit], texts[valid], config.cpu)\\n    qwen = ensemble._qwen_probabilities(features[fit], truth[fit], features[valid], config.qwen)\\n    return ensemble._blend(cpu, qwen, weight), weight\\n\\n\\ndef current_inner(train: pd.DataFrame, features: np.ndarray, plan: pd.DataFrame, outer_fold: int,\\n                  config: ensemble.BlendConfig) -> dict[int, tuple[np.ndarray, np.ndarray, float]]:\\n    \\"\\"\\"Refit the current system on each inner training split of one outer fold.\\"\\"\\"\\n    column, outer = plan[f\\"inner_{outer_fold}\\"].to_numpy(), plan[\\"outer_fold\\"].to_numpy()\\n    result = {}\\n    for inner_fold in sorted(set(column[outer != outer_fold])):\\n        fit, valid = np.flatnonzero((outer != outer_fold) & (column != inner_fold)), np.flatnonzero(column == inner_fold)\\n        probabilities, weight = current_on_split(train, features, fit, valid, config,\\n                                                 10 * (outer_fold + 1) + inner_fold)\\n        result[inner_fold] = (valid, probabilities, weight)\\n    return result\\n\\n\\ndef choose_weight(train: pd.DataFrame, current: dict, inner: pd.DataFrame, outer_fold: int, setting: str) -> dict:\\n    truth = ensemble._targets(train)\\n    position = pd.Series(np.arange(len(train)), index=train[\\"id\\"])\\n    scores = {weight: [] for weight in WEIGHTS}\\n    for inner_fold, (valid, current_probabilities, _) in current.items():\\n        rows = inner[(inner[\\"outer_fold\\"] == outer_fold) & (inner[\\"inner_fold\\"] == inner_fold)\\n                     & (inner[\\"setting\\"] == setting)]\\n        row_positions = position.loc[rows[\\"id\\"]].to_numpy()\\n        if len(row_positions) != len(valid) or set(row_positions) != set(valid):\\n            raise ValueError(\\"Inner adapted predictions must cover the inner validation rows exactly\\")\\n        index = {row: offset for offset, row in enumerate(valid)}\\n        adapted = np.zeros_like(current_probabilities)\\n        adapted[[index[value] for value in row_positions]] = rows[PROBABILITIES].to_numpy()\\n        for weight in WEIGHTS:\\n            blended = ensemble._blend(current_probabilities, adapted, weight)\\n            scores[weight].append(float(balanced_accuracy_score(truth[valid], blended.argmax(axis=1))))\\n    means = {weight: float(np.mean(values)) for weight, values in scores.items()}\\n    # Ties keep the smaller adapted weight, i.e. stay with the validated system.\\n    return {\\"weight\\": first_best(WEIGHTS, means.__getitem__),\\n            \\"inner_balanced_accuracy\\": {str(weight): values for weight, values in scores.items()}}\\n\\n\\ndef compare(train: pd.DataFrame, features: np.ndarray, plan: pd.DataFrame, current_oof: pd.DataFrame,\\n            runs: dict[str, str | Path], output_dir: str | Path, config: ensemble.BlendConfig | None = None,\\n            samples: int = 10000) -> dict:\\n    config = config or ensemble.BlendConfig()\\n    plan = validate_plan(train, plan, AdapterConfig())\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    truth, prompts, folds = ensemble._targets(train), train[\\"prompt_id\\"].to_numpy(), plan[\\"outer_fold\\"].to_numpy()\\n    current_frame = current_oof.set_index(\\"id\\").loc[train[\\"id\\"]]\\n    if (current_frame[\\"true_model\\"].to_numpy() != train[\\"model\\"].to_numpy()).any():\\n        raise ValueError(\\"Current-system OOF labels differ from the official rows\\")\\n    if (current_frame[\\"fold\\"].to_numpy() != folds).any():\\n        raise ValueError(\\"Current-system OOF must come from the plan\'s outer folds\\")\\n    current = current_frame[[f\\"blend::{label}\\" for label in LABELS]].to_numpy()\\n    current_inner_cache = {fold: current_inner(train, features, plan, fold, config) for fold in range(5)}\\n    probabilities, selection = {\\"current\\": current}, {}\\n    for name, directory in runs.items():\\n        adapted, inner, logs = load_runs(directory, train)\\n        probabilities[name] = adapted\\n        blended, chosen = np.empty_like(current), []\\n        for fold in range(5):\\n            choice = choose_weight(train, current_inner_cache[fold], inner, fold, logs[fold][\\"selected_setting\\"])\\n            mask = folds == fold\\n            blended[mask] = ensemble._blend(current[mask], adapted[mask], choice[\\"weight\\"])\\n            chosen.append({\\"outer_fold\\": fold, \\"selected_setting\\": logs[fold][\\"selected_setting\\"],\\n                           \\"current_inner_weight\\": [item[2] for item in current_inner_cache[fold].values()],\\n                           **choice})\\n        probabilities[f\\"{name}+current\\"] = blended\\n        selection[name] = chosen\\n    predictions = {name: values.argmax(axis=1) for name, values in probabilities.items()}\\n    systems = {name: ensemble._system_metrics(truth, values, folds) for name, values in predictions.items()}\\n    paired = {f\\"{name}_vs_current\\": ensemble.prompt_bootstrap(truth, predictions[\\"current\\"], values, prompts,\\n                                                               samples, config.seed)\\n              for name, values in predictions.items() if name != \\"current\\"}\\n    if {\\"lora\\", \\"control\\"} <= set(predictions):\\n        paired[\\"lora_vs_control\\"] = ensemble.prompt_bootstrap(truth, predictions[\\"control\\"], predictions[\\"lora\\"],\\n                                                              prompts, samples, config.seed)\\n    frame = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].rename(columns={\\"model\\": \\"true_model\\"}).assign(outer_fold=folds)\\n    for name, values in predictions.items():\\n        frame[f\\"prediction::{name}\\"] = np.asarray(LABELS)[values]\\n    frame.to_parquet(output / \\"predictions.parquet\\", index=False)\\n    result = {\\"systems\\": systems, \\"paired\\": paired, \\"selection\\": selection,\\n              \\"reference_current_balanced_accuracy\\": systems[\\"current\\"][\\"balanced_accuracy\\"]}\\n    (output / \\"comparison.json\\").write_text(json.dumps(result, indent=2, allow_nan=False) + \\"\\\\n\\")\\n    return result\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--plan\\", type=Path, required=True)\\n    parser.add_argument(\\"--features\\", type=Path, required=True)\\n    parser.add_argument(\\"--current-oof\\", type=Path, required=True, help=\\"blend-cv oof.parquet\\")\\n    parser.add_argument(\\"--run\\", nargs=2, action=\\"append\\", metavar=(\\"NAME\\", \\"DIRECTORY\\"), required=True)\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    args = parser.parse_args(argv)\\n    train = read_train(args.train)\\n    features, _ = ensemble.load_archive(args.features, train, neural.FrozenConfig())\\n    plan = pd.read_csv(args.plan, dtype={\\"id\\": str, \\"prompt_id\\": str}, keep_default_na=False)\\n    result = compare(train, features, plan, pd.read_parquet(args.current_oof), dict(args.run), args.output_dir)\\n    print(json.dumps({\\"balanced_accuracy\\": {name: values[\\"balanced_accuracy\\"]\\n                                            for name, values in result[\\"systems\\"].items()},\\n                      \\"weights\\": {name: [item[\\"weight\\"] for item in items] for name, items in result[\\"selection\\"].items()},\\n                      \\"paired\\": {name: [round(values[\\"balanced_accuracy_delta\\"], 4)] + [\\n                          round(bound, 4) for bound in values[\\"delta_95_interval\\"]]\\n                                 for name, values in result[\\"paired\\"].items()}}, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/lora_final.py": "\\"\\"\\"Final QLoRA model and offline scoring for the current system blended with adapted Qwen.\\n\\nThe validated procedure is applied once to all 2,699 rows: two inner prompt folds choose the\\nlearning rate, duration and the weight v of (1 - v) * current + v * adapted, exactly as inside each\\nouter training partition. Adapters and head are then trained on every row and saved.\\nScoring loads the backbone once and reads frozen (adapters disabled) and adapted features\\nfrom it; the current system\'s fitted CPU and frozen-Qwen classifiers come from `ntl.ensemble fit`.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport time\\nfrom dataclasses import asdict\\nfrom pathlib import Path\\n\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import balanced_accuracy_score\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom . import ensemble, lora, lora_compare, neural\\nfrom .constants import LABELS\\nfrom .data import fingerprint_frame, read_test, read_train, write_submission\\nfrom .folds import inner_prompt_splits\\nfrom .selection import first_best\\n\\n# The outer folds used seed offsets 0-4; the one full-data selection gets its own.\\nFINAL_SEED_OFFSET = 5\\n\\n\\ndef _json(path: Path, value: dict) -> None:\\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef final_plan(train: pd.DataFrame, config: lora.AdapterConfig) -> pd.DataFrame:\\n    \\"\\"\\"Inner prompt folds over every official row, for the final setting and weight choice.\\"\\"\\"\\n    splits, _ = inner_prompt_splits(ensemble._targets(train), train[\\"prompt_id\\"].to_numpy(),\\n                                    config.inner_folds, config.seed + FINAL_SEED_OFFSET)\\n    inner = np.full(len(train), -1)\\n    for inner_fold, (_, valid) in enumerate(splits):\\n        inner[valid] = inner_fold\\n    return train[[\\"id\\", \\"prompt_id\\"]].assign(inner=inner)\\n\\n\\ndef _aligned_plan(train: pd.DataFrame, plan: pd.DataFrame, config: lora.AdapterConfig) -> np.ndarray:\\n    plan = train[[\\"id\\"]].merge(plan, on=\\"id\\", how=\\"left\\", validate=\\"one_to_one\\")\\n    inner = plan[\\"inner\\"].to_numpy()\\n    if plan[\\"inner\\"].isna().any() or set(inner) != set(range(config.inner_folds)):\\n        raise ValueError(\\"Final plan must assign every row to an inner fold\\")\\n    if (plan[\\"prompt_id\\"] != train[\\"prompt_id\\"]).any():\\n        raise ValueError(\\"Final plan must carry every row\'s own prompt\\")\\n    if plan.groupby(\\"prompt_id\\")[\\"inner\\"].nunique().max() != 1:\\n        raise ValueError(\\"Final inner folds must keep prompts together\\")\\n    return inner.astype(int)\\n\\n\\ndef run_inner(train: pd.DataFrame, plan: pd.DataFrame, experiment: lora.Experiment, inner_fold: int,\\n              output_dir: str | Path) -> Path:\\n    config = experiment.config\\n    inner = _aligned_plan(train, plan, config)\\n    fit_rows, eval_rows = np.flatnonzero(inner != inner_fold), np.flatnonzero(inner == inner_fold)\\n    probabilities, checkpoints = {}, []\\n    for rate in sorted(config.learning_rates):\\n        result = experiment.run(fit_rows, eval_rows, rate, config.trained_durations)\\n        probabilities[(0.0, 0.0)] = result[\\"probabilities\\"][0.0]\\n        probabilities.update({(rate, duration): result[\\"probabilities\\"][duration]\\n                              for duration in config.trained_durations})\\n        checkpoints.append({\\"learning_rate\\": rate, \\"checkpoints\\": result[\\"checkpoints\\"]})\\n    frames = []\\n    for (rate, duration), values in probabilities.items():\\n        frame = pd.DataFrame(values, columns=lora_compare.PROBABILITIES)\\n        frame.insert(0, \\"duration\\", duration)\\n        frame.insert(0, \\"learning_rate\\", rate)\\n        frame.insert(0, \\"setting\\", lora.setting_name(rate, duration))\\n        frame.insert(0, \\"inner_fold\\", inner_fold)\\n        frame.insert(0, \\"id\\", train[\\"id\\"].to_numpy()[eval_rows])\\n        frames.append(frame)\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    pd.concat(frames, ignore_index=True).to_parquet(output / f\\"inner{inner_fold}.parquet\\", index=False)\\n    _json(output / f\\"inner{inner_fold}.json\\", {\\"inner_fold\\": inner_fold, \\"fit_rows\\": len(fit_rows),\\n                                               \\"eval_rows\\": len(eval_rows), \\"runs\\": checkpoints})\\n    return output / f\\"inner{inner_fold}.parquet\\"\\n\\n\\ndef choose(train: pd.DataFrame, features: np.ndarray, plan: pd.DataFrame, inner: pd.DataFrame,\\n           config: lora.AdapterConfig, blend: ensemble.BlendConfig | None = None) -> dict:\\n    \\"\\"\\"(Learning rate, duration), then blend weight, from the full-data inner folds only.\\"\\"\\"\\n    blend = blend or ensemble.BlendConfig()\\n    folds = _aligned_plan(train, plan, config)\\n    truth = ensemble._targets(train)\\n    position = pd.Series(np.arange(len(train)), index=train[\\"id\\"])\\n    settings = {lora.setting_name(*setting): setting for setting in config.settings()}\\n    scores = {name: [] for name in settings}\\n    for inner_fold in range(config.inner_folds):\\n        for name in settings:\\n            rows = inner[(inner[\\"inner_fold\\"] == inner_fold) & (inner[\\"setting\\"] == name)]\\n            if set(position.loc[rows[\\"id\\"]]) != set(np.flatnonzero(folds == inner_fold)):\\n                raise ValueError(\\"Final inner predictions must cover their inner fold exactly\\")\\n            predictions = rows[lora_compare.PROBABILITIES].to_numpy().argmax(axis=1)\\n            scores[name].append(float(balanced_accuracy_score(truth[position.loc[rows[\\"id\\"]]], predictions)))\\n    means = {name: float(np.mean(values)) for name, values in scores.items()}\\n    # Ties keep the shorter duration (zero is the frozen head), then the smaller learning rate.\\n    chosen = first_best(sorted(settings, key=lambda name: (settings[name][1], settings[name][0])), means.__getitem__)\\n    rate, duration = settings[chosen]\\n    current = {}\\n    for inner_fold in range(config.inner_folds):\\n        fit, valid = np.flatnonzero(folds != inner_fold), np.flatnonzero(folds == inner_fold)\\n        probabilities, weight = lora_compare.current_on_split(train, features, fit, valid, blend,\\n                                                              100 + inner_fold)\\n        current[inner_fold] = (valid, probabilities, weight)\\n    weight = lora_compare.choose_weight(train, current, inner.assign(outer_fold=-1), -1, chosen)\\n    return {\\"schema_version\\": 1, \\"kind\\": \\"current-plus-adapted-qwen-recipe\\", \\"labels\\": list(LABELS),\\n            \\"setting\\": chosen, \\"learning_rate\\": rate, \\"duration\\": duration,\\n            \\"adapted_weight\\": weight[\\"weight\\"], \\"current_weight\\": 0.25,\\n            \\"inner_mean_balanced_accuracy_by_setting\\": means,\\n            \\"inner_balanced_accuracy_by_weight\\": weight[\\"inner_balanced_accuracy\\"],\\n            \\"current_inner_weights\\": [item[2] for item in current.values()],\\n            \\"train_fingerprint\\": fingerprint_frame(train), \\"adapter\\": asdict(config),\\n            \\"model_id\\": neural.MODEL_ID, \\"model_revision\\": neural.MODEL_REVISION}\\n\\n\\ndef fit_all(train: pd.DataFrame, experiment: lora.Experiment, recipe: dict, output_dir: str | Path) -> dict:\\n    \\"\\"\\"Train adapters and head on every row for the chosen setting; save both.\\"\\"\\"\\n    if recipe[\\"train_fingerprint\\"] != fingerprint_frame(train):\\n        raise ValueError(\\"Training data differ from the rows used to choose the recipe\\")\\n    output = Path(output_dir)\\n    if (output / \\"head.npz\\").exists():\\n        raise ValueError(\\"Use a new output directory; a fitted adapter must not be overwritten\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    started = time.perf_counter()\\n    rows = np.arange(len(train))\\n    duration = recipe[\\"duration\\"]\\n    result = experiment.run(rows, rows[:0], recipe[\\"learning_rate\\"], [duration] if duration > 0 else [])\\n    head = result[\\"head\\"]\\n    np.savez(output / \\"head.npz\\", mean=head.mean.cpu().numpy(), scale=head.scale.cpu().numpy(),\\n             weight=head.linear.weight.detach().cpu().numpy(), bias=head.linear.bias.detach().cpu().numpy())\\n    if experiment.adapters:\\n        experiment.model.save_pretrained(output / \\"adapter\\")\\n    _json(output / \\"recipe.json\\", recipe)\\n    log = {\\"checkpoints\\": result[\\"checkpoints\\"], \\"rows\\": len(train), \\"elapsed_seconds\\": time.perf_counter() - started}\\n    _json(output / \\"fit.json\\", log)\\n    return log\\n\\n\\ndef load_adapter(base, adapter_dir: Path, device):\\n    \\"\\"\\"Wrap the backbone as training did and load the saved adapter weights onto its device.\\n\\n    PeftModel.from_pretrained re-dispatches CPU-mapped backbones to the default accelerator.\\n    \\"\\"\\"\\n    from peft import PeftConfig, get_peft_model, set_peft_model_state_dict\\n    from safetensors.torch import load_file\\n\\n    model = get_peft_model(base, PeftConfig.from_pretrained(str(adapter_dir)))\\n    weights = load_file(str(adapter_dir / \\"adapter_model.safetensors\\"), device=str(device))\\n    result = set_peft_model_state_dict(model, weights)\\n    loaded = {name for name, _ in model.named_parameters() if \\"lora_\\" in name}\\n    if result.unexpected_keys or len(loaded) != len(weights):\\n        raise ValueError(\\"Saved adapter weights do not match the adapter structure\\")\\n    return model\\n\\n\\ndef predict(final_dir: str | Path, blend_dir: str | Path, test: pd.DataFrame, model_path: str | Path,\\n            output: str | Path, device: str = \\"auto\\") -> pd.DataFrame:\\n    \\"\\"\\"Score every current test row: current system (frozen features) blended with adapted Qwen.\\"\\"\\"\\n    import torch\\n\\n    final_dir = Path(final_dir)\\n    recipe = json.loads((final_dir / \\"recipe.json\\").read_text())\\n    if recipe.get(\\"kind\\") != \\"current-plus-adapted-qwen-recipe\\" or recipe.get(\\"labels\\") != list(LABELS):\\n        raise ValueError(\\"Unsupported adapted-Qwen recipe\\")\\n    # joblib is trusted project output written by `ntl.ensemble fit` in this same run.\\n    current = joblib.load(Path(blend_dir) / \\"blend.joblib\\")\\n    weight, _, frozen = ensemble.load_recipe(current[\\"recipe\\"])\\n    if weight != recipe[\\"current_weight\\"]:\\n        raise ValueError(\\"Current-system weight differs from the adapted recipe\\")\\n    frozen = neural.FrozenConfig(**{**asdict(frozen), \\"device\\": device})\\n    started = time.perf_counter()\\n    tokenizer, base, device, _, _ = neural.load_backbone(model_path, frozen)\\n    model = load_adapter(base, final_dir / \\"adapter\\", device).eval()\\n    stored = np.load(final_dir / \\"head.npz\\")\\n    head = lora.head_module(torch, {key: stored[key] for key in (\\"mean\\", \\"scale\\", \\"weight\\", \\"bias\\")}, device).eval()\\n    texts = test[\\"text\\"].tolist()\\n\\n    def pooled(batch):\\n        encoded = tokenizer(batch, padding=True, truncation=True, max_length=frozen.max_length,\\n                            return_tensors=\\"pt\\").to(device)\\n        output = model(**encoded, output_hidden_states=True, use_cache=False)\\n        return neural.masked_layer_mean(output.hidden_states, encoded[\\"attention_mask\\"], frozen.pool_layers)\\n\\n    with torch.inference_mode():\\n        # Frozen features exactly as validated: adapters off, batch size 2, file order.\\n        with model.disable_adapter():\\n            frozen_features = np.concatenate([pooled(texts[start:start + frozen.batch_size]).cpu().numpy()\\n                                              for start in range(0, len(texts), frozen.batch_size)])\\n        # Adapted features as in the adapter validation: length-sorted batches of eval_batch.\\n        lengths = np.asarray(tokenizer(texts, truncation=True, max_length=frozen.max_length,\\n                                       return_length=True)[\\"length\\"])\\n        order, size = np.argsort(lengths, kind=\\"stable\\"), recipe[\\"adapter\\"][\\"eval_batch\\"]\\n        adapted = np.empty((len(texts), len(LABELS)))\\n        for start in range(0, len(texts), size):\\n            positions = order[start:start + size]\\n            logits = head(pooled([texts[index] for index in positions]))\\n            adapted[positions] = torch.softmax(logits.float(), dim=-1).cpu().numpy()\\n    with threadpool_limits(limits=2):\\n        cpu = ensemble.aligned_probabilities(current[\\"cpu\\"], test[\\"text\\"])\\n        qwen = neural._probabilities(current[\\"qwen_head\\"], frozen_features)\\n    current_scores = ensemble._blend(cpu, qwen, weight)\\n    final = ensemble._blend(current_scores, adapted, recipe[\\"adapted_weight\\"])\\n    if not np.isfinite(final).all() or not np.allclose(final.sum(axis=1), 1.0):\\n        raise ValueError(\\"Final scores must be finite probability vectors\\")\\n    submission = write_submission(test[\\"id\\"], np.asarray(LABELS)[final.argmax(axis=1)].tolist(), output)\\n    output = Path(output)\\n    _json(output.with_name(f\\"{output.stem}_profile.json\\"),\\n          {\\"rows\\": len(test), \\"adapted_weight\\": recipe[\\"adapted_weight\\"], \\"setting\\": recipe[\\"setting\\"],\\n           \\"elapsed_seconds\\": time.perf_counter() - started})\\n    return submission\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    commands = parser.add_subparsers(dest=\\"command\\", required=True)\\n    plan_command = commands.add_parser(\\"plan\\")\\n    plan_command.add_argument(\\"--train\\", type=Path, required=True)\\n    plan_command.add_argument(\\"--output\\", type=Path, required=True)\\n    for name in (\\"inner\\", \\"choose\\", \\"fit\\"):\\n        sub = commands.add_parser(name)\\n        sub.add_argument(\\"--train\\", type=Path, required=True)\\n        sub.add_argument(\\"--plan\\", type=Path, required=True)\\n        sub.add_argument(\\"--features\\", type=Path, required=True)\\n        sub.add_argument(\\"--output-dir\\", type=Path, required=True)\\n        if name != \\"choose\\":\\n            sub.add_argument(\\"--model-path\\", type=Path, required=True)\\n            sub.add_argument(\\"--device\\", default=\\"cuda:0\\")\\n    commands.choices[\\"inner\\"].add_argument(\\"--inner-fold\\", type=int, required=True)\\n    commands.choices[\\"fit\\"].add_argument(\\"--recipe\\", type=Path, required=True)\\n    predict_command = commands.add_parser(\\"predict\\")\\n    for argument in (\\"--final-dir\\", \\"--blend-dir\\", \\"--test\\", \\"--model-path\\", \\"--output\\"):\\n        predict_command.add_argument(argument, type=Path, required=True)\\n    predict_command.add_argument(\\"--device\\", default=\\"auto\\")\\n    args = parser.parse_args(argv)\\n    if args.command == \\"predict\\":\\n        submission = predict(args.final_dir, args.blend_dir, read_test(args.test), args.model_path, args.output,\\n                             args.device)\\n        print(json.dumps({\\"submission\\": str(args.output), \\"rows\\": len(submission)}))\\n        return 0\\n    train = read_train(args.train)\\n    config = lora.AdapterConfig()\\n    if args.command == \\"plan\\":\\n        final_plan(train, config).to_csv(args.output, index=False)\\n        return 0\\n    plan = pd.read_csv(args.plan, dtype={\\"id\\": str, \\"prompt_id\\": str}, keep_default_na=False)\\n    cached, _ = ensemble.load_archive(args.features, train, neural.FrozenConfig())\\n    if args.command == \\"choose\\":\\n        inner = pd.concat([pd.read_parquet(path) for path in sorted(args.output_dir.glob(\\"inner*.parquet\\"))])\\n        recipe = choose(train, cached, plan, inner, config)\\n        _json(args.output_dir / \\"recipe.json\\", recipe)\\n        print(json.dumps({key: recipe[key] for key in (\\"setting\\", \\"adapted_weight\\", \\"inner_mean_balanced_accuracy_by_setting\\")}))\\n        return 0\\n    experiment = lora.Experiment(train, cached, adapters=True, model_path=args.model_path, device=args.device)\\n    if args.command == \\"inner\\":\\n        run_inner(train, plan, experiment, args.inner_fold, args.output_dir)\\n        return 0\\n    log = fit_all(train, experiment, json.loads(args.recipe.read_text()), args.output_dir / \\"final\\")\\n    print(json.dumps({\\"rows\\": log[\\"rows\\"], \\"elapsed_seconds\\": log[\\"elapsed_seconds\\"]}))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/modeling.py": "\\"\\"\\"Fold-local CPU training, aligned OOF scores and portable inference artifacts.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport platform\\nfrom dataclasses import asdict, dataclass\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\n\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nimport sklearn\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.metrics import (\\n    accuracy_score,\\n    balanced_accuracy_score,\\n    confusion_matrix,\\n    precision_recall_fscore_support,\\n)\\nfrom sklearn.pipeline import Pipeline\\n\\nfrom . import __version__\\nfrom .constants import DEFAULT_SEED, LABELS\\nfrom .data import (\\n    data_quality_report,\\n    fingerprint_frame,\\n    validate_test,\\n    validate_train,\\n    write_submission,\\n)\\nfrom .features import build_features\\nfrom .folds import build_fold_manifest, load_fold_manifest, save_fold_manifest\\n\\n\\n@dataclass(frozen=True)\\nclass BaselineConfig:\\n    seed: int = DEFAULT_SEED\\n    char_max_features: int = 50000\\n    word_max_features: int = 30000\\n    C: float = 4.0\\n    max_iter: int = 1000\\n\\n\\ndef build_pipeline(config: BaselineConfig | None = None) -> Pipeline:\\n    config = config or BaselineConfig()\\n    if config.C <= 0 or config.max_iter < 1 or min(config.char_max_features, config.word_max_features) < 1:\\n        raise ValueError(\\"Feature limits, C and max_iter must be positive\\")\\n    return Pipeline([\\n        (\\"features\\", build_features(config.char_max_features, config.word_max_features)),\\n        (\\"classifier\\", LogisticRegression(\\n            C=config.C, max_iter=config.max_iter, class_weight=\\"balanced\\", solver=\\"lbfgs\\",\\n            random_state=config.seed,\\n        )),\\n    ])\\n\\n\\ndef aligned_probabilities(pipeline: Pipeline, texts) -> np.ndarray:\\n    \\"\\"\\"Match estimator classes to LABELS explicitly; sklearn sorts labels itself.\\"\\"\\"\\n    classes = list(pipeline.named_steps[\\"classifier\\"].classes_)\\n    if len(classes) != len(LABELS) or set(classes) != set(LABELS):\\n        raise ValueError(\\"The fitted classifier does not contain all 18 competition labels\\")\\n    probabilities = pipeline.predict_proba(texts)\\n    if probabilities.shape != (len(texts), len(classes)):\\n        raise ValueError(\\"Unexpected classifier probability shape\\")\\n    aligned = probabilities[:, [classes.index(label) for label in LABELS]]\\n    if not np.isfinite(aligned).all() or not np.allclose(aligned.sum(axis=1), 1.0):\\n        raise ValueError(\\"Invalid prediction probabilities\\")\\n    return aligned\\n\\n\\ndef _write_json(path: Path, value: dict) -> None:\\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef _metadata(train: pd.DataFrame, config: BaselineConfig) -> dict:\\n    return {\\n        \\"schema_version\\": 1,\\n        \\"package_version\\": __version__,\\n        \\"created_at_utc\\": datetime.now(timezone.utc).isoformat(),\\n        \\"model_type\\": \\"char_word_stylometry_logistic_regression\\",\\n        \\"labels\\": list(LABELS),\\n        \\"train_fingerprint\\": fingerprint_frame(train),\\n        \\"n_rows\\": len(train),\\n        \\"n_prompts\\": int(train[\\"prompt_id\\"].nunique()),\\n        \\"configuration\\": asdict(config),\\n        \\"data_quality\\": data_quality_report(train),\\n        \\"versions\\": {\\"python\\": platform.python_version(), \\"scikit_learn\\": sklearn.__version__,\\n                     \\"numpy\\": np.__version__, \\"pandas\\": pd.__version__, \\"joblib\\": joblib.__version__},\\n        \\"text_processing\\": \\"Case and punctuation preserved; TF-IDF vocabularies and numeric scaling fit on training rows only.\\",\\n        \\"probability_note\\": \\"Class-weighted classifier probabilities are scores, not calibrated deployment posteriors.\\",\\n    }\\n\\n\\ndef run_cross_validation(\\n    train: pd.DataFrame, output_dir: str | Path, n_splits: int = 5,\\n    config: BaselineConfig | None = None, fold_manifest: str | Path | None = None,\\n) -> dict:\\n    train = validate_train(train).reset_index(drop=True)\\n    config = config or BaselineConfig()\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    if fold_manifest is None:\\n        manifest, manifest_metadata = build_fold_manifest(train, n_splits=n_splits, seed=config.seed)\\n    else:\\n        manifest, manifest_metadata = load_fold_manifest(train, fold_manifest)\\n        if manifest_metadata[\\"n_splits\\"] != n_splits:\\n            raise ValueError(\\"Requested fold count differs from the supplied manifest\\")\\n    save_fold_manifest(manifest, manifest_metadata, output)\\n    folds = train[\\"id\\"].map(manifest.set_index(\\"id\\")[\\"fold\\"]).to_numpy()\\n    probabilities = np.full((len(train), len(LABELS)), np.nan)\\n    counts = np.zeros(len(train), dtype=np.int64)\\n    fold_metrics = []\\n    for fold in range(n_splits):\\n        fit_mask, valid_mask = folds != fold, folds == fold\\n        if set(train.loc[fit_mask, \\"prompt_id\\"]) & set(train.loc[valid_mask, \\"prompt_id\\"]):\\n            raise ValueError(\\"Prompt leakage detected before model fitting\\")\\n        # A new pipeline per fold ensures no vocabulary/scaler is fit on held-out rows.\\n        pipeline = build_pipeline(config)\\n        pipeline.fit(train.loc[fit_mask, \\"text\\"], train.loc[fit_mask, \\"model\\"])\\n        fold_probabilities = aligned_probabilities(pipeline, train.loc[valid_mask, \\"text\\"])\\n        probabilities[valid_mask] = fold_probabilities\\n        counts[valid_mask] += 1\\n        predictions = np.asarray(LABELS)[fold_probabilities.argmax(axis=1)]\\n        fold_metrics.append({\\n            \\"fold\\": fold,\\n            \\"n_train\\": int(fit_mask.sum()), \\"n_validation\\": int(valid_mask.sum()),\\n            \\"n_train_prompts\\": int(train.loc[fit_mask, \\"prompt_id\\"].nunique()),\\n            \\"n_validation_prompts\\": int(train.loc[valid_mask, \\"prompt_id\\"].nunique()),\\n            \\"balanced_accuracy\\": float(balanced_accuracy_score(train.loc[valid_mask, \\"model\\"], predictions)),\\n        })\\n    if not np.all(counts == 1) or not np.isfinite(probabilities).all():\\n        raise ValueError(\\"Every training row must receive exactly one finite OOF prediction\\")\\n    predictions = np.asarray(LABELS)[probabilities.argmax(axis=1)]\\n    oof = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].rename(columns={\\"model\\": \\"true_model\\"}).copy()\\n    oof[\\"fold\\"] = folds\\n    oof[\\"predicted_model\\"] = predictions\\n    for index, label in enumerate(LABELS):\\n        oof[f\\"probability::{label}\\"] = probabilities[:, index]\\n    oof.to_parquet(output / \\"oof.parquet\\", index=False)\\n    precision, recall, f1, support = precision_recall_fscore_support(\\n        train[\\"model\\"], predictions, labels=list(LABELS), zero_division=0,\\n    )\\n    pd.DataFrame({\\"model\\": LABELS, \\"precision\\": precision, \\"recall\\": recall,\\n                  \\"f1\\": f1, \\"support\\": support}).to_csv(output / \\"class_metrics.csv\\", index=False)\\n    pd.DataFrame(confusion_matrix(train[\\"model\\"], predictions, labels=list(LABELS)),\\n                 index=pd.Index(LABELS, name=\\"true_model\\"), columns=LABELS).to_csv(output / \\"confusion_matrix.csv\\")\\n    metrics = {\\n        \\"balanced_accuracy\\": float(balanced_accuracy_score(train[\\"model\\"], predictions)),\\n        \\"accuracy\\": float(accuracy_score(train[\\"model\\"], predictions)),\\n        \\"fold_balanced_accuracy_mean\\": float(np.mean([m[\\"balanced_accuracy\\"] for m in fold_metrics])),\\n        \\"fold_balanced_accuracy_std\\": float(np.std([m[\\"balanced_accuracy\\"] for m in fold_metrics])),\\n        \\"n_rows\\": len(train), \\"n_splits\\": n_splits, \\"folds\\": fold_metrics,\\n        \\"synthetic_data\\": bool(train[\\"id\\"].str.startswith(\\"synthetic-\\").all()),\\n    }\\n    _write_json(output / \\"metrics.json\\", metrics)\\n    metadata = _metadata(train, config)\\n    metadata[\\"validation\\"] = manifest_metadata\\n    _write_json(output / \\"metadata.json\\", metadata)\\n    return metrics\\n\\n\\ndef train_baseline(train: pd.DataFrame, output_dir: str | Path, config: BaselineConfig | None = None) -> Path:\\n    train = validate_train(train)\\n    config = config or BaselineConfig()\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    pipeline = build_pipeline(config)\\n    pipeline.fit(train[\\"text\\"], train[\\"model\\"])\\n    # Fail early if an accidental training subset omitted or reordered labels.\\n    aligned_probabilities(pipeline, train[\\"text\\"].iloc[:1])\\n    metadata = _metadata(train, config)\\n    metadata[\\"estimator_classes\\"] = list(pipeline.named_steps[\\"classifier\\"].classes_)\\n    model_path = output / \\"model.joblib\\"\\n    joblib.dump({\\"pipeline\\": pipeline, \\"metadata\\": metadata}, model_path)\\n    _write_json(output / \\"metadata.json\\", metadata)\\n    return model_path\\n\\n\\ndef predict_baseline(model_path: str | Path, test: pd.DataFrame, output_path: str | Path) -> pd.DataFrame:\\n    test = validate_test(test)\\n    # joblib loads executable Python objects: only load artifacts you created/trust.\\n    artifact = joblib.load(model_path)\\n    metadata = artifact.get(\\"metadata\\", {})\\n    if metadata.get(\\"schema_version\\") != 1 or metadata.get(\\"labels\\") != list(LABELS):\\n        raise ValueError(\\"Unsupported artifact schema or label order\\")\\n    probabilities = aligned_probabilities(artifact[\\"pipeline\\"], test[\\"text\\"])\\n    predictions = np.asarray(LABELS)[probabilities.argmax(axis=1)].tolist()\\n    return write_submission(test[\\"id\\"], predictions, output_path)\\n", "ntl/neural.py": "\\"\\"\\"Offline frozen Qwen features and a prompt-grouped linear classification head.\\n\\nThis is a diagnostic baseline. It does not train LoRA adapters or claim a T4 benchmark.\\nRun ``python -m ntl.neural --help``. Neural dependencies are imported only when needed.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport gc\\nimport importlib.metadata\\nimport json\\nimport time\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\n\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score\\nfrom sklearn.pipeline import make_pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom .constants import LABELS\\nfrom .data import fingerprint_frame, read_test, read_train, write_submission\\nfrom .folds import build_fold_manifest, load_fold_manifest, save_fold_manifest\\nfrom .gemini import _initialize_cuda_memory_stats\\n\\nMODEL_ID = \\"Qwen/Qwen3-8B-Base\\"\\nMODEL_REVISION = \\"49e3418fbbbca6ecbdf9608b4d22e5a407081db4\\"\\n\\n\\n@dataclass(frozen=True)\\nclass FrozenConfig:\\n    retained_layers: int = 24\\n    pool_layers: tuple[int, ...] = (7, 22)\\n    quantization: str = \\"nf4\\"\\n    max_length: int = 512\\n    batch_size: int = 2\\n    C: float = 0.005\\n    seed: int = 2026\\n    device: str = \\"auto\\"\\n\\n    def validate(self) -> None:\\n        if not 1 <= self.retained_layers <= 36:\\n            raise ValueError(\\"retained_layers must be between 1 and 36\\")\\n        if not self.pool_layers or len(set(self.pool_layers)) != len(self.pool_layers):\\n            raise ValueError(\\"pool_layers must be nonempty and unique\\")\\n        if any(layer < 1 or layer > self.retained_layers for layer in self.pool_layers):\\n            raise ValueError(\\"Each pooled layer must be a retained decoder block (1-based)\\")\\n        if self.quantization not in {\\"none\\", \\"nf4\\"}:\\n            raise ValueError(\\"quantization must be none or nf4\\")\\n        if self.max_length < 16 or self.batch_size < 1 or self.C <= 0:\\n            raise ValueError(\\"Invalid token limit, batch size, or regularization\\")\\n\\n\\ndef validate_snapshot(directory: str | Path, *, weights: bool = True) -> dict:\\n    \\"\\"\\"Reject mutable/wrong model inputs before loading any serialized weights.\\n\\n    With weights=False, a tokenizer-only snapshot of the pinned revision is accepted.\\n    \\"\\"\\"\\n    directory = Path(directory)\\n    manifest_path = directory / \\"snapshot_manifest.json\\"\\n    if not manifest_path.is_file():\\n        raise ValueError(\\"Missing snapshot_manifest.json; run scripts/prepare_model.py first\\")\\n    manifest = json.loads(manifest_path.read_text())\\n    if (\\n        manifest.get(\\"schema_version\\") != 1\\n        or manifest.get(\\"repo_id\\") != MODEL_ID\\n        or manifest.get(\\"revision\\") != MODEL_REVISION\\n        or (weights and manifest.get(\\"includes_weights\\") is not True)\\n    ):\\n        raise ValueError(f\\"Snapshot must contain the pinned Qwen3-8B-Base {\'weights\' if weights else \'tokenizer\'}\\")\\n    entries = manifest.get(\\"files\\", [])\\n    if not entries:\\n        raise ValueError(\\"Snapshot has no recorded files\\")\\n    for item in entries:\\n        relative = Path(item[\\"path\\"])\\n        if relative.is_absolute() or \\"..\\" in relative.parts:\\n            raise ValueError(\\"Invalid path in snapshot manifest\\")\\n        target = directory / relative\\n        if not target.is_file() or target.stat().st_size != item[\\"size\\"]:\\n            raise ValueError(f\\"Snapshot file missing or size changed: {relative}\\")\\n    if weights and not any(str(item[\\"path\\"]).endswith(\\".safetensors\\") for item in entries):\\n        raise ValueError(\\"Expected safetensors model weights\\")\\n    architecture = json.loads((directory / \\"config.json\\").read_text())\\n    if (\\n        architecture.get(\\"model_type\\") != \\"qwen3\\"\\n        or architecture.get(\\"num_hidden_layers\\") != 36\\n        or architecture.get(\\"hidden_size\\") != 4096\\n    ):\\n        raise ValueError(\\"Snapshot architecture is not the selected full Qwen3-8B-Base\\")\\n    return manifest\\n\\n\\ndef masked_layer_mean(hidden_states, attention_mask, layers: tuple[int, ...]):\\n    \\"\\"\\"Concatenate float32 means; padding must contribute neither mass nor count.\\"\\"\\"\\n    import torch\\n\\n    mask = attention_mask.unsqueeze(-1).to(dtype=torch.float32)\\n    counts = mask.sum(dim=1)\\n    if torch.any(counts == 0):\\n        raise ValueError(\\"Cannot pool an empty token sequence\\")\\n    pooled = [(hidden_states[layer].float() * mask).sum(dim=1) / counts for layer in layers]\\n    return torch.cat(pooled, dim=-1)\\n\\n\\ndef _versions() -> dict:\\n    versions = {}\\n    for package in (\\"torch\\", \\"transformers\\", \\"bitsandbytes\\", \\"numpy\\", \\"scikit-learn\\"):\\n        try:\\n            versions[package] = importlib.metadata.version(package)\\n        except importlib.metadata.PackageNotFoundError:\\n            versions[package] = None\\n    return versions\\n\\n\\ndef load_tokenizer(model_path: str | Path):\\n    \\"\\"\\"The snapshot\'s tokenizer as the frozen recipe configures it; reads no model weights.\\"\\"\\"\\n    from transformers import AutoTokenizer\\n\\n    tokenizer = AutoTokenizer.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False, padding_side=\\"right\\"\\n    )\\n    if tokenizer.pad_token_id is None:\\n        if tokenizer.eos_token_id is None:\\n            raise ValueError(\\"Tokenizer has neither pad nor EOS token\\")\\n        tokenizer.pad_token = tokenizer.eos_token\\n    return tokenizer\\n\\n\\ndef load_backbone(model_path: str | Path, config: FrozenConfig) -> tuple:\\n    \\"\\"\\"Load the pinned tokenizer and truncated backbone exactly as the frozen recipe does.\\n\\n    Returns (tokenizer, model, device, dtype, manifest); shared by extraction and adaptation.\\n    \\"\\"\\"\\n    config.validate()\\n    manifest = validate_snapshot(model_path)\\n    try:\\n        import torch\\n        from transformers import AutoConfig, AutoModel, BitsAndBytesConfig\\n    except ImportError as exc:\\n        raise RuntimeError(\\"Install the neural extra; NF4 also needs the Linux gpu extra\\") from exc\\n    device = config.device\\n    if device == \\"auto\\":\\n        device = \\"cuda:0\\" if torch.cuda.is_available() else \\"cpu\\"\\n    if device.startswith(\\"cuda\\") and not torch.cuda.is_available():\\n        raise ValueError(\\"CUDA requested but unavailable\\")\\n    if config.quantization == \\"nf4\\" and not device.startswith(\\"cuda\\"):\\n        raise ValueError(\\"NF4 requires a CUDA GPU; use --quantization none for CPU diagnostics\\")\\n    torch.manual_seed(config.seed)\\n    if device.startswith(\\"cuda\\"):\\n        torch.cuda.manual_seed_all(config.seed)\\n        _initialize_cuda_memory_stats(torch, device)\\n    tokenizer = load_tokenizer(model_path)\\n    architecture = AutoConfig.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False\\n    )\\n    architecture.num_hidden_layers = config.retained_layers\\n    architecture.use_cache = False\\n    dtype = torch.float16 if device.startswith(\\"cuda\\") else torch.float32\\n    load_args = {\\n        \\"config\\": architecture,\\n        \\"local_files_only\\": True,\\n        \\"trust_remote_code\\": False,\\n        \\"dtype\\": dtype,\\n        \\"attn_implementation\\": \\"sdpa\\",\\n        \\"use_safetensors\\": True,\\n        \\"low_cpu_mem_usage\\": True,\\n        \\"device_map\\": {\\"\\": device},\\n    }\\n    if config.quantization == \\"nf4\\":\\n        load_args[\\"quantization_config\\"] = BitsAndBytesConfig(\\n            load_in_4bit=True,\\n            bnb_4bit_quant_type=\\"nf4\\",\\n            bnb_4bit_use_double_quant=True,\\n            bnb_4bit_compute_dtype=torch.float16,\\n        )\\n    # AutoModel loads Qwen3Model, avoiding the unused language-model vocabulary head.\\n    # num_hidden_layers controls instantiation, so discarded upper blocks never reside on GPU.\\n    model = AutoModel.from_pretrained(str(model_path), **load_args).eval()\\n    return tokenizer, model, device, dtype, manifest\\n\\n\\ndef extract_features(\\n    texts: list[str], model_path: str | Path, config: FrozenConfig\\n) -> tuple[np.ndarray, dict]:\\n    \\"\\"\\"Read only a local pinned snapshot, with no network fallback or remote code.\\"\\"\\"\\n    if not texts or any(not isinstance(text, str) or not text.strip() for text in texts):\\n        raise ValueError(\\"Expected nonempty passage strings\\")\\n    import torch\\n\\n    started = time.perf_counter()\\n    tokenizer, model, device, dtype, manifest = load_backbone(model_path, config)\\n    chunks = []\\n    token_lengths: list[int] = []\\n    try:\\n        with torch.inference_mode():\\n            for start in range(0, len(texts), config.batch_size):\\n                batch_text = texts[start : start + config.batch_size]\\n                lengths = tokenizer(batch_text, truncation=False, return_length=True)[\\"length\\"]\\n                token_lengths.extend(lengths)\\n                encoded = tokenizer(\\n                    batch_text,\\n                    padding=True,\\n                    truncation=True,\\n                    max_length=config.max_length,\\n                    return_tensors=\\"pt\\",\\n                ).to(device)\\n                output = model(**encoded, output_hidden_states=True, use_cache=False)\\n                pooled = masked_layer_mean(\\n                    output.hidden_states, encoded[\\"attention_mask\\"], config.pool_layers\\n                )\\n                chunks.append(pooled.cpu().numpy())\\n                del output, pooled, encoded\\n        if device.startswith(\\"cuda\\"):\\n            torch.cuda.synchronize(device)\\n        profile = {\\n            \\"elapsed_seconds\\": time.perf_counter() - started,\\n            \\"device\\": device,\\n            \\"device_name\\": torch.cuda.get_device_name(device) if device.startswith(\\"cuda\\") else \\"CPU\\",\\n            \\"compute_dtype\\": str(dtype),\\n            \\"peak_cuda_allocated_bytes\\": (\\n                torch.cuda.max_memory_allocated(device) if device.startswith(\\"cuda\\") else None\\n            ),\\n            \\"rows\\": len(texts),\\n            \\"truncated_rows\\": sum(length > config.max_length for length in token_lengths),\\n            \\"token_length_quantiles\\": {\\n                str(q): float(np.quantile(token_lengths, q)) for q in (0.5, 0.9, 0.99, 1.0)\\n            },\\n            \\"model_id\\": manifest[\\"repo_id\\"],\\n            \\"model_revision\\": manifest[\\"revision\\"],\\n            \\"versions\\": _versions(),\\n        }\\n        features = np.concatenate(chunks).astype(np.float32, copy=False)\\n        if not np.isfinite(features).all():\\n            raise ValueError(\\"Backbone produced non-finite features; inspect precision and inputs\\")\\n        return features, profile\\n    finally:\\n        del model\\n        gc.collect()\\n        if device.startswith(\\"cuda\\"):\\n            torch.cuda.empty_cache()\\n\\n\\ndef _head(config: FrozenConfig):\\n    return make_pipeline(\\n        StandardScaler(),\\n        LogisticRegression(C=config.C, max_iter=2000, random_state=config.seed),\\n    )\\n\\n\\ndef _json(path: Path, value: dict) -> None:\\n    path.write_text(json.dumps(value, indent=2) + \\"\\\\n\\")\\n\\n\\ndef _labels(train: pd.DataFrame) -> np.ndarray:\\n    return np.array([LABELS.index(label) for label in train[\\"model\\"]], dtype=np.int64)\\n\\n\\ndef _probabilities(head, features: np.ndarray) -> np.ndarray:\\n    if not np.array_equal(head.classes_, np.arange(len(LABELS))):\\n        raise ValueError(\\"Classifier does not contain the complete canonical label order\\")\\n    probabilities = head.predict_proba(features)\\n    if not np.isfinite(probabilities).all():\\n        raise ValueError(\\"Classifier produced non-finite probabilities\\")\\n    return probabilities\\n\\n\\ndef run_cv(\\n    train: pd.DataFrame,\\n    model_path: str | Path,\\n    output_dir: str | Path,\\n    config: FrozenConfig,\\n    n_splits: int = 5,\\n    fold_manifest: str | Path | None = None,\\n) -> dict:\\n    manifest, fold_metadata = (\\n        load_fold_manifest(train, fold_manifest)\\n        if fold_manifest is not None\\n        else build_fold_manifest(train, n_splits=n_splits, seed=config.seed)\\n    )\\n    features, profile = extract_features(train[\\"text\\"].tolist(), model_path, config)\\n    targets = _labels(train)\\n    assignments = train[\\"id\\"].map(manifest.set_index(\\"id\\")[\\"fold\\"]).to_numpy()\\n    oof = np.full((len(train), len(LABELS)), np.nan, dtype=np.float64)\\n    scores = []\\n    for fold in range(fold_metadata[\\"n_splits\\"]):\\n        fit, valid = assignments != fold, assignments == fold\\n        head = _head(config)\\n        with threadpool_limits(limits=2):\\n            head.fit(features[fit], targets[fit])\\n            oof[valid] = _probabilities(head, features[valid])\\n        scores.append(float(balanced_accuracy_score(targets[valid], oof[valid].argmax(axis=1))))\\n    if not np.isfinite(oof).all():\\n        raise ValueError(\\"OOF predictions are incomplete or non-finite\\")\\n    predictions = oof.argmax(axis=1)\\n    metrics = {\\n        \\"balanced_accuracy\\": float(balanced_accuracy_score(targets, predictions)),\\n        \\"fold_scores\\": scores,\\n        \\"labels\\": list(LABELS),\\n        \\"config\\": asdict(config),\\n        \\"train_fingerprint\\": fingerprint_frame(train),\\n        \\"model_id\\": MODEL_ID,\\n        \\"model_revision\\": MODEL_REVISION,\\n        \\"validation\\": \\"prompt-grouped; frozen unsupervised feature extraction, fold-fitted heads\\",\\n    }\\n    directory = Path(output_dir)\\n    directory.mkdir(parents=True, exist_ok=True)\\n    save_fold_manifest(manifest, fold_metadata, directory)\\n    frame = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].copy()\\n    frame[\\"fold\\"] = assignments\\n    frame[\\"prediction\\"] = [LABELS[index] for index in predictions]\\n    for index, label in enumerate(LABELS):\\n        frame[f\\"probability__{label}\\"] = oof[:, index]\\n    frame.to_parquet(directory / \\"oof.parquet\\", index=False)\\n    pd.DataFrame({\\n        \\"model\\": LABELS,\\n        \\"recall\\": recall_score(targets, predictions, labels=np.arange(len(LABELS)), average=None),\\n    }).to_csv(directory / \\"class_metrics.csv\\", index=False)\\n    pd.DataFrame(\\n        confusion_matrix(targets, predictions, labels=np.arange(len(LABELS))),\\n        index=pd.Index(LABELS, name=\\"true_model\\"), columns=LABELS,\\n    ).to_csv(directory / \\"confusion_matrix.csv\\")\\n    _json(directory / \\"metrics.json\\", metrics)\\n    _json(directory / \\"feature_profile.json\\", profile)\\n    return metrics\\n\\n\\ndef train_head(\\n    train: pd.DataFrame, model_path: str | Path, output_dir: str | Path, config: FrozenConfig\\n) -> dict:\\n    features, profile = extract_features(train[\\"text\\"].tolist(), model_path, config)\\n    head = _head(config)\\n    with threadpool_limits(limits=2):\\n        head.fit(features, _labels(train))\\n    metadata = {\\n        \\"schema_version\\": 1,\\n        \\"kind\\": \\"frozen-qwen-linear-head\\",\\n        \\"labels\\": list(LABELS),\\n        \\"model_id\\": MODEL_ID,\\n        \\"model_revision\\": MODEL_REVISION,\\n        \\"snapshot_path\\": str(Path(model_path).resolve()),\\n        \\"config\\": asdict(config),\\n        \\"train_fingerprint\\": fingerprint_frame(train),\\n        \\"n_train\\": len(train),\\n        \\"versions\\": _versions(),\\n    }\\n    directory = Path(output_dir)\\n    directory.mkdir(parents=True, exist_ok=True)\\n    joblib.dump({\\"head\\": head, \\"metadata\\": metadata}, directory / \\"head.joblib\\")\\n    _json(directory / \\"metadata.json\\", metadata)\\n    _json(directory / \\"feature_profile.json\\", profile)\\n    return metadata\\n\\n\\ndef predict_head(\\n    model_dir: str | Path,\\n    test: pd.DataFrame,\\n    output: str | Path,\\n    model_path: str | Path | None = None,\\n) -> pd.DataFrame:\\n    # joblib is trusted project output; it must never load an untrusted third-party pickle.\\n    artifact = joblib.load(Path(model_dir) / \\"head.joblib\\")\\n    metadata = artifact[\\"metadata\\"]\\n    if (\\n        metadata.get(\\"kind\\") != \\"frozen-qwen-linear-head\\"\\n        or metadata.get(\\"labels\\") != list(LABELS)\\n        or metadata.get(\\"model_id\\") != MODEL_ID\\n        or metadata.get(\\"model_revision\\") != MODEL_REVISION\\n    ):\\n        raise ValueError(\\"Incompatible neural artifact metadata or label order\\")\\n    config_values = dict(metadata[\\"config\\"])\\n    config_values[\\"pool_layers\\"] = tuple(config_values[\\"pool_layers\\"])\\n    config = FrozenConfig(**config_values)\\n    features, profile = extract_features(\\n        test[\\"text\\"].tolist(), model_path or metadata[\\"snapshot_path\\"], config\\n    )\\n    with threadpool_limits(limits=2):\\n        probabilities = _probabilities(artifact[\\"head\\"], features)\\n    result = write_submission(test[\\"id\\"], [LABELS[i] for i in probabilities.argmax(axis=1)], output)\\n    output_path = Path(output)\\n    _json(output_path.with_name(f\\"{output_path.stem}_profile.json\\"), profile)\\n    return result\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    commands = parser.add_subparsers(dest=\\"command\\", required=True)\\n    for name in (\\"cv\\", \\"train\\"):\\n        sub = commands.add_parser(name)\\n        sub.add_argument(\\"--train\\", required=True, type=Path)\\n        sub.add_argument(\\"--model-path\\", required=True, type=Path)\\n        sub.add_argument(\\"--output-dir\\", required=True, type=Path)\\n        sub.add_argument(\\"--quantization\\", choices=(\\"none\\", \\"nf4\\"), default=\\"nf4\\")\\n        sub.add_argument(\\"--batch-size\\", type=int, default=2)\\n        sub.add_argument(\\"--max-length\\", type=int, default=512)\\n        sub.add_argument(\\"--retained-layers\\", type=int, default=24)\\n        sub.add_argument(\\"--pool-layers\\", type=int, nargs=\\"+\\", default=[7, 22])\\n        sub.add_argument(\\"--C\\", type=float, default=0.005)\\n        sub.add_argument(\\"--seed\\", type=int, default=2026)\\n        sub.add_argument(\\"--device\\", default=\\"auto\\")\\n        if name == \\"cv\\":\\n            sub.add_argument(\\"--folds\\", type=int, default=5)\\n            sub.add_argument(\\"--fold-manifest\\", type=Path)\\n    predict = commands.add_parser(\\"predict\\")\\n    predict.add_argument(\\"--model-dir\\", required=True, type=Path)\\n    predict.add_argument(\\"--model-path\\", type=Path)\\n    predict.add_argument(\\"--test\\", required=True, type=Path)\\n    predict.add_argument(\\"--output\\", required=True, type=Path)\\n    args = parser.parse_args(argv)\\n    if args.command == \\"predict\\":\\n        result = predict_head(args.model_dir, read_test(args.test), args.output, args.model_path)\\n        print(f\\"Wrote {len(result)} predictions to {args.output}\\")\\n        return 0\\n    config = FrozenConfig(\\n        retained_layers=args.retained_layers, pool_layers=tuple(args.pool_layers),\\n        quantization=args.quantization, max_length=args.max_length, batch_size=args.batch_size,\\n        C=args.C, seed=args.seed, device=args.device,\\n    )\\n    config.validate()\\n    train = read_train(args.train)\\n    if args.command == \\"cv\\":\\n        result = run_cv(\\n            train, args.model_path, args.output_dir, config, args.folds, args.fold_manifest\\n        )\\n        print(f\\"Prompt-grouped balanced accuracy: {result[\'balanced_accuracy\']:.5f}\\")\\n    else:\\n        train_head(train, args.model_path, args.output_dir, config)\\n        print(f\\"Saved frozen feature classifier to {args.output_dir}\\")\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/selection.py": "\\"\\"\\"Nested selection: the best option by mean inner score, under a declared tie order.\\"\\"\\"\\n\\n# Inner balanced accuracies are rationals with small denominators, so distinct option\\n# scores differ by far more than this, while float means of equal scores can differ in\\n# the last bit: an exact tie at 902/2160 once came out one ulp apart.\\nTIE_TOLERANCE = 1e-12\\n\\n\\ndef first_best(options, score):\\n    \\"\\"\\"The first option, in tie order, whose score is the best up to float rounding.\\"\\"\\"\\n    options = list(options)\\n    best = max(score(option) for option in options)\\n    return next(option for option in options if score(option) >= best - TIE_TOLERANCE)\\n", "ntl/smoke.py": "\\"\\"\\"Clearly synthetic fixtures for exercising plumbing, never competition scoring.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\n\\nimport pandas as pd\\n\\nfrom .constants import LABELS\\nfrom .modeling import BaselineConfig, predict_baseline, run_cross_validation, train_baseline\\n\\n\\ndef synthetic_frames(n_prompts: int = 6) -> tuple[pd.DataFrame, pd.DataFrame]:\\n    \\"\\"\\"Artificial class markers make these data unsuitable for modeling claims.\\"\\"\\"\\n    rows = []\\n    for prompt in range(n_prompts):\\n        for model_index, label in enumerate(LABELS):\\n            rows.append({\\n                \\"id\\": f\\"synthetic-{prompt:03d}-{model_index:02d}\\",\\n                \\"prompt_id\\": f\\"synthetic-prompt-{prompt:03d}\\",\\n                \\"text\\": (f\\"SYNTHETIC FIXTURE ONLY. ClassMarker{model_index:02d} \\"\\n                         f\\"ClassMarker{model_index:02d}. Prompt theme {prompt}; \\"\\n                         \\"Case, punctuation\\u2014\\u2018quotes\\u2019\\u2014and line breaks\\\\nare preserved. \\"\\n                         \\"This manufactured sample is not an authentic model response.\\"),\\n                \\"model\\": label,\\n            })\\n    train = pd.DataFrame(rows)\\n    test = pd.DataFrame({\\n        \\"id\\": [\\"synthetic-test-000\\", \\"synthetic-test-001\\", \\"synthetic-test-002\\"],\\n        \\"text\\": [rows[index][\\"text\\"] for index in (0, 7, 17)],\\n    })\\n    return train, test\\n\\n\\ndef run_smoke(output_dir: str | Path) -> dict:\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    train, test = synthetic_frames()\\n    train.to_csv(output / \\"synthetic_train.csv\\", index=False)\\n    test.to_csv(output / \\"synthetic_test.csv\\", index=False)\\n    config = BaselineConfig(char_max_features=3000, word_max_features=1000, max_iter=300)\\n    cv = run_cross_validation(train, output / \\"cv\\", n_splits=5, config=config)\\n    model_path = train_baseline(train, output / \\"model\\", config=config)\\n    submission = predict_baseline(model_path, test, output / \\"submission.parquet\\")\\n    summary = {\\"synthetic_data\\": True, \\"purpose\\": \\"Pipeline plumbing only; scores have no competition meaning.\\",\\n               \\"training_rows\\": len(train), \\"test_rows\\": len(test), \\"submission_rows\\": len(submission),\\n               \\"cv_completed\\": True, \\"model_saved\\": str(model_path), \\"cv\\": cv}\\n    (output / \\"smoke_summary.json\\").write_text(json.dumps(summary, indent=2) + \\"\\\\n\\")\\n    return summary\\n", "ntl/specialist_mix.py": "\\"\\"\\"Does a further frozen encoder add to the Claude-GLM specialist?\\n\\nThe incumbent is the validated system: the current CPU + frozen-Qwen blend whose\\nClaude-GLM probability mass the Qwen + StyleDistance specialist redistributes at the\\nweight s its inner folds choose (`claude_glm.evaluate`). A candidate head appends one\\nmore frozen encoder block, block-variance balanced, with the same C. The two seven-class\\ndistributions are mixed, q_mix = (1 - a) q_existing + a q_augmented, and fed through the\\nsame integration at the same s, so a = 0 is the incumbent exactly.\\n\\nInner prompt folds of each outer training partition choose s first, as the incumbent\\ndoes, then the encoder and a from the unchanged reference and every encoder at\\na in {0.25, 0.5, 1}. Ties keep the reference, then the smaller a, then the encoder\\nlisted first.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import balanced_accuracy_score\\n\\nfrom . import claude_glm, ensemble, neural\\nfrom .constants import LABELS\\nfrom .data import fingerprint_frame, read_train\\nfrom .frozen_gemini import load_feature_archive\\nfrom .lora import AdapterConfig, validate_plan\\nfrom .selection import first_best\\nfrom .style_encoders import ENCODERS\\n\\nINCUMBENT = \\"qwen+styledistance\\"\\nMIX_WEIGHTS = (0.25, 0.5, 1.0)\\nREFERENCE = \\"reference\\"\\n# Confusions the experiment targets: (true, predicted).\\nCONFUSIONS = ((\\"claude-fable-5\\", \\"glm-5.3\\"), (\\"claude-fable-5\\", \\"glm-5.3-flash\\"),\\n              (\\"glm-5.3\\", \\"claude-fable-5\\"), (\\"glm-5.3-flash\\", \\"claude-fable-5\\"),\\n              (\\"glm-5.3\\", \\"glm-5.3-flash\\"), (\\"glm-5.3-flash\\", \\"glm-5.3\\"))\\n\\n\\ndef option_name(encoder: str | None, weight: float) -> str:\\n    return REFERENCE if encoder is None or weight == 0 else f\\"{encoder}@{weight:g}\\"\\n\\n\\ndef options(encoders: tuple[str, ...]) -> list[tuple[str | None, float]]:\\n    \\"\\"\\"In tie order: the reference, then by increasing a, then in the order the encoders are listed.\\"\\"\\"\\n    return [(None, 0.0)] + [(encoder, weight) for weight in MIX_WEIGHTS for encoder in encoders]\\n\\n\\ndef mixed(existing: np.ndarray, augmented: dict[str, np.ndarray], encoder: str | None, weight: float) -> np.ndarray:\\n    if encoder is None or weight == 0:\\n        return existing\\n    return (1 - weight) * existing + weight * augmented[encoder]\\n\\n\\ndef _ba(truth: np.ndarray, probabilities: np.ndarray) -> float:\\n    return float(balanced_accuracy_score(truth, probabilities.argmax(axis=1)))\\n\\n\\ndef _confusions(truth: np.ndarray, predictions: np.ndarray) -> dict[str, dict]:\\n    result = {}\\n    for true, predicted in CONFUSIONS:\\n        rows = truth == LABELS.index(true)\\n        count = int((predictions[rows] == LABELS.index(predicted)).sum())\\n        result[f\\"{true}->{predicted}\\"] = {\\"count\\": count, \\"share\\": count / int(rows.sum())}\\n    return result\\n\\n\\ndef evaluate(train: pd.DataFrame, blocks: dict[str, np.ndarray], encoders: tuple[str, ...], plan: pd.DataFrame,\\n             current_oof: pd.DataFrame, current_cache: pd.DataFrame, output_dir: str | Path, *,\\n             frozen: neural.FrozenConfig | None = None, samples: int = 10000, seed: int = 2026,\\n             provenance: dict | None = None) -> dict:\\n    frozen = frozen or neural.FrozenConfig()\\n    if (not encoders or len(set(encoders)) != len(encoders)\\n            or any(name not in blocks or name in (\\"qwen\\", \\"styledistance\\") or \\"+\\" in name for name in encoders)):\\n        raise ValueError(\\"Name each added encoder\'s features once; Qwen and StyleDistance are the incumbent\'s\\")\\n    plan = validate_plan(train, plan, AdapterConfig())\\n    output = Path(output_dir)\\n    if (output / \\"comparison.json\\").exists():\\n        raise ValueError(\\"Use a new output directory; an existing comparison must not be overwritten\\")\\n    if set(current_cache[\\"groups_sha256\\"]) != {claude_glm.groups_sha256(train)}:\\n        raise ValueError(\\"Cached current-system inner predictions were built under a different row grouping\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    truth, prompts = ensemble._targets(train), train[\\"prompt_id\\"].to_numpy()\\n    folds = plan[\\"outer_fold\\"].to_numpy()\\n    current = claude_glm._aligned_current(current_oof, train, folds)\\n    grid = options(encoders)\\n    incumbent, candidate = np.empty_like(current), np.empty_like(current)\\n    by_option = {option_name(*option): np.empty_like(current) for option in grid}\\n    selection = []\\n    for fold in range(5):\\n        column = plan[f\\"inner_{fold}\\"].to_numpy()\\n        inner = []\\n        for inner_fold, (valid, inner_current) in claude_glm._inner_current(current_cache, train, plan,\\n                                                                            fold).items():\\n            fit = np.flatnonzero((folds != fold) & (column != inner_fold))\\n            existing = claude_glm.specialist_probabilities(blocks, INCUMBENT, truth, fit, valid, frozen)\\n            augmented = {encoder: claude_glm.specialist_probabilities(blocks, f\\"{INCUMBENT}+{encoder}\\", truth,\\n                                                                      fit, valid, frozen)\\n                         for encoder in encoders}\\n            inner.append((valid, inner_current, existing, augmented))\\n        # The incumbent\'s own choice of s, exactly as claude_glm.evaluate makes it.\\n        s_scores = {s: float(np.mean([_ba(truth[valid], claude_glm.redistribute(base, existing, s))\\n                                      for valid, base, existing, _ in inner]))\\n                    for s in claude_glm.WEIGHTS}\\n        s = first_best(claude_glm.WEIGHTS, s_scores.__getitem__)\\n        scores = {option_name(*option): [_ba(truth[valid], claude_glm.redistribute(\\n            base, mixed(existing, augmented, *option), s)) for valid, base, existing, augmented in inner]\\n            for option in grid}\\n        means = {name: float(np.mean(values)) for name, values in scores.items()}\\n        # Options are in tie order, and the first of equal scores is kept.\\n        chosen = first_best(grid, lambda option: means[option_name(*option)])\\n        rows = np.flatnonzero(folds == fold)\\n        fit = np.flatnonzero(folds != fold)\\n        existing = claude_glm.specialist_probabilities(blocks, INCUMBENT, truth, fit, rows, frozen)\\n        augmented = {encoder: claude_glm.specialist_probabilities(blocks, f\\"{INCUMBENT}+{encoder}\\", truth, fit,\\n                                                                  rows, frozen)\\n                     for encoder in encoders}\\n        incumbent[rows] = claude_glm.redistribute(current[rows], existing, s)\\n        candidate[rows] = claude_glm.redistribute(current[rows], mixed(existing, augmented, *chosen), s)\\n        for option in grid:\\n            by_option[option_name(*option)][rows] = claude_glm.redistribute(\\n                current[rows], mixed(existing, augmented, *option), s)\\n        selection.append({\\"outer_fold\\": fold, \\"s\\": s, \\"selected\\": option_name(*chosen),\\n                          \\"inner_mean_balanced_accuracy\\": means,\\n                          \\"inner_balanced_accuracy\\": scores, \\"s_inner_mean_balanced_accuracy\\":\\n                              {str(key): value for key, value in s_scores.items()}})\\n    predictions = {\\"incumbent\\": incumbent.argmax(axis=1), \\"candidate\\": candidate.argmax(axis=1)}\\n    systems = {name: ensemble._system_metrics(truth, values, folds) for name, values in predictions.items()}\\n    paired = ensemble.prompt_bootstrap(truth, predictions[\\"incumbent\\"], predictions[\\"candidate\\"], prompts,\\n                                       samples, seed)\\n    outside = ~np.isin(truth, claude_glm.INDEX)\\n    gained = (predictions[\\"incumbent\\"] != truth) & (predictions[\\"candidate\\"] == truth)\\n    lost = (predictions[\\"incumbent\\"] == truth) & (predictions[\\"candidate\\"] != truth)\\n    frame = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].rename(columns={\\"model\\": \\"true_model\\"}).assign(outer_fold=folds)\\n    for name, values in predictions.items():\\n        frame[f\\"prediction::{name}\\"] = np.asarray(LABELS)[values]\\n    frame.to_parquet(output / \\"predictions.parquet\\", index=False)\\n    result = {\\n        \\"incumbent\\": INCUMBENT, \\"encoders\\": list(encoders), \\"mix_weights\\": list(MIX_WEIGHTS),\\n        \\"systems\\": systems, \\"paired\\": paired, \\"selection\\": selection,\\n        \\"specialist_recall\\": {name: {label: systems[name][\\"recall\\"][label] for label in claude_glm.SPECIALIST}\\n                              for name in systems},\\n        \\"confusions\\": {name: _confusions(truth, values) for name, values in predictions.items()},\\n        \\"changes\\": {\\"gained\\": int(gained.sum()), \\"lost\\": int(lost.sum()),\\n                    \\"gained_outside_specialist\\": int((gained & outside).sum()),\\n                    \\"lost_outside_specialist\\": int((lost & outside).sum())},\\n        # Each option applied in every fold: a diagnostic, never a selection.\\n        \\"option_balanced_accuracy\\": {name: _ba(truth, values) for name, values in by_option.items()},\\n        \\"groups\\": int(train[\\"prompt_id\\"].nunique()), \\"groups_sha256\\": claude_glm.groups_sha256(train),\\n        \\"train_fingerprint\\": fingerprint_frame(train), \\"provenance\\": provenance or {},\\n    }\\n    claude_glm._json(output / \\"comparison.json\\", result)\\n    return result\\n\\n\\ndef load_encoder(name: str, path: Path, train: pd.DataFrame) -> np.ndarray:\\n    \\"\\"\\"An added encoder\'s archive, which must come from that pinned encoder.\\"\\"\\"\\n    if name not in ENCODERS:\\n        raise ValueError(f\\"Unknown encoder {name!r}; expected one of {sorted(ENCODERS)}\\")\\n    features, metadata = load_feature_archive(path, train)\\n    spec = ENCODERS[name]\\n    if (metadata.get(\\"model_id\\"), metadata.get(\\"model_revision\\")) != (spec[\\"model_id\\"], spec[\\"model_revision\\"]):\\n        raise ValueError(f\\"{path} does not hold {name} features at the pinned revision\\")\\n    return features\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--plan\\", type=Path, required=True)\\n    parser.add_argument(\\"--qwen-features\\", type=Path, required=True)\\n    parser.add_argument(\\"--styledistance-features\\", type=Path, required=True)\\n    parser.add_argument(\\"--encoder\\", nargs=2, action=\\"append\\", metavar=(\\"NAME\\", \\"ARCHIVE\\"), required=True,\\n                        help=\\"an added encoder\'s feature archive, e.g. luar artifacts/.../luar-train.npz\\")\\n    parser.add_argument(\\"--current-oof\\", type=Path, required=True)\\n    parser.add_argument(\\"--current-inner\\", type=Path, nargs=\\"+\\", required=True)\\n    parser.add_argument(\\"--groups\\", type=Path, help=\\"CSV of id, prompt_id regrouping rows (e.g. by topic)\\")\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    args = parser.parse_args(argv)\\n    train = read_train(args.train)\\n    blocks = claude_glm._load_blocks(train, args.qwen_features, args.styledistance_features)\\n    for name, path in args.encoder:\\n        blocks[name] = load_encoder(name, Path(path), train)\\n    plan = pd.read_csv(args.plan, dtype={\\"id\\": str, \\"prompt_id\\": str}, keep_default_na=False)\\n    train, plan = claude_glm._grouped(train, plan, args.groups)\\n    cache = pd.concat([pd.read_parquet(path) for path in args.current_inner], ignore_index=True)\\n    inputs = [args.plan, args.qwen_features, args.styledistance_features, args.current_oof, *args.current_inner,\\n              *(Path(path) for _, path in args.encoder), *([args.groups] if args.groups else [])]\\n    result = evaluate(train, blocks, tuple(name for name, _ in args.encoder), plan,\\n                      pd.read_parquet(args.current_oof), cache, args.output_dir,\\n                      provenance={\\"input_sha256\\": {str(path): claude_glm._sha256(path) for path in inputs}})\\n    paired = result[\\"paired\\"]\\n    print(json.dumps({\\n        \\"balanced_accuracy\\": {name: round(values[\\"balanced_accuracy\\"], 4) for name, values in result[\\"systems\\"].items()},\\n        \\"delta\\": round(paired[\\"balanced_accuracy_delta\\"], 4),\\n        \\"delta_95_interval\\": [round(bound, 4) for bound in paired[\\"delta_95_interval\\"]],\\n        \\"selected\\": [item[\\"selected\\"] for item in result[\\"selection\\"]], \\"s\\": [item[\\"s\\"] for item in result[\\"selection\\"]],\\n        \\"changes\\": result[\\"changes\\"],\\n        \\"option_balanced_accuracy\\": {name: round(value, 4) for name, value in result[\\"option_balanced_accuracy\\"].items()},\\n    }, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    from ntl.specialist_mix import main as imported_main\\n\\n    raise SystemExit(imported_main())\\n", "ntl/style_encoders.py": "\\"\\"\\"Frozen LUAR-MUD and LUSR passage embeddings for the Claude-GLM specialist experiment.\\n\\nReadouts, fixed before validation (docs/claude-glm.md, Added encoders):\\n- Each passage is encoded on its own; for LUAR that is one passage per episode.\\n- At most 512 tokens per passage (no training passage exceeds 387, so none is truncated).\\n- Masked mean over the last hidden state, special tokens included, as both models pool.\\n- LUAR then applies its trained 768 -> 512 projection. With one passage per episode its\\n  episode attention and max are identities, so this is its whole forward pass, built\\n  here from the checkpoint\'s weights: no remote code and no einops are needed offline.\\n- Every embedding is L2-normalized, as both models are compared by cosine similarity.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport gc\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\nfrom .data import fingerprint_frame, read_train\\nfrom .frozen_gemini import save_feature_archive\\n\\nENCODERS = {\\n    \\"luar-mud\\": {\\"model_id\\": \\"rrivera1849/LUAR-MUD\\", \\"model_revision\\": \\"f1db50251805ed69b43cf4f72ea2f0e231f36a1c\\",\\n                 \\"model_type\\": \\"LUAR\\", \\"layers\\": 6, \\"hidden\\": 768, \\"dimension\\": 512,\\n                 \\"weights_sha256\\": \\"443939bb2df7a31f73890e6013ae20021e1fffed47f6142fe813257e6501e63b\\"},\\n    \\"lusr\\": {\\"model_id\\": \\"rrivera1849/LUSR\\", \\"model_revision\\": \\"ab7852825ad1be40112bbfdaf1f9e2315a1727b7\\",\\n             \\"model_type\\": \\"roberta\\", \\"layers\\": 12, \\"hidden\\": 768, \\"dimension\\": 768,\\n             \\"weights_sha256\\": \\"9dc936bd31f349d874a79880549a5a9a9ca0a7ead64bebb83904cac2666cd5b9\\"},\\n}\\nMAX_LENGTH = 512\\nPOOLING = \\"masked_mean_including_special_tokens\\"\\n\\n\\ndef validate_snapshot(directory: str | Path, key: str) -> dict:\\n    \\"\\"\\"The pinned revision and weights, exactly the recorded files at their recorded sizes (nothing\\n    unlisted, no code), and the reviewed architecture.\\"\\"\\"\\n    spec = ENCODERS[key]\\n    directory = Path(directory)\\n    manifest = json.loads((directory / \\"snapshot_manifest.json\\").read_text())\\n    if (manifest.get(\\"schema_version\\") != 1 or manifest.get(\\"repo_id\\") != spec[\\"model_id\\"]\\n            or manifest.get(\\"revision\\") != spec[\\"model_revision\\"] or manifest.get(\\"includes_weights\\") is not True):\\n        raise ValueError(\\"Expected a full snapshot of the pinned encoder revision\\")\\n    names = set()\\n    for entry in manifest.get(\\"files\\", []):\\n        relative = Path(entry[\\"path\\"])\\n        if relative.is_absolute() or \\"..\\" in relative.parts:\\n            raise ValueError(\\"Unsafe path in encoder snapshot manifest\\")\\n        path = directory / relative\\n        if path.is_symlink() or not path.is_file() or path.stat().st_size != entry[\\"size\\"]:\\n            raise ValueError(f\\"Missing or changed encoder snapshot file: {relative}\\")\\n        names.add(str(relative))\\n    on_disk = {str(path.relative_to(directory)) for path in directory.rglob(\\"*\\")\\n               if path.is_file() and path.relative_to(directory).parts[0] != \\".cache\\"\\n               and path.name != \\"snapshot_manifest.json\\"}\\n    if \\"model.safetensors\\" not in names or any(name.endswith(\\".py\\") for name in on_disk | names):\\n        raise ValueError(\\"The encoders load safetensors weights and never remote code\\")\\n    if on_disk != names:\\n        raise ValueError(\\"Encoder snapshot holds files its manifest does not record\\")\\n    digest = hashlib.sha256()\\n    with (directory / \\"model.safetensors\\").open(\\"rb\\") as handle:\\n        for block in iter(lambda: handle.read(1 << 20), b\\"\\"):\\n            digest.update(block)\\n    if digest.hexdigest() != spec[\\"weights_sha256\\"]:\\n        raise ValueError(\\"Encoder weights differ from the pinned checkpoint\\")\\n    config = json.loads((directory / \\"config.json\\").read_text())\\n    backbone = config.get(\\"backbone_config\\", {}) if key == \\"luar-mud\\" else config\\n    if (config.get(\\"model_type\\"), backbone.get(\\"num_hidden_layers\\"), backbone.get(\\"hidden_size\\")) != (\\n            spec[\\"model_type\\"], spec[\\"layers\\"], spec[\\"hidden\\"]):\\n        raise ValueError(\\"Encoder architecture differs from the reviewed checkpoint\\")\\n    return manifest\\n\\n\\ndef load(directory: str | Path, key: str, device: str = \\"cpu\\"):\\n    \\"\\"\\"(tokenizer, RoBERTa backbone, LUAR projection or None), from the checkpoint\'s own tensors.\\"\\"\\"\\n    import torch\\n    from safetensors.torch import load_file\\n    from transformers import AutoTokenizer, RobertaConfig, RobertaModel\\n\\n    directory = Path(directory)\\n    config = json.loads((directory / \\"config.json\\").read_text())\\n    state = load_file(str(directory / \\"model.safetensors\\"))\\n    projection = None\\n    if key == \\"luar-mud\\":\\n        backbone_config = RobertaConfig(**{name: value for name, value in config[\\"backbone_config\\"].items()\\n                                           if name != \\"model_type\\"})\\n        backbone_state = {name.removeprefix(\\"transformer.\\"): value for name, value in state.items()\\n                          if name.startswith(\\"transformer.\\")}\\n        projection = torch.nn.Linear(config[\\"backbone_config\\"][\\"hidden_size\\"], config[\\"embedding_size\\"])\\n        projection.load_state_dict({\\"weight\\": state[\\"linear.weight\\"], \\"bias\\": state[\\"linear.bias\\"]})\\n        if len(backbone_state) + 2 != len(state):\\n            raise ValueError(\\"Unexpected tensors in the LUAR checkpoint\\")\\n    else:\\n        backbone_config = RobertaConfig.from_pretrained(str(directory), local_files_only=True)\\n        backbone_state = state\\n    model = RobertaModel(backbone_config, add_pooling_layer=any(name.startswith(\\"pooler.\\") for name in backbone_state))\\n    result = model.load_state_dict(backbone_state, strict=False)\\n    if result.unexpected_keys or [name for name in result.missing_keys if not name.endswith(\\"position_ids\\")]:\\n        raise ValueError(\\"The checkpoint does not match the RoBERTa backbone exactly\\")\\n    model.to(device).eval().requires_grad_(False)\\n    if projection is not None:\\n        projection.to(device).eval().requires_grad_(False)\\n    tokenizer = AutoTokenizer.from_pretrained(str(directory), local_files_only=True, trust_remote_code=False,\\n                                              padding_side=\\"right\\", use_fast=True)\\n    return tokenizer, model, projection\\n\\n\\ndef embed(texts: list[str], tokenizer, model, projection, *, batch_size: int, device: str) -> np.ndarray:\\n    import torch\\n\\n    chunks = []\\n    with torch.inference_mode():\\n        for start in range(0, len(texts), batch_size):\\n            batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True, max_length=MAX_LENGTH,\\n                              return_tensors=\\"pt\\").to(device)\\n            hidden = model(**batch).last_hidden_state\\n            mask = batch[\\"attention_mask\\"].unsqueeze(-1).to(hidden.dtype)\\n            vectors = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)\\n            if projection is not None:\\n                vectors = projection(vectors)\\n            norms = vectors.norm(p=2, dim=1, keepdim=True)\\n            if not torch.isfinite(vectors).all() or torch.any(norms <= 0):\\n                raise ValueError(\\"Encoder produced non-finite or zero embeddings\\")\\n            chunks.append((vectors / norms).cpu().numpy())\\n    return np.concatenate(chunks).astype(np.float32, copy=False)\\n\\n\\ndef extract(texts: list[str], directory: str | Path, key: str, *, batch_size: int = 16,\\n            device: str = \\"cpu\\") -> tuple[np.ndarray, dict]:\\n    if not texts or any(not isinstance(text, str) or not text.strip() for text in texts):\\n        raise ValueError(\\"Encoder inputs must be nonempty passage strings\\")\\n    spec = ENCODERS[key]\\n    validate_snapshot(directory, key)\\n    started = time.perf_counter()\\n    tokenizer, model, projection = load(directory, key, device)\\n    try:\\n        lengths = [len(ids) for ids in tokenizer(texts, truncation=False)[\\"input_ids\\"]]\\n        embeddings = embed(texts, tokenizer, model, projection, batch_size=batch_size, device=device)\\n    finally:\\n        del model, projection\\n        gc.collect()\\n    if embeddings.shape != (len(texts), spec[\\"dimension\\"]) or not np.allclose(\\n            np.linalg.norm(embeddings, axis=1), 1, atol=1e-5):\\n        raise ValueError(\\"Embeddings must be unit vectors of the reviewed dimension\\")\\n    options = {\\"pooling\\": POOLING, \\"max_length\\": MAX_LENGTH, \\"normalization\\": \\"l2\\", \\"padding_side\\": \\"right\\",\\n               \\"compute_dtype\\": \\"float32\\", \\"one_passage_per_episode\\": True,\\n               \\"projection\\": \\"luar_768_to_512\\" if key == \\"luar-mud\\" else None, \\"remote_code\\": False}\\n    metadata = {\\"model_id\\": spec[\\"model_id\\"], \\"model_revision\\": spec[\\"model_revision\\"], \\"encoder_frozen\\": True,\\n                \\"feature_options\\": options,\\n                \\"profile\\": {\\"rows\\": len(texts), \\"dimension\\": spec[\\"dimension\\"], \\"batch_size\\": batch_size,\\n                            \\"elapsed_seconds\\": time.perf_counter() - started, \\"device\\": device,\\n                            \\"truncated_rows\\": int(sum(length > MAX_LENGTH for length in lengths)),\\n                            \\"maximum_input_tokens\\": int(max(lengths)),\\n                            \\"token_length_quantiles\\": {str(q): float(np.quantile(lengths, q)) for q in (0.5, 0.9, 0.99)},\\n                            \\"versions\\": {package: importlib.metadata.version(package)\\n                                         for package in (\\"torch\\", \\"transformers\\", \\"numpy\\")}}}\\n    return embeddings, metadata\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--model-key\\", choices=tuple(ENCODERS), required=True)\\n    parser.add_argument(\\"--model-path\\", type=Path, required=True)\\n    parser.add_argument(\\"--output\\", type=Path, required=True)\\n    parser.add_argument(\\"--device\\", default=\\"cpu\\")\\n    parser.add_argument(\\"--batch-size\\", type=int, default=16)\\n    args = parser.parse_args(argv)\\n    if args.output.exists():\\n        raise ValueError(\\"Feature output already exists; preserve it or choose another output path\\")\\n    train = read_train(args.train)\\n    embeddings, metadata = extract(train[\\"text\\"].tolist(), args.model_path, args.model_key,\\n                                   batch_size=args.batch_size, device=args.device)\\n    metadata[\\"train_fingerprint\\"] = fingerprint_frame(train)\\n    saved = save_feature_archive(args.output, train[\\"id\\"].tolist(), embeddings, metadata, texts=train[\\"text\\"].tolist())\\n    args.output.with_suffix(\\".json\\").write_text(json.dumps(saved, indent=2) + \\"\\\\n\\")\\n    print(json.dumps(metadata[\\"profile\\"], indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    from ntl.style_encoders import main as imported_main\\n\\n    raise SystemExit(imported_main())\\n", "ntl/styledistance_head.py": "\\"\\"\\"Does the StyleDistance signal help beyond Claude-GLM? An 18-class Qwen + StyleDistance head.\\n\\nThe incumbent is the validated system: the current CPU + frozen-Qwen blend whose\\nClaude-GLM probability mass the Qwen + StyleDistance specialist redistributes at the\\nweight s its inner folds choose (`claude_glm.evaluate`). The candidate head is the\\nspecialist\'s classifier (block-variance balanced, the same C) trained on all 18 classes\\nand blended after the incumbent: p = (1 - b) p_incumbent + b r, so b = 0 is the\\nincumbent exactly.\\n\\nInner prompt folds of each outer training partition choose s first, as the incumbent\\ndoes, then b from {0, 0.25, 0.5, 0.75, 1}, scored on all 18 classes; ties keep the\\nsmaller b. A Qwen-only 18-class head, selected and blended the same way, is a diagnostic\\ncontrol: it is the blend\'s own Qwen head, so it separates StyleDistance from more weight\\non Qwen. Only the candidate is compared against the promotion rule.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import balanced_accuracy_score\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom . import claude_glm, ensemble, neural\\nfrom .constants import LABELS\\nfrom .data import fingerprint_frame, read_train\\nfrom .lora import AdapterConfig, validate_plan\\nfrom .selection import first_best\\n\\nINCUMBENT = \\"qwen+styledistance\\"\\nCANDIDATE = \\"qwen+styledistance\\"\\nCONTROL = \\"qwen\\"\\nHEADS = {\\"candidate\\": CANDIDATE, \\"control\\": CONTROL}\\nWEIGHTS = (0.0, 0.25, 0.5, 0.75, 1.0)\\nGPT = (\\"gpt-5.6-luna\\", \\"gpt-5.6-sol\\", \\"gpt-5.6-terra\\")\\n\\n\\ndef head_probabilities(blocks: dict[str, np.ndarray], feature_set: str, truth: np.ndarray, fit: np.ndarray,\\n                       rows: np.ndarray, frozen: neural.FrozenConfig) -> np.ndarray:\\n    \\"\\"\\"The specialist\'s classifier fit on every class of `fit`; 18-class probabilities for `rows`.\\"\\"\\"\\n    names = claude_glm.feature_names(feature_set)\\n    features = np.hstack([blocks[name] for name in names])\\n    head = claude_glm.specialist_head(tuple(blocks[name].shape[1] for name in names), frozen)\\n    with threadpool_limits(limits=2):\\n        head.fit(features[fit], truth[fit])\\n        return neural._probabilities(head, features[rows])\\n\\n\\ndef blended(incumbent: np.ndarray, head: np.ndarray, weight: float) -> np.ndarray:\\n    return incumbent if weight == 0 else (1 - weight) * incumbent + weight * head\\n\\n\\ndef _ba(truth: np.ndarray, probabilities: np.ndarray) -> float:\\n    return float(balanced_accuracy_score(truth, probabilities.argmax(axis=1)))\\n\\n\\ndef _gpt_confusion(truth: np.ndarray, predictions: np.ndarray) -> dict:\\n    \\"\\"\\"Luna/Sol/Terra rows by predicted label (outside the three as \\"other\\"), and the share\\n    of those rows predicted as another of the three.\\"\\"\\"\\n    matrix, within = {}, 0\\n    for label in GPT:\\n        predicted = predictions[truth == LABELS.index(label)]\\n        counts = {other: int((predicted == LABELS.index(other)).sum()) for other in GPT}\\n        counts[\\"other\\"] = int(len(predicted) - sum(counts.values()))\\n        matrix[label] = counts\\n        within += sum(count for other, count in counts.items() if other not in (label, \\"other\\"))\\n    rows = int(np.isin(truth, [LABELS.index(label) for label in GPT]).sum())\\n    return {\\"matrix\\": matrix, \\"within_family_errors\\": within, \\"within_family_error_share\\": within / rows}\\n\\n\\ndef _contributions(first: dict, second: dict) -> dict:\\n    \\"\\"\\"Each class\'s part of the balanced-accuracy change: its recall change over 18.\\"\\"\\"\\n    by_class = {label: (second[\\"recall\\"][label] - first[\\"recall\\"][label]) / len(LABELS) for label in LABELS}\\n    delta = second[\\"balanced_accuracy\\"] - first[\\"balanced_accuracy\\"]\\n    gpt = sum(by_class[label] for label in GPT)\\n    return {\\"by_class\\": by_class, \\"gpt_5_6\\": gpt, \\"gpt_5_6_share\\": gpt / delta if delta else None}\\n\\n\\ndef _changes(truth: np.ndarray, first: np.ndarray, second: np.ndarray) -> dict:\\n    gained, lost = (first != truth) & (second == truth), (first == truth) & (second != truth)\\n    return {\\"gained\\": int(gained.sum()), \\"lost\\": int(lost.sum()),\\n            \\"by_class\\": {label: {\\"gained\\": int(gained[truth == index].sum()), \\"lost\\": int(lost[truth == index].sum())}\\n                         for index, label in enumerate(LABELS)}}\\n\\n\\ndef evaluate(train: pd.DataFrame, blocks: dict[str, np.ndarray], plan: pd.DataFrame, current_oof: pd.DataFrame,\\n             current_cache: pd.DataFrame, output_dir: str | Path, *, frozen: neural.FrozenConfig | None = None,\\n             samples: int = 10000, seed: int = 2026, provenance: dict | None = None) -> dict:\\n    frozen = frozen or neural.FrozenConfig()\\n    plan = validate_plan(train, plan, AdapterConfig())\\n    output = Path(output_dir)\\n    if (output / \\"comparison.json\\").exists():\\n        raise ValueError(\\"Use a new output directory; an existing comparison must not be overwritten\\")\\n    if set(current_cache[\\"groups_sha256\\"]) != {claude_glm.groups_sha256(train)}:\\n        raise ValueError(\\"Cached current-system inner predictions were built under a different row grouping\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    truth, prompts = ensemble._targets(train), train[\\"prompt_id\\"].to_numpy()\\n    folds = plan[\\"outer_fold\\"].to_numpy()\\n    current = claude_glm._aligned_current(current_oof, train, folds)\\n    incumbent = np.empty_like(current)\\n    heads = {system: np.empty_like(current) for system in HEADS}\\n    probabilities = {system: np.empty_like(current) for system in HEADS}\\n    by_weight = {system: {weight: np.empty_like(current) for weight in WEIGHTS} for system in HEADS}\\n    selection = []\\n    for fold in range(5):\\n        column = plan[f\\"inner_{fold}\\"].to_numpy()\\n        inner = []\\n        for inner_fold, (valid, inner_current) in claude_glm._inner_current(current_cache, train, plan,\\n                                                                            fold).items():\\n            fit = np.flatnonzero((folds != fold) & (column != inner_fold))\\n            specialist = claude_glm.specialist_probabilities(blocks, INCUMBENT, truth, fit, valid, frozen)\\n            inner.append((valid, inner_current, specialist,\\n                          {system: head_probabilities(blocks, name, truth, fit, valid, frozen)\\n                           for system, name in HEADS.items()}))\\n        # The incumbent\'s own choice of s, as claude_glm.evaluate makes it.\\n        s_scores = {s: float(np.mean([_ba(truth[valid], claude_glm.redistribute(base, specialist, s))\\n                                      for valid, base, specialist, _ in inner]))\\n                    for s in claude_glm.WEIGHTS}\\n        s = first_best(claude_glm.WEIGHTS, s_scores.__getitem__)\\n        scores = {system: {weight: [_ba(truth[valid], blended(claude_glm.redistribute(base, specialist, s),\\n                                                               head[system], weight))\\n                                    for valid, base, specialist, head in inner]\\n                           for weight in WEIGHTS}\\n                  for system in HEADS}\\n        means = {system: {weight: float(np.mean(values)) for weight, values in by.items()}\\n                 for system, by in scores.items()}\\n        # Ties, including those float rounding splits, keep the smaller b: stay with the incumbent.\\n        chosen = {system: first_best(WEIGHTS, means[system].__getitem__) for system in HEADS}\\n        rows, fit = np.flatnonzero(folds == fold), np.flatnonzero(folds != fold)\\n        specialist = claude_glm.specialist_probabilities(blocks, INCUMBENT, truth, fit, rows, frozen)\\n        incumbent[rows] = claude_glm.redistribute(current[rows], specialist, s)\\n        for system, name in HEADS.items():\\n            heads[system][rows] = head_probabilities(blocks, name, truth, fit, rows, frozen)\\n            for weight in WEIGHTS:\\n                by_weight[system][weight][rows] = blended(incumbent[rows], heads[system][rows], weight)\\n            probabilities[system][rows] = by_weight[system][chosen[system]][rows]\\n        selection.append({\\"outer_fold\\": fold, \\"s\\": s, \\"b\\": chosen,\\n                          \\"inner_mean_balanced_accuracy\\": {system: {str(key): value for key, value in by.items()}\\n                                                           for system, by in means.items()},\\n                          \\"inner_balanced_accuracy\\": {system: {str(key): value for key, value in by.items()}\\n                                                      for system, by in scores.items()},\\n                          \\"s_inner_mean_balanced_accuracy\\": {str(key): value for key, value in s_scores.items()}})\\n    predictions = {\\"incumbent\\": incumbent.argmax(axis=1)} | {system: values.argmax(axis=1)\\n                                                             for system, values in probabilities.items()}\\n    systems = {name: ensemble._system_metrics(truth, values, folds) for name, values in predictions.items()}\\n    paired = {f\\"{name}_vs_incumbent\\": ensemble.prompt_bootstrap(truth, predictions[\\"incumbent\\"], predictions[name],\\n                                                                prompts, samples, seed)\\n              for name in HEADS}\\n    paired[\\"candidate_vs_control\\"] = ensemble.prompt_bootstrap(truth, predictions[\\"control\\"],\\n                                                               predictions[\\"candidate\\"], prompts, samples, seed)\\n    frame = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].rename(columns={\\"model\\": \\"true_model\\"}).assign(outer_fold=folds)\\n    for name, values in predictions.items():\\n        frame[f\\"prediction::{name}\\"] = np.asarray(LABELS)[values]\\n    frame.to_parquet(output / \\"predictions.parquet\\", index=False)\\n    for system, values in heads.items():\\n        np.save(output / f\\"head-{HEADS[system]}.npy\\", values)\\n    result = {\\n        \\"incumbent\\": INCUMBENT, \\"heads\\": HEADS, \\"weights\\": list(WEIGHTS), \\"gpt_classes\\": list(GPT),\\n        \\"systems\\": systems, \\"paired\\": paired, \\"selection\\": selection,\\n        \\"changes\\": {system: _changes(truth, predictions[\\"incumbent\\"], predictions[system]) for system in HEADS},\\n        \\"contributions\\": {system: _contributions(systems[\\"incumbent\\"], systems[system]) for system in HEADS},\\n        \\"gpt_confusion\\": {name: _gpt_confusion(truth, values) for name, values in predictions.items()},\\n        # Each head alone, and each b applied in every fold: diagnostics, never a selection.\\n        \\"head_alone\\": {system: ensemble._system_metrics(truth, values.argmax(axis=1), folds)\\n                       for system, values in heads.items()},\\n        \\"weight_balanced_accuracy\\": {system: {str(weight): _ba(truth, values) for weight, values in by.items()}\\n                                     for system, by in by_weight.items()},\\n        \\"groups\\": int(train[\\"prompt_id\\"].nunique()), \\"groups_sha256\\": claude_glm.groups_sha256(train),\\n        \\"train_fingerprint\\": fingerprint_frame(train), \\"provenance\\": provenance or {},\\n    }\\n    claude_glm._json(output / \\"comparison.json\\", result)\\n    return result\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--plan\\", type=Path, required=True)\\n    parser.add_argument(\\"--qwen-features\\", type=Path, required=True)\\n    parser.add_argument(\\"--styledistance-features\\", type=Path, required=True)\\n    parser.add_argument(\\"--current-oof\\", type=Path, required=True)\\n    parser.add_argument(\\"--current-inner\\", type=Path, nargs=\\"+\\", required=True)\\n    parser.add_argument(\\"--groups\\", type=Path, help=\\"CSV of id, prompt_id regrouping rows (e.g. by topic)\\")\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    args = parser.parse_args(argv)\\n    train = read_train(args.train)\\n    blocks = claude_glm._load_blocks(train, args.qwen_features, args.styledistance_features)\\n    plan = pd.read_csv(args.plan, dtype={\\"id\\": str, \\"prompt_id\\": str}, keep_default_na=False)\\n    train, plan = claude_glm._grouped(train, plan, args.groups)\\n    cache = pd.concat([pd.read_parquet(path) for path in args.current_inner], ignore_index=True)\\n    inputs = [args.plan, args.qwen_features, args.styledistance_features, args.current_oof, *args.current_inner,\\n              *([args.groups] if args.groups else [])]\\n    result = evaluate(train, blocks, plan, pd.read_parquet(args.current_oof), cache, args.output_dir,\\n                      provenance={\\"input_sha256\\": {str(path): claude_glm._sha256(path) for path in inputs}})\\n    print(json.dumps({\\n        \\"balanced_accuracy\\": {name: round(values[\\"balanced_accuracy\\"], 4) for name, values in result[\\"systems\\"].items()},\\n        \\"paired\\": {name: [round(values[\\"balanced_accuracy_delta\\"], 4)]\\n                   + [round(bound, 4) for bound in values[\\"delta_95_interval\\"]]\\n                   for name, values in result[\\"paired\\"].items()},\\n        \\"s\\": [item[\\"s\\"] for item in result[\\"selection\\"]],\\n        \\"b\\": {system: [item[\\"b\\"][system] for item in result[\\"selection\\"]] for system in HEADS},\\n        \\"gpt_recall\\": {name: {label: round(values[\\"recall\\"][label], 3) for label in GPT}\\n                       for name, values in result[\\"systems\\"].items()},\\n        \\"gpt_5_6_points\\": {system: round(100 * values[\\"gpt_5_6\\"], 3) for system, values in result[\\"contributions\\"].items()},\\n        \\"changes\\": {system: [values[\\"gained\\"], values[\\"lost\\"]] for system, values in result[\\"changes\\"].items()},\\n        \\"head_alone\\": {system: round(values[\\"balanced_accuracy\\"], 4) for system, values in result[\\"head_alone\\"].items()},\\n    }, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    from ntl.styledistance_head import main as imported_main\\n\\n    raise SystemExit(imported_main())\\n", "ntl/topic_cv.py": "\\"\\"\\"Topic-grouped validation: does the CPU-Qwen blend lose more than CPU on unseen topics?\\n\\nThe data carry no prompt text, so each prompt\'s topic is the content-word TF-IDF of its\\n18 responses (label-free). Prompts are clustered into K topics, and whole topics are held\\nout together in both the outer folds and the blend\'s inner weight-selection folds. Coarser\\nK means a larger topic distance between training and held-out prompts. Row-level folds\\n(prompt leakage) and the original prompt folds bracket the topic conditions.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom dataclasses import asdict\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy.stats import spearmanr\\nfrom sklearn.cluster import KMeans\\nfrom sklearn.decomposition import TruncatedSVD\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.preprocessing import normalize\\n\\nfrom . import ensemble, neural\\nfrom .data import read_train, validate_train\\nfrom .folds import build_fold_manifest, save_fold_manifest\\n\\n\\ndef prompt_topics(train: pd.DataFrame, seed: int = 2026, components: int = 60) -> dict:\\n    \\"\\"\\"Label-free topic vectors: one TF-IDF document per prompt, from all its responses.\\"\\"\\"\\n    train = validate_train(train)\\n    documents = train.groupby(\\"prompt_id\\")[\\"text\\"].apply(\\"\\\\n\\".join).sort_index()\\n    vectorizer = TfidfVectorizer(lowercase=True, stop_words=\\"english\\", min_df=2, max_df=0.5,\\n                                 sublinear_tf=True, token_pattern=r\\"(?u)\\\\b[a-zA-Z]{3,}\\\\b\\")\\n    tfidf = vectorizer.fit_transform(documents.to_numpy())\\n    components = min(components, tfidf.shape[1] - 1, len(documents) - 1)\\n    latent = normalize(TruncatedSVD(n_components=components, random_state=seed).fit_transform(tfidf))\\n    similarity = (tfidf @ tfidf.T).toarray()\\n    np.fill_diagonal(similarity, 0.0)\\n    return {\\"prompts\\": documents.index.to_numpy(), \\"tfidf\\": tfidf, \\"latent\\": latent,\\n            \\"similarity\\": similarity, \\"terms\\": np.asarray(vectorizer.get_feature_names_out())}\\n\\n\\ndef topic_clusters(topics: dict, k: int, seed: int = 2026) -> dict[str, str]:\\n    labels = KMeans(n_clusters=k, n_init=50, random_state=seed).fit_predict(topics[\\"latent\\"])\\n    return {prompt: f\\"topic{k}-{label:02d}\\" for prompt, label in zip(topics[\\"prompts\\"], labels, strict=True)}\\n\\n\\ndef describe_clusters(topics: dict, groups: dict[str, str], top: int = 8) -> dict[str, dict]:\\n    prompts = topics[\\"prompts\\"]\\n    result = {}\\n    for group in sorted(set(groups.values())):\\n        mask = np.asarray([groups[p] == group for p in prompts])\\n        centroid = np.asarray(topics[\\"tfidf\\"][mask].mean(axis=0)).ravel()\\n        result[group] = {\\"prompts\\": int(mask.sum()), \\"terms\\": topics[\\"terms\\"][np.argsort(-centroid)[:top]].tolist()}\\n    return result\\n\\n\\ndef regroup(train: pd.DataFrame, groups: dict[str, str]) -> pd.DataFrame:\\n    \\"\\"\\"Replace prompt_id by its group so every outer and inner split holds out whole groups.\\"\\"\\"\\n    regrouped = train.copy()\\n    regrouped[\\"prompt_id\\"] = regrouped[\\"prompt_id\\"].map(groups)\\n    if regrouped[\\"prompt_id\\"].isna().any():\\n        raise ValueError(\\"Every prompt needs a group\\")\\n    return regrouped\\n\\n\\ndef nearest_training_similarity(topics: dict, prompt_folds: dict[str, int]) -> np.ndarray:\\n    \\"\\"\\"Per prompt, the topic cosine to its most similar prompt in the training partition.\\"\\"\\"\\n    folds = np.asarray([prompt_folds[p] for p in topics[\\"prompts\\"]])\\n    return np.asarray([topics[\\"similarity\\"][index, folds != folds[index]].max()\\n                       for index in range(len(folds))])\\n\\n\\ndef _prompt_folds(train: pd.DataFrame, manifest: pd.DataFrame) -> dict[str, int]:\\n    folds = train[[\\"id\\", \\"prompt_id\\"]].merge(manifest[[\\"id\\", \\"fold\\"]], on=\\"id\\", validate=\\"one_to_one\\")\\n    per_prompt = folds.groupby(\\"prompt_id\\")[\\"fold\\"].agg([\\"min\\", \\"max\\"])\\n    if (per_prompt[\\"min\\"] != per_prompt[\\"max\\"]).any():\\n        return {}  # Row-level folds split prompts; nearest training prompt is the prompt itself.\\n    return per_prompt[\\"min\\"].to_dict()\\n\\n\\ndef similarity_bins(train: pd.DataFrame, oof: pd.DataFrame, topics: dict, prompt_folds: dict[str, int]) -> dict:\\n    \\"\\"\\"On one condition\'s OOF: does each system\'s accuracy depend on topic distance?\\"\\"\\"\\n    nearest = pd.Series(nearest_training_similarity(topics, prompt_folds), index=topics[\\"prompts\\"])\\n    frame = oof.assign(nearest=oof[\\"id\\"].map(train.set_index(\\"id\\")[\\"prompt_id\\"]).map(nearest))\\n    for name in (\\"cpu\\", \\"qwen\\", \\"blend\\"):\\n        frame[f\\"{name}_correct\\"] = frame[f\\"prediction::{name}\\"] == frame[\\"true_model\\"]\\n    frame[\\"tercile\\"] = pd.qcut(frame[\\"nearest\\"], 3, labels=[\\"least similar\\", \\"middle\\", \\"most similar\\"])\\n    bins = {str(tercile): {\\"rows\\": len(group), \\"median_similarity\\": float(group[\\"nearest\\"].median()),\\n                           **{name: float(group[f\\"{name}_correct\\"].mean()) for name in (\\"cpu\\", \\"qwen\\", \\"blend\\")}}\\n            for tercile, group in frame.groupby(\\"tercile\\", observed=True)}\\n    per_prompt = frame.assign(prompt=frame[\\"id\\"].map(train.set_index(\\"id\\")[\\"prompt_id\\"])).groupby(\\"prompt\\").agg(\\n        nearest=(\\"nearest\\", \\"first\\"), cpu=(\\"cpu_correct\\", \\"mean\\"), qwen=(\\"qwen_correct\\", \\"mean\\"),\\n        blend=(\\"blend_correct\\", \\"mean\\"))\\n    def correlation(values):\\n        # Undefined (None) when a system scores every prompt identically.\\n        if values.nunique() < 2:\\n            return None\\n        return float(spearmanr(per_prompt[\\"nearest\\"], values).statistic)\\n\\n    return {\\"terciles\\": bins, \\"spearman_nearest_similarity_vs\\": {\\n        \\"qwen_minus_cpu\\": correlation(per_prompt[\\"qwen\\"] - per_prompt[\\"cpu\\"]),\\n        \\"blend_minus_cpu\\": correlation(per_prompt[\\"blend\\"] - per_prompt[\\"cpu\\"]),\\n        \\"cpu\\": correlation(per_prompt[\\"cpu\\"]), \\"qwen\\": correlation(per_prompt[\\"qwen\\"])}}\\n\\n\\ndef run(train: pd.DataFrame, features: np.ndarray, output_dir: str | Path, ks: tuple[int, ...] = (30, 15, 8),\\n        include_rows: bool = True, config: ensemble.BlendConfig | None = None) -> dict:\\n    config = config or ensemble.BlendConfig()\\n    train = validate_train(train).reset_index(drop=True)\\n    output = Path(output_dir)\\n    if output.exists() and any(output.iterdir()):\\n        raise ValueError(\\"Use a new output directory; an existing experiment must not be overwritten\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    topics = prompt_topics(train, config.seed)\\n    conditions = ([(\\"rows\\", dict(zip(train[\\"prompt_id\\"], train[\\"prompt_id\\"])), True)] if include_rows else [])\\n    conditions.append((\\"prompts\\", {p: p for p in topics[\\"prompts\\"]}, False))\\n    conditions += [(f\\"topics-{k}\\", topic_clusters(topics, k, config.seed), False) for k in ks]\\n    summary = {\\"configuration\\": asdict(config), \\"conditions\\": {}}\\n    for name, groups, by_row in conditions:\\n        frame = train.assign(prompt_id=train[\\"id\\"]) if by_row else regroup(train, groups)\\n        manifest, metadata = build_fold_manifest(frame, n_splits=5, seed=config.seed)\\n        directory = output / name\\n        manifest_path = save_fold_manifest(manifest, metadata, directory / \\"folds\\")\\n        comparison = ensemble.evaluate_blend(frame, features, manifest_path, directory / \\"blend\\", config)\\n        prompt_folds = {} if by_row else _prompt_folds(train, manifest)\\n        nearest = (np.ones(len(topics[\\"prompts\\"])) if by_row\\n                   else nearest_training_similarity(topics, prompt_folds))\\n        record = {\\n            \\"groups\\": int(frame[\\"prompt_id\\"].nunique()),\\n            \\"median_nearest_training_prompt_similarity\\": float(np.median(nearest)),\\n            \\"balanced_accuracy\\": {system: values[\\"balanced_accuracy\\"] for system, values in comparison[\\"systems\\"].items()},\\n            \\"fold_balanced_accuracy\\": {system: values[\\"fold_balanced_accuracy\\"]\\n                                       for system, values in comparison[\\"systems\\"].items()},\\n            \\"selected_weight_per_outer_fold\\": comparison[\\"selected_weight_per_outer_fold\\"],\\n            \\"blend_vs_cpu\\": comparison[\\"paired\\"][\\"blend_vs_cpu\\"],\\n            \\"qwen_vs_cpu\\": comparison[\\"paired\\"][\\"qwen_vs_cpu\\"],\\n        }\\n        if not by_row:\\n            record[\\"clusters\\"] = describe_clusters(topics, groups) if name != \\"prompts\\" else None\\n            record[\\"by_topic_distance\\"] = similarity_bins(\\n                train, pd.read_parquet(directory / \\"blend\\" / \\"oof.parquet\\"), topics, prompt_folds)\\n        summary[\\"conditions\\"][name] = record\\n        (output / \\"summary.json\\").write_text(json.dumps(summary, indent=2, allow_nan=False) + \\"\\\\n\\")\\n        print(json.dumps({\\"condition\\": name, \\"median_nearest_similarity\\": record[\\"median_nearest_training_prompt_similarity\\"],\\n                          **record[\\"balanced_accuracy\\"], \\"weights\\": record[\\"selected_weight_per_outer_fold\\"]}),\\n              flush=True)\\n    return summary\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--features\\", type=Path, required=True)\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    parser.add_argument(\\"--k\\", type=int, nargs=\\"+\\", default=[30, 15, 8])\\n    args = parser.parse_args(argv)\\n    train = read_train(args.train)\\n    features, _ = ensemble.load_archive(args.features, train, neural.FrozenConfig())\\n    run(train, features, args.output_dir, tuple(args.k))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n"}'
SOURCE_SHA256 = 'b5ba081f9a1c0c5955fd93e27751f0f8d80e4738f270f5dadb53700e62cde97f'
assert hashlib.sha256(SOURCE_JSON.encode()).hexdigest() == SOURCE_SHA256
SOURCE_ROOT = Path("/kaggle/working/ntl_source")
for relative, content in json.loads(SOURCE_JSON).items():
    target = SOURCE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()
print("Embedded ntl source SHA-256:", SOURCE_SHA256)


In [ ]:
REQUIRED_PACKAGES = ['numpy==2.3.5', 'pandas==2.2.3', 'scipy==1.16.2', 'scikit-learn==1.7.2', 'pyarrow==21.0.0', 'joblib==1.5.2', 'threadpoolctl==3.6.0', 'packaging==26.3', 'torch==2.8.0', 'transformers==4.57.1', 'accelerate==1.10.1', 'peft==0.17.1', 'huggingface-hub==0.35.3', 'safetensors==0.6.2', 'sentencepiece==0.2.1', "bitsandbytes==0.48.1; sys_platform == 'linux' and platform_machine == 'x86_64'"]
WHEEL_PROFILE = 'neural'
WHEEL_REQUIREMENTS_SHA256 = '7474801f2822a03f36a9a2739fbdee7ca0ae55f30ee3c3d7bfa185582dc23a55'
# Attach the pinned wheel bundle from kaggle_prepare_dependencies.ipynb.
# Set an explicit folder if more than one bundle is attached.
WHEEL_DIR = None  # directory containing wheel_manifest.json and requirements.lock
if WHEEL_DIR is None:
    wheel_candidates = []
    for wheel_manifest in sorted(Path("/kaggle/input").rglob("wheel_manifest.json")):
        try:
            candidate_manifest = json.loads(wheel_manifest.read_text())
        except (OSError, ValueError):
            continue
        if candidate_manifest.get("profile") == WHEEL_PROFILE:
            wheel_candidates.append(wheel_manifest)
    if len(wheel_candidates) == 1:
        WHEEL_DIR = str(wheel_candidates[0].parent)
    elif len(wheel_candidates) > 1:
        raise RuntimeError("Several wheel bundles are attached; set WHEEL_DIR explicitly")
if WHEEL_DIR is None:
    raise RuntimeError(
        f"Attach the prepared {WHEEL_PROFILE} wheel bundle from kaggle_prepare_dependencies.ipynb"
    )
import platform
import subprocess
import venv
bundle_manifest = json.loads((Path(WHEEL_DIR) / "wheel_manifest.json").read_text())
if bundle_manifest.get("profile") != WHEEL_PROFILE:
    raise RuntimeError(f"Attach the {WHEEL_PROFILE} wheel bundle for this notebook")
if (bundle_manifest.get("python") != list(sys.version_info[:2])
        or bundle_manifest.get("machine") != platform.machine()
        or bundle_manifest.get("system") != platform.system()):
    raise RuntimeError("Prepare the wheel bundle on the same Kaggle Python/Linux image as this run")
wheel_requirements_path = Path(WHEEL_DIR) / "requirements.lock"
if not wheel_requirements_path.is_file():
    raise RuntimeError("Wheel bundle is missing requirements.lock")
if hashlib.sha256(wheel_requirements_path.read_bytes()).hexdigest() != WHEEL_REQUIREMENTS_SHA256:
    raise RuntimeError("Wheel requirements differ from this notebook; attach its matching prepared bundle")

# Train and infer in a fresh package environment, independent of packages that
# Kaggle's notebook kernel may already have imported (including audio/vision).
RUNTIME_DIR = Path("/kaggle/working/ntl_env") / (WHEEL_PROFILE + "-" + WHEEL_REQUIREMENTS_SHA256[:12])
# Kaggle can disable ensurepip; the notebook's existing pip can install into
# a pip-less virtual environment through its supported --python option.
venv.EnvBuilder(with_pip=False, system_site_packages=False).create(RUNTIME_DIR)
RUNTIME_PYTHON = str(RUNTIME_DIR / "bin" / "python")
RUNTIME_ENV = dict(os.environ)
RUNTIME_ENV.pop("PYTHONHOME", None)
RUNTIME_ENV["PYTHONPATH"] = str(SOURCE_ROOT)
RUNTIME_ENV["PYTHONNOUSERSITE"] = "1"
RUNTIME_ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
subprocess.run(
    [sys.executable, "-m", "pip", "--python", RUNTIME_PYTHON,
     "install", "--no-index", "--find-links",
     str(WHEEL_DIR), "--require-hashes", "-r", str(wheel_requirements_path)],
    check=True, env=RUNTIME_ENV,
)
version_probe = """
from importlib.metadata import PackageNotFoundError, version
from packaging.requirements import Requirement
import json
import sys
for raw_requirement in json.loads(sys.argv[1]):
    requirement = Requirement(raw_requirement)
    if requirement.marker and not requirement.marker.evaluate():
        continue
    package = requirement.name
    try:
        installed_version = version(package)
    except PackageNotFoundError as exc:
        raise RuntimeError(
            f"Missing {package} in the isolated runtime; recreate the matching wheel bundle."
        ) from exc
    if installed_version not in requirement.specifier:
        raise RuntimeError(
            f"Expected {requirement}; installed {package}=={installed_version}. "
            "Recreate the matching wheel bundle before rerunning."
        )
    print(package, installed_version)
"""
subprocess.run([RUNTIME_PYTHON, "-c", version_probe, json.dumps(REQUIRED_PACKAGES)],
               check=True, env=RUNTIME_ENV)
def run_runtime(module, arguments):
    subprocess.run([RUNTIME_PYTHON, "-m", module, *arguments],
                   check=True, env=RUNTIME_ENV)
print("Isolated runtime:", RUNTIME_PYTHON)


In [ ]:
import csv

# Usually leave this unset. Never discover train.csv recursively: attached
# external datasets may also contain a file with that name.
COMPETITION_DIR = None
official_locations = [
    Path("/kaggle/input/competitions/name-that-llm"),
    Path("/kaggle/input/name-that-llm"),
]
if COMPETITION_DIR is not None:
    official_locations = [Path(COMPETITION_DIR)]
available = [p for p in official_locations
             if (p / "train.csv").is_file() and (p / "test.csv").is_file()]
if len(available) != 1:
    raise RuntimeError(
        "Attach the official Name That LLM competition data. If both official "
        "paths exist, set COMPETITION_DIR explicitly. Found: " + str(available)
    )
DATA_DIR = available[0]
TRAIN = str(DATA_DIR / "train.csv")
TEST = str(DATA_DIR / "test.csv")
SUBMISSION = "/kaggle/working/submission.parquet"
# A failed fresh run must not leave an earlier prediction file available.
Path(SUBMISSION).unlink(missing_ok=True)
def read_input_csv(path, name):
    with open(path, encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if not {"id", "text"}.issubset(reader.fieldnames or []):
            raise ValueError(f"{name} requires id and text columns")
        rows = list(reader)
    if not rows or any(not row["id"] or not row["text"] for row in rows):
        raise ValueError(f"{name} is empty or contains null IDs/text")
    ids = [row["id"] for row in rows]
    if len(set(ids)) != len(ids):
        raise ValueError(f"{name} has duplicate IDs")
    return rows
train_rows = read_input_csv(TRAIN, "train")
test_rows = read_input_csv(TEST, "test")
VALID_LABELS = {
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3",
}
if {row.get("model") for row in train_rows} != VALID_LABELS:
    raise ValueError("Official training labels do not match the expected 18 labels")
print(f"Official train: {len(train_rows):,} rows; current test: {len(test_rows):,} rows")
print("The visible test is a smoke test; scoring replaces it with the hidden test.")


In [ ]:
# Attach the Qwen3-8B-Base snapshot prepared by scripts/prepare_model.py.
# An explicit local folder is safest when several snapshots are attached.
MODEL_DIR = None  # e.g. "/kaggle/input/your-qwen3-8b-base-snapshot"
if MODEL_DIR is None:
    candidates = []
    for manifest_path in Path("/kaggle/input").rglob("snapshot_manifest.json"):
        try:
            manifest = json.loads(manifest_path.read_text())
        except (OSError, ValueError):
            continue
        if (manifest.get("repo_id") == "Qwen/Qwen3-8B-Base"
                and manifest.get("revision") == "49e3418fbbbca6ecbdf9608b4d22e5a407081db4"
                and manifest.get("includes_weights") is True):
            candidates.append(manifest_path.parent)
    if len(candidates) != 1:
        raise RuntimeError(
            "Attach one pinned Qwen3-8B-Base snapshot and set MODEL_DIR explicitly "
            "if automatic discovery is ambiguous. Found: " + str(candidates)
        )
    MODEL_DIR = str(candidates[0])
manifest = json.loads((Path(MODEL_DIR) / "snapshot_manifest.json").read_text())
if (manifest.get("schema_version") != 1
        or manifest.get("repo_id") != "Qwen/Qwen3-8B-Base"
        or manifest.get("revision") != "49e3418fbbbca6ecbdf9608b4d22e5a407081db4"
        or manifest.get("includes_weights") is not True):
    raise ValueError("Model snapshot manifest does not match the pinned base model")
for entry in manifest["files"]:
    relative = Path(entry["path"])
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError("Unsafe path in model snapshot manifest")
    artifact = Path(MODEL_DIR) / relative
    if not artifact.is_file() or artifact.stat().st_size != entry["size"]:
        raise ValueError(f"Missing or incomplete snapshot file: {relative}")
model_config = json.loads((Path(MODEL_DIR) / "config.json").read_text())
if (model_config.get("model_type"), model_config.get("hidden_size"),
        model_config.get("num_hidden_layers")) != ("qwen3", 4096, 36):
    raise ValueError("MODEL_DIR is not a Qwen3-8B architecture")
print("Local model:", MODEL_DIR)


In [ ]:
def run_neural(arguments):
    run_runtime("ntl.neural", arguments)
gpu_probe = """
import torch
import bitsandbytes as bnb
from transformers import Qwen3Model
assert torch.cuda.is_available(), 'Enable a free Kaggle T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('Free / total GPU bytes:', torch.cuda.mem_get_info(0))
layer = bnb.nn.Linear4bit(16, 8, bias=False, compute_dtype=torch.float16,
                         quant_type='nf4').to('cuda:0')
inputs = torch.randn(2, 16, device='cuda:0', dtype=torch.float16)
result = layer(inputs)
assert torch.isfinite(result).all(), 'NF4 smoke probe returned nonfinite values'
print('Tiny NF4 forward probe passed:', tuple(result.shape))
del layer, inputs, result
torch.cuda.empty_cache()
"""
subprocess.run([RUNTIME_PYTHON, "-c", gpu_probe], check=True, env=RUNTIME_ENV)
RUN_GROUPED_CV = False
NEURAL_MODEL_DIR = "/kaggle/working/qwen-frozen-model"
FEATURE_ARGS = ["--model-path", str(MODEL_DIR), "--quantization", "nf4",
                "--batch-size", "2", "--max-length", "512"]
if RUN_GROUPED_CV:
    run_neural(["cv", "--train", TRAIN,
                "--output-dir", "/kaggle/working/qwen-frozen-cv", *FEATURE_ARGS])


In [ ]:
run_neural(["train", "--train", TRAIN,
            "--output-dir", NEURAL_MODEL_DIR, *FEATURE_ARGS])
run_neural(["predict", "--model-dir", NEURAL_MODEL_DIR,
            "--test", TEST, "--output", SUBMISSION])


In [ ]:
# Validate the file the competition will score, including arbitrary test size.
validation_code = """
import csv
import json
import sys
import pandas as pd
SUBMISSION, TEST, labels_json = sys.argv[1:]
VALID_LABELS = set(json.loads(labels_json))
with open(TEST, encoding="utf-8-sig", newline="") as handle:
    expected_ids = [row["id"] for row in csv.DictReader(handle)]
submission = pd.read_parquet(SUBMISSION)
if list(submission.columns) != ["id", "model"]:
    raise ValueError("submission.parquet must contain exactly id, model in that order")
if len(submission) != len(expected_ids):
    raise ValueError("Submission row count differs from the current test")
if submission.isna().any().any() or submission["id"].duplicated().any():
    raise ValueError("Submission contains null values or duplicate IDs")
if submission["id"].astype("string").tolist() != expected_ids:
    raise ValueError("Submission must preserve the official test IDs and row order")
if not submission["model"].isin(VALID_LABELS).all():
    raise ValueError("Submission contains a model outside the 18 allowed labels")
print(f"Validated {len(submission):,} predictions: {SUBMISSION}")
print(submission["model"].value_counts().to_string())
print(submission.head().to_string(index=False))
"""
subprocess.run([RUNTIME_PYTHON, "-c", validation_code, SUBMISSION, TEST,
                json.dumps(sorted(VALID_LABELS))], check=True, env=RUNTIME_ENV)


Submit the saved notebook's `submission.parquet`. Keep all model inputs
attached for the offline hidden rerun. The visible example's runtime
does not establish the runtime on about 4,000 hidden rows; keep a large
margin below Kaggle's nine-hour limit. Review the completed run and
release its GPU runtime when finished.
